# Upload cropped volume and meshes to analyze wall heterogenity
Loads a `.vti`/`.vtr` reconstruction, crops the z-stack to the true material thickness (symmetric around a known center voxel), and plots the center-y slice (Mz contrast) before/after to verify the crop landed on the right region.

In [ ]:
# --- Cell: imports ---
import os
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

import sys
sys.path.append('.')  # folder containing vti_magnetic_import.py
from vti_magnetic_import import load_vti_sample, inspect_vti



from scipy.ndimage import gaussian_filter, map_coordinates
from skimage.measure import marching_cubes
import pyvista as pv

import matplotlib.ticker as mticker
import pandas as pd

import json

import Functions_3D as vis
import curvature_segmentation as curve

from scipy import stats

from scipy.optimize import curve_fit
from matplotlib.gridspec import GridSpec


from scipy.spatial import cKDTree
from scipy.ndimage import map_coordinates



# Load the volume

In [ ]:
def load_volume_analysis(save_dir, sample_name, date, smooth):
    """
    Load back everything saved by save_volume_analysis: cropped volume,
    smoothed volume (if present), and all derived-field dicts, fully
    reconstructed including their non-array metadata (shape, spacing, etc).

    Returns
    -------
    cropped : dict
    volume_smoothed : dict or None
    fields : dict of dict
        e.g. fields['jacobian'], fields['curl'], fields['divergence'], etc.,
        each reconstructed as a full dict just like the original.
    metadata : dict
        Top-level metadata, useful for provenance/sanity checks.
    """
    meta_path = os.path.join(save_dir, f'{sample_name}_volume_analysis_metadata_{date}.json')
    with open(meta_path) as f:
        metadata = json.load(f)

    if smooth: 
        # --- Smoothed volume, if it was saved ---
        smoothed_path = os.path.join(save_dir, f'{sample_name}_smoothed_{date}.npz')
        volume = None
        if os.path.exists(smoothed_path):
            smoothed_npz = np.load(smoothed_path)
            volume = {
                'Mx': smoothed_npz['Mx'], 'My': smoothed_npz['My'], 'Mz': smoothed_npz['Mz'],
                'magnitude': smoothed_npz['magnitude'],
                'shape': tuple(smoothed_npz['shape']),
                'sigma': metadata.get('sigma'),
            }

    else:

        cropped_npz = np.load(os.path.join(save_dir, f'{sample_name}_volume_{date}.npz'))
        volume = {
            'Mx': cropped_npz['Mx'], 'My': cropped_npz['My'], 'Mz': cropped_npz['Mz'],
            'magnitude': cropped_npz['magnitude'],
            'shape': tuple(cropped_npz['shape']),
        }


    # --- Each derived-field dict ---
    fields = {}
    for field_name in metadata.get('saved_fields', []):
        field_path = os.path.join(save_dir, f'{sample_name}_{field_name}_{date}.npz')
        field_npz = np.load(field_path)
        field_dict = {key: field_npz[key] for key in field_npz.files}
        field_meta = metadata.get(f'{field_name}_meta', {})
        for k, v in field_meta.items():
            field_dict[k] = tuple(v) if isinstance(v, list) else v
        fields[field_name] = field_dict

    return volume, fields, metadata

In [ ]:
sample_names = ['452', '270', '105', '665'] 
dates = ['mag_const_updated_interpolation', '14AUG26_interpolated', '22AUG26_interpolated', 'mag_const_TV_interpolated']
#dates = ['mag_const_NOTinterpolated_48', '14AUG26_interpolated', '22AUG26_interpolated', '22AUG26_interpolated']
#dates = ['normalized_interpolated','normalized_interpolated'  ]

sample_colors = ['limegreen', 'deeppink', 'cyan', 'gold']
use_smooth = [False, False, False, False]
thickness_list_nm = [33, 66, 45,51]


sample = 0
thickness = thickness_list_nm[sample]
sample_name = sample_names[sample]
date = dates[sample]
sample_color = sample_colors[sample]
smooth = use_smooth[sample]

dz, dy, dx = 2.0, 2.0, 2.0  # nm, 2.0, 2.0, 2.0  # nm,
voxel_nm = 2.0
if sample == 3: 
    crop_size = 400
else: 
    crop_size = 40

# dz, dy, dx = 8.0, 8.0, 8.0  # nm, 2.0, 2.0, 2.0  # nm,
# voxel_nm = 8.0
# crop_size = 5

data_dir = f'C:/Users/emmab/Projects/Vector_Tomography/data_dir/{sample_name}/'

volume, fields, metadata = load_volume_analysis(data_dir, sample_name, date, smooth)

jacobian = fields['jacobian']
curl = fields['curl']
divergence = fields['divergence']
energies = fields['energies']
topology = fields['Nsk']
curvature = fields['curvature']

print("Loaded fields:", list(fields.keys()))
print("Volume shape:", volume['shape'])
print("Volume size:", 16*404*404)
print(volume.keys())

crop = True

# crop volume to get rid of edge effects

In [ ]:
def crop_xy_edges(sample, n_pix=50, verbose=True):
    """
    Remove `n_pix` voxels from each x and y edge of a sample dict, leaving z intact.
    Works on the output of crop_volume_by_thickness / crop_interpolate_volume_by_thickness.
    """
    nz, ny, nx = (int(n) for n in sample['shape'])
    if 2 * n_pix >= min(ny, nx):
        raise ValueError(f"n_pix={n_pix} is too large for an (ny, nx) = ({ny}, {nx}) volume.")

    ys = slice(n_pix, ny - n_pix)
    xs = slice(n_pix, nx - n_pix)
    ny_c, nx_c = ny - 2 * n_pix, nx - 2 * n_pix

    out = dict(sample)
    for key, val in sample.items():
        if not isinstance(val, np.ndarray):
            continue
        if key == 'y_coords':
            out[key] = val[ys].copy()
        elif key == 'x_coords':
            out[key] = val[xs].copy()
        elif val.ndim == 3 and val.shape == (nz, ny, nx):
            out[key] = val[:, ys, xs].copy()
        elif val.ndim == 4 and val.shape[:3] == (nz, ny, nx):
            out[key] = val[:, ys, xs, :].copy()
        # else: leave untouched (z_coords, etc.)

    out['shape'] = (nz, ny_c, nx_c)
    out['dims'] = (nx_c, ny_c, nz)
    out['crop_xy_pixels'] = n_pix

    if verbose:
        dv = sample.get('crop_dz', None)
        nm_txt = f" ({n_pix * dv:g} nm per edge)" if dv else ""
        print(f"xy crop: removed {n_pix} vox{nm_txt} from each edge: "
              f"{(nz, ny, nx)} -> {(nz, ny_c, nx_c)}")
    return out

cropped_xy = crop_xy_edges(volume, n_pix=crop_size)
print(cropped_xy['shape'])   # (16, 304, 304)

if crop == True: 
    volume = cropped_xy

In [ ]:
stop

# Smooth Mz for iso-surface

In [ ]:
sigmas_to_test = [1]

for s in sigmas_to_test:
    vis.apply_gaussian_blur_and_view(volume, field='Mz', sigma=s)


############## SELECT SIGMA TO SAVE ##############
sigma_save = 1
Mz_smooth = volume['smoothed']['Mz'][sigma_save]
##################################################

In [ ]:
#stop

# Marching cubes isosurface extraction at Mz = 0

In [ ]:
# %% [Cell 5: Interpolate Mx, My, Mz onto mesh vertices]
def sample_volume_at_points(volume, points_idx):
    """
    Trilinear interpolation of `volume` at fractional (z, y, x) index
    coordinates given in points_idx, shape (N, 3).
    """
    coords = points_idx.T  # map_coordinates wants shape (ndim, N)
    return map_coordinates(volume, coords, order=1, mode='nearest')

## Creat Iso surface

In [ ]:
Mz = Mz_smooth       # shape (nz, ny, nx)
Mx = volume['Mx']
My = volume['My']

# spacing here scales the output vertex coordinates and normals into physical
# Level = 0 because this is where the domain wall center is!!! 
verts_phys, faces, normals_phys, values = marching_cubes(Mz, level=0.0, spacing=(dz, dy, dx), allow_degenerate=True) # Using allow_degenerate=False didn't visually change anything for 452

print(f"Extracted mesh: {verts_phys.shape[0]} vertices, {faces.shape[0]} triangles")

# Convert physical-space vertex coordinates back into fractional voxel-index
# coordinates (z_idx, y_idx, x_idx), since map_coordinates() needs index space.
verts_idx = verts_phys / np.array([dz, dy, dx])

#Correct normal sign convention]
###########  We want normals to consistently point from -Mz domain into +Mz domain.
# Check by sampling Mz slightly ahead of / behind each vertex along its normal.

eps_phys = dx  # step size in physical units; scale relative to your voxel size 
eps_idx = eps_phys / np.array([dz, dy, dx])

# normals_phys is in (z, y, x) order to match marching_cubes' internal axis order
step = normals_phys * eps_idx  # broadcast scale per-axis

pts_forward = verts_idx + step
pts_backward = verts_idx - step

Mz_forward = sample_volume_at_points(Mz, pts_forward)
Mz_backward = sample_volume_at_points(Mz, pts_backward)

# define chirality at +Mz to -Mz
flip_mask = Mz_forward > Mz_backward
normals_phys[flip_mask] *= -1

print(f"Flipped {flip_mask.sum()} / {len(flip_mask)} normals to enforce -Mz -> +Mz convention")

# Reorder normals and vertex from (z, y, x) to (x, y, z) for consistency with M_v and PyVista
normals_xyz = normals_phys[:, ::-1]
verts_xyz = verts_phys[:, ::-1]  


## Sample Volume at points to create useful mesh

In [ ]:
Mx_v = sample_volume_at_points(Mx, verts_idx)
My_v = sample_volume_at_points(My, verts_idx)
Mz_v = sample_volume_at_points(Mz, verts_idx)  # sample raw (unsmoothed) Mz for the actual physical vector
M_v = np.stack([Mx_v, My_v, Mz_v], axis=1)      # shape (N, 3), order (x, y, z) per vertex

## Calculate dw angles

In [ ]:
def compute_chirality_angle(M_v, normals_xyz, up_axis=np.array([0.0, 0.0, 1.0])):
    """
    Compute a signed angle capturing both wall type (Neel vs Bloch) and
    chirality (rotation sense), for each mesh vertex.

    Parameters
    ----------
    M_v : (N, 3) array
        Magnetization vectors at each mesh vertex (x, y, z order).
    normals_xyz : (N, 3) array
        Wall normal vectors at each mesh vertex (x, y, z order), already
        sign-corrected to point from -Mz into +Mz domain.
    up_axis : (3,) array
        Reference axis used to define tangential direction, defaults to
        the film's easy axis (z), appropriate for PMA films.

    Returns
    -------
    psi_chiral : (N,) array, degrees, range (-180, 180]
        0 deg   -> Neel, pointing along +n_hat
        180 deg -> Neel, pointing along -n_hat (opposite polarity)
        +90 deg -> Bloch, curling in +t_hat direction
        -90 deg -> Bloch, curling in -t_hat direction
    t_hat : (N, 3) array
        The tangential unit vector used at each vertex, in case you want
        to visualize it directly (e.g. as arrows alongside the normals).
    degenerate_mask : (N,) bool array
        True where n_hat was parallel to up_axis, making t_hat undefined
        (chirality is meaningless there; check how many/where this occurs).
    """
    up = np.broadcast_to(up_axis, normals_xyz.shape)
    t_vec = np.cross(up, normals_xyz)  # automatically perpendicular to normals_xyz
    t_norm = np.linalg.norm(t_vec, axis=1)

    degenerate_mask = t_norm < 1e-8
    if np.any(degenerate_mask):
        print(f"Warning: {degenerate_mask.sum()} / {len(degenerate_mask)} vertices "
              f"have normals parallel to up_axis; chirality undefined there.")
        t_norm = np.where(degenerate_mask, 1.0, t_norm)  # avoid divide-by-zero

    t_hat = t_vec / t_norm[:, None]

    M_n = np.einsum('ij,ij->i', M_v, normals_xyz)   # signed, along normal
    M_t_signed = np.einsum('ij,ij->i', M_v, t_hat)   # signed, along tangent

    psi_chiral = np.degrees(np.arctan2(M_t_signed, M_n))

    return psi_chiral, t_hat, degenerate_mask

In [ ]:
# Compute Bloch/Néel character angle at each vertex]
M_n = np.einsum('ij,ij->i', M_v, normals_xyz)          # M component along normal
#M_n = np.linalg.norm(M_v, axis=1)
M_t_vec = M_v - M_n[:, None] * normals_xyz
M_t_mag = np.linalg.norm(M_t_vec, axis=1)
psi_unsigned = np.degrees(np.arctan2(M_t_mag, M_n))             # 0/180 = Néel-like, +-90 = Bloch-like
psi_chiral, t_hat, degenerate_mask = compute_chirality_angle(M_v, normals_xyz)


# Normalize M_n for future calculations. Allows the assignment of Neel and Bloch to be more robust. 
M_mag = np.linalg.norm(M_v, axis=1)               # local |M| at each vertex

# guard against divide-by-zero where |M| is ~0
eps = 1e-12
M_n_n = np.where(M_mag > eps, M_n / np.maximum(M_mag, eps), np.nan)

In [ ]:
div_x = sample_volume_at_points(divergence['div_x'], verts_idx)
div_y = sample_volume_at_points(divergence['div_y'], verts_idx)
div_z = sample_volume_at_points(divergence['div_z'], verts_idx)  # sample raw (unsmoothed) Mz for the actual physical vector
div_M= sample_volume_at_points(divergence['div_M'], verts_idx) 
source = sample_volume_at_points(divergence['source_mask'], verts_idx) 
sink = sample_volume_at_points(divergence['sink_mask'], verts_idx) 

# kappa = sample_volume_at_points(curvature['kappa_map'], verts_idx)


Convert Kappa to a sample-able volume

In [ ]:
# n_layers = 16
# Kappa_vol = np.repeat(curvature['kappa_map'][np.newaxis, :, :], n_layers, axis=0)   # shape (16, 404, 404)


# valid = ~np.isnan(vol)
# filled = np.where(valid, vol, 0.0)

# # coords: shape (3, N), in the same index order as vol
# num = map_coordinates(filled, coords, order=1, mode='nearest')
# den = map_coordinates(valid.astype(float), coords, order=1, mode='nearest')
# sampled = np.where(den > 0, num / den, np.nan)

## save iso cut in df

In [ ]:
def build_wall_dataframe(verts_xyz, normals_xyz, t_hat, M_v,
                          psi_unsigned, psi_chiral, M_normal, degenerate_mask,
                          spacing=(2.0, 2.0, 2.0)):
    """
    Consolidate all per-vertex wall quantities into a single tidy DataFrame,
    one row per mesh vertex. spacing = (dz, dy, dx) in physical units, used
    to also store voxel-index coordinates for binning/slicing by layer.
    """
    dz, dy, dx = spacing
    df = pd.DataFrame({
        'x': verts_xyz[:, 0], 'y': verts_xyz[:, 1], 'z': verts_xyz[:, 2],
        'ix': verts_xyz[:, 0] / dx, 'iy': verts_xyz[:, 1] / dy, 'iz': verts_xyz[:, 2] / dz,
        'nx': normals_xyz[:, 0], 'ny': normals_xyz[:, 1], 'nz': normals_xyz[:, 2],
        'tx': t_hat[:, 0], 'ty': t_hat[:, 1], 'tz': t_hat[:, 2],
        'Mx': M_v[:, 0], 'My': M_v[:, 1], 'Mz': M_v[:, 2],
        'M_normal': M_n,
        'psi_unsigned': psi_unsigned,
        'psi_chiral': psi_chiral,
        'degenerate': degenerate_mask,
    })
    df['vertex_id'] = df.index
    return df

wall_df = build_wall_dataframe(verts_xyz, normals_xyz, t_hat, M_v,
                                psi_unsigned, psi_chiral, M_n, degenerate_mask, spacing=(dz, dy, dx))

In [ ]:
wall_df['curl_magnitude'] = sample_volume_at_points(curl['curl_magnitude'], verts_idx)
wall_df['PMA'] = sample_volume_at_points(energies['PMA'], verts_idx)
wall_df['Exchange'] = sample_volume_at_points(energies['Divergence'], verts_idx)
wall_df['iDMI'] = sample_volume_at_points(energies['iDMI'], verts_idx)
wall_df['Nsk'] = sample_volume_at_points(topology['q_density'], verts_idx)

wall_df['Mx'] = Mx_v
wall_df['My'] = My_v

wall_df['div_M'] = div_M
wall_df['source'] = source
wall_df['sink'] = sink

# wall_df['kappa'] = kappa

wall_df['M_n_n'] = M_n_n


In [ ]:
assert np.allclose(wall_df['M_n_n'], np.cos(np.radians(wall_df['psi_unsigned'])), equal_nan=True)

# Plot Iso-Surface

In [ ]:
def add_volume_slice_pv(plotter, sample, field='Mz', axis='z', index=None,
                         sigma=None, spacing=(2.0, 2.0, 2.0),
                         cmap='viridis', opacity=0.7, vmin=None, vmax=None,
                         show_scalar_bar=False):
    """
    Add a single axis-aligned voxel slice of `field` to a PyVista plotter,
    using the same index convention as plot_three_slice (axis + voxel index).

    axis : 'x', 'y', or 'z' -- which axis to slice along
    index : voxel index along that axis (defaults to center)
    sigma : if not None, use the cached smoothed volume at that sigma
            (sample['smoothed'][key][sigma], as populated by plot_three_slice)
    spacing : (dz, dy, dx) physical voxel size, e.g. (8.0, 8.0, 8.0) for your data
    """
    field_aliases = {'mag': 'magnitude', 'magnitude': 'magnitude',
                      'mx': 'Mx', 'my': 'My', 'mz': 'Mz'}
    key = field_aliases.get(field.lower())
    data = sample[key] if sigma is None else sample['smoothed'][key][sigma]

    nz, ny, nx = sample['shape']
    dz, dy, dx = spacing

    if vmin is None:
        vmin = np.nanmin(sample[key])
    if vmax is None:
        vmax = np.nanmax(sample[key])

    # Build the full 3D grid with correctly-ordered cell data
    grid = pv.ImageData()
    grid.dimensions = np.array((nx, ny, nz)) + 1  # point dims = cell dims + 1
    grid.spacing = (dx, dy, dz)
    grid.origin = (0, 0, 0)
    grid.cell_data[key] = data.flatten(order='C')  # corrected order

    # Default index = center of chosen axis, same as plot_three_slice
    default_index = {'x': nx // 2, 'y': ny // 2, 'z': nz // 2}
    if index is None:
        index = default_index[axis]

    normal_map = {'x': (1, 0, 0), 'y': (0, 1, 0), 'z': (0, 0, 1)}
    spacing_map = {'x': dx, 'y': dy, 'z': dz}
    axis_pos_idx = {'x': 0, 'y': 1, 'z': 2}

    # Place the cutting plane at the *center* of the chosen voxel layer,
    # so the cut passes through the middle of those cells and returns their
    # exact values rather than something interpolated between layers.
    origin = [0.0, 0.0, 0.0]
    origin[axis_pos_idx[axis]] = (index + 0.5) * spacing_map[axis]

    sliced = grid.slice(normal=normal_map[axis], origin=tuple(origin))

    plotter.add_mesh(sliced, scalars=key, cmap=cmap, opacity=opacity,
                      clim=[vmin, vmax], show_scalar_bar=show_scalar_bar)
    return sliced

def plot_map(sample, verts_xyz, faces, plot_dict, sargs, voxel_nm = 2.0, show_grid = True, field = 'Mz', axis = 'z'):
    """
    Visualize the signed chirality/wall-type angle on the domain wall mesh.

    Parameters
    ----------
    verts_xyz, faces : mesh geometry, as produced by marching_cubes
    psi_chiral : (N,) array, degrees, from compute_chirality_angle
    cmap : colormap; 'twilight' or 'hsv' are good choices since they're
        cyclic and represent the wraparound at +-180 deg without a
        visually misleading hard color jump
    show_normals, show_tangents : optionally overlay arrows for n_hat
        and/or t_hat (pass normals_xyz / t_hat if enabling these)
    """

    axis_view = axis
    field_view = field

    cmap = plot_dict['color_map']
    min = plot_dict['bounds'][0]
    max = plot_dict['bounds'][1]


    faces_pv = np.hstack([np.full((faces.shape[0], 1), 3), faces]).astype(np.int64)
    mesh = pv.PolyData(verts_xyz, faces_pv)
    mesh['key_to_map'] = plot_dict['view']

    plotter = pv.Plotter()
    plotter.add_mesh(
        mesh, scalars='key_to_map', cmap=cmap, clim=[min, max], scalar_bar_args=sargs)

    if show_grid: 
        M = sample[field]
        grid = pv.ImageData()
        grid.dimensions = np.array(Mz.shape[::-1]) + 1   # (Mz = samples[sample_name]['cropped']['Mz'] )
        # print(np.array(Mz.shape[::-1]) +1 )
        grid.spacing = (voxel_nm, voxel_nm, voxel_nm) #################### HARD CODED FOR 8nm VOXEL SIZE 
        grid.origin = (0, 0, 0)
        grid.cell_data['Mz'] = Mz.flatten(order='C')  # note: flatten order matters, check orientation
        add_volume_slice_pv(plotter, sample, field=field_view, axis=axis_view, index=0, spacing=(voxel_nm, voxel_nm, voxel_nm), opacity=0.2)  # defaults to center


    plotter.view_isometric()      # view_isometric   view_xy
    plotter.camera.zoom(plot_dict['zoom'])
    plotter.camera.azimuth += 270
    plotter.add_axes()
    plotter.show()
    
    return mesh

In [ ]:
# mesh['Normals'] = normals_xyz
# mesh['Mn'] = M_n
# mesh['psi_deg'] = psi_unsigned
# mesh['Mz_at_vertex'] = Mz_v
# mesh['Mx'] = Mx_v
# mesh['My'] = My_v

## Computed Angles

In [ ]:
# # # # # Wall surface colored by psi character
# sargs = dict(
#     position_x=0.4,  # Move towards the right edge
#     position_y=0.85,   # Move near the bottom
#     height=0.05,       # Make it take up 50% of window height
#     width=0.5,        # Width of the bar
#     vertical=False,    # Orientation
#     title = f'{sample_name} \n psi (deg): 0/180=Neel, +-90=Bloch'
# )

# psi_plot_dict = dict(
#     view= psi_unsigned, 
#     bounds = [0, 90],
#     color_map = 'seismic', 
#     zoom = 1
# )

# plot_map(volume, verts_xyz, faces, psi_plot_dict, sargs, show_grid = True, field = 'Mz', axis = 'z')

In [ ]:
sargs = dict(
    position_x=0.4,  # Move towards the right edge
    position_y=0.85,   # Move near the bottom
    height=0.05,       # Make it take up 50% of window height
    width=0.5,        # Width of the bar
    vertical=False,    # Orientation
    title = f'{sample_name} \n Surface Normal 0=Bloch, +-Mn=Neel'
)

psi_plot_dict = dict(
    view= M_n, 
    bounds = [np.min(My_v), np.max(My_v)],
    color_map = 'brg', 
    zoom = 1
)

plot_map(volume, verts_xyz, faces, psi_plot_dict, sargs, voxel_nm = voxel_nm,  show_grid = True, field = 'Mz', axis = 'z')


sargs = dict(
    position_x=0.4,  # Move towards the right edge
    position_y=0.85,   # Move near the bottom
    height=0.05,       # Make it take up 50% of window height
    width=0.5,        # Width of the bar
    vertical=False,    # Orientation
    title = f'{sample_name} \n Surface Normal 0=Bloch, +-1=Neel'
)

psi_plot_dict = dict(
    view= M_n_n, 
    bounds = [-1, 1],
    color_map = 'brg', 
    zoom = 1
)

plot_map(volume, verts_xyz, faces, psi_plot_dict, sargs, voxel_nm = voxel_nm,  show_grid = True, field = 'Mz', axis = 'z')


In [ ]:
# sargs = dict(
#     position_x=0.4,  # Move towards the right edge
#     position_y=0.85,   # Move near the bottom
#     height=0.05,       # Make it take up 50% of window height
#     width=0.5,        # Width of the bar
#     vertical=False,    # Orientation
#     title = f'{sample_name} \n Psi Chiral '
# )

# psi_plot_dict = dict(
#     view= psi_chiral, 
#     bounds = [-180, 180],
#     color_map = 'seismic', 
#     zoom = 1
# )

# plot_map(volume, verts_xyz, faces, psi_plot_dict, sargs, show_grid = True, field = 'Mz', axis = 'z')


## Computed Vectors and Energies

In [ ]:
# sargs = dict(
#     position_x=0.4,  # Move towards the right edge
#     position_y=0.85,   # Move near the bottom
#     height=0.05,       # Make it take up 50% of window height
#     width=0.5,        # Width of the bar
#     vertical=False,    # Orientation
#     title = f'{sample_name} \n Kappa'
# )

# psi_plot_dict = dict(
#     view= wall_df['kappa'].values, 
#     bounds = [-0.001, 0.001],
#     color_map = 'coolwarm', 
#     zoom = 1
# )

# plot_map(volume, verts_xyz, faces, psi_plot_dict, sargs, show_grid = True, field = 'Mz', axis = 'z')

In [ ]:
# sargs = dict(
#     position_x=0.4,  # Move towards the right edge
#     position_y=0.85,   # Move near the bottom
#     height=0.05,       # Make it take up 50% of window height
#     width=0.5,        # Width of the bar
#     vertical=False,    # Orientation
#     title = f'{sample_name} \n Divergence'
# )

# psi_plot_dict = dict(
#     view= wall_df['Exchange'].values *10000000, 
#     bounds = [0, 1],
#     color_map = 'Purples', 
#     zoom = 1
# )

# plot_map(volume, verts_xyz, faces, psi_plot_dict, sargs, show_grid = True, field = 'Mz', axis = 'z')

In [ ]:
# sargs = dict(
#     position_x=0.4,  # Move towards the right edge
#     position_y=0.85,   # Move near the bottom
#     height=0.05,       # Make it take up 50% of window height
#     width=0.5,        # Width of the bar
#     vertical=False,    # Orientation
#     title = f'{sample_name} \n PMA'
# )

# psi_plot_dict = dict(
#     view= wall_df['PMA'].values *1000000 , 
#     bounds = [0, 1],
#     color_map = 'Oranges', 
#     zoom = 1
# )

# plot_map(volume, verts_xyz, faces, psi_plot_dict, sargs, show_grid = True, field = 'Mz', axis = 'z')

In [ ]:
# sargs = dict(
#     position_x=0.4,  # Move towards the right edge
#     position_y=0.85,   # Move near the bottom
#     height=0.05,       # Make it take up 50% of window height
#     width=0.5,        # Width of the bar
#     vertical=False,    # Orientation
#     title = f'{sample_name} \n Curl'
# )

# psi_plot_dict = dict(
#     view= wall_df['curl_magnitude'].values *1000 , 
#     bounds = [-0.1, 0.1],
#     color_map = 'RdBu_r', 
#     zoom = 1
# )

# plot_map(volume, verts_xyz, faces, psi_plot_dict, sargs, show_grid = True, field = 'Mz', axis = 'z')

In [ ]:
# sargs = dict(
#     position_x=0.4,  # Move towards the right edge
#     position_y=0.85,   # Move near the bottom
#     height=0.05,       # Make it take up 50% of window height
#     width=0.5,        # Width of the bar
#     vertical=False,    # Orientation
#     title = f'{sample_name} \n Divergence'
# )

# psi_plot_dict = dict(
#     view= wall_df['div_M'].values *10000 , 
#     bounds = [-0.1, 0.1],
#     color_map = 'plasma', 
#     zoom = 1
# )

# plot_map(volume, verts_xyz, faces, psi_plot_dict, sargs, show_grid = True, field = 'Mz', axis = 'z')

In [ ]:
# sargs = dict(
#     position_x=0.4,  # Move towards the right edge
#     position_y=0.85,   # Move near the bottom
#     height=0.05,       # Make it take up 50% of window height
#     width=0.5,        # Width of the bar
#     vertical=False,    # Orientation
#     title = f'{sample_name} \n source'
# )

# psi_plot_dict = dict(
#     view= wall_df['source'].values, 
#     bounds = [-1, 1],
#     color_map = 'Greys', 
#     zoom = 1
# )

# plot_map(volume, verts_xyz, faces, psi_plot_dict, sargs, show_grid = True, field = 'Mz', axis = 'z')

In [ ]:
# sargs = dict(
#     position_x=0.4,  # Move towards the right edge
#     position_y=0.85,   # Move near the bottom
#     height=0.05,       # Make it take up 50% of window height
#     width=0.5,        # Width of the bar
#     vertical=False,    # Orientation
#     title = f'{sample_name} \n sink'
# )

# psi_plot_dict = dict(
#     view= wall_df['sink'].values, 
#     bounds = [-1, 1],
#     color_map = 'Greys', 
#     zoom = 1
# )

# plot_map(volume, verts_xyz, faces, psi_plot_dict, sargs, show_grid = True, field = 'Mz', axis = 'z')

In [ ]:
# sargs = dict(
#     position_x=0.4,  # Move towards the right edge
#     position_y=0.85,   # Move near the bottom
#     height=0.05,       # Make it take up 50% of window height
#     width=0.5,        # Width of the bar
#     vertical=False,    # Orientation
#     title = f'{sample_name} \n  iDMI'
# )

# psi_plot_dict = dict(
#     view= wall_df['iDMI'].values * 10, 
#     bounds = [-0.5, 0.5],
#     color_map = 'Greens', 
#     zoom = 1
# )

# plot_map(volume, verts_xyz, faces, psi_plot_dict, sargs, show_grid = True, field = 'Mz', axis = 'z')

In [ ]:
# sargs = dict(
#     position_x=0.4,  # Move towards the right edge
#     position_y=0.85,   # Move near the bottom
#     height=0.05,       # Make it take up 50% of window height
#     width=0.5,        # Width of the bar
#     vertical=False,    # Orientation
#     title = f'{sample_name} \n  Nsk'
# )

# psi_plot_dict = dict(
#     view= wall_df['Nsk'].values*1000, 
#     bounds = [-1, 1],
#     color_map = 'seismic', 
#     zoom = 1
# )

# plot_map(volume, verts_xyz, faces, psi_plot_dict, sargs, show_grid = True, field = 'Mz', axis = 'z')

# Analyze ISO-surface

## Bloch vs thickness first

### Bloch vs z using m = 0 crossing

In [ ]:
def plot_Bloch_angle_vs_z_m0(wall_df, bounds, angle_col='M_n_n',
                          bloch_axis=None, neel_axis=None, title=None,
                          drop_degenerate=False, dz=2.0):
    """"
    Plots (top) mean angle_col vs z voxel layer and (bottom) the number of
    M_normal = 0 crossings found in each z voxel layer.

    Zero crossings are found per (x, y) voxel column by sorting vertices in z
    and linearly interpolating wherever angle_col changes sign (or is exactly 0).

    Parameters
    ----------
    max_gap : float
        Max z separation (in voxels) between consecutive vertices across which a
        sign change is trusted. Prevents interpolating across holes in the mesh.
    drop_degenerate : bool
        Exclude vertices flagged degenerate when finding crossings.
    dz : float
        z spacing in physical units, used only for the 'z0_phys' column.

    Returns
    -------
    grouped : DataFrame, mean/std/count of angle_col per z layer (as before)
    zero_df : DataFrame, one row per column with exactly ONE crossing
              columns: col_x, col_y, z0_vox (fractional voxel), z0_layer (rounded), z0_phys
    misc_df : DataFrame, columns with no crossing or multiple crossings
              columns: col_x, col_y, status ('none' or 'multiple'), z0_list (voxel units)
    """
    df = wall_df.copy()
    z_lo, z_hi = bounds
    z_min = z_lo - z_lo
    z_max = z_hi - z_lo

    max_gap = dz

    df['z_layer'] = df['iz'].round().astype(int)

    # Restrict to the physical film thickness
    n_before = len(df)
    df = df[(df['z_layer'] >= z_min) & (df['z_layer'] <= z_max)]
    print(f"{len(df)} / {n_before} vertices fall within layers {z_min}-{z_max}")

    # ---- mean profile (same as before) ----
    grouped = df.groupby('z_layer')[angle_col].agg(['mean', 'std', 'count'])

    # ---- zero crossings per (x, y) column ----
    d = df
    if drop_degenerate and 'degenerate' in d.columns:
        d = d[~d['degenerate'].astype(bool)]
    d = d.dropna(subset=[angle_col]).copy()
    d['col_x'] = np.rint(d['ix']).astype(int)
    d['col_y'] = np.rint(d['iy']).astype(int)

    good_rows, misc_rows = [], []
    for (cx, cy), g in d.groupby(['col_x', 'col_y']):
        g = g.sort_values('iz')
        z = g['iz'].to_numpy()
        m = g[angle_col].to_numpy()

        crossings = z[m == 0].tolist()                      # exact zeros
        if len(z) >= 2:
            s = np.sign(m)
            for i in np.where((s[:-1] * s[1:]) < 0)[0]:     # sign changes
                if (z[i + 1] - z[i]) <= max_gap:
                    crossings.append(z[i] - m[i] * (z[i + 1] - z[i]) / (m[i + 1] - m[i]))

        if len(crossings) == 1:
            z0 = crossings[0]
            good_rows.append({'col_x': cx, 'col_y': cy, 'z0_vox': z0,
                              'z0_layer': int(round(z0)), 'z0_phys': z0 * dz})
        elif len(crossings) == 0:
            misc_rows.append({'col_x': cx, 'col_y': cy, 'status': 'none', 'z0_list': []})
        else:
            misc_rows.append({'col_x': cx, 'col_y': cy, 'status': 'multiple',
                              'z0_list': sorted(crossings)})

    zero_df = pd.DataFrame(good_rows, columns=['col_x', 'col_y', 'z0_vox', 'z0_layer', 'z0_phys'])
    misc_df = pd.DataFrame(misc_rows, columns=['col_x', 'col_y', 'status', 'z0_list'])

    n_cols = len(zero_df) + len(misc_df)
    print(f"{len(zero_df)} / {n_cols} columns have exactly one {angle_col}=0 crossing")
    if len(misc_df):
        print(misc_df['status'].value_counts().to_string())

    # ---- plots ----
    fig, (ax, ax2) = plt.subplots(1, 2, figsize=(18, 6), sharex=True)

    ax.errorbar(grouped.index, grouped['mean'], yerr=grouped['std'],
                fmt='o-', capsize=3)
    ax.axhline(0, color='k', lw=0.8, ls=':')
    if bloch_axis is not None:
        for deg in bloch_axis:
            ax.axhline(deg, color='magenta', label='Bloch Wall')
    if neel_axis is not None:
        for deg in neel_axis:
            ax.axhline(deg, color='cyan', label='Neel Wall')
    ax.set_ylabel(angle_col, fontsize = '15')
    ax.set_xlabel('z voxel layer', fontsize = '15')

    layers = np.arange(z_min, z_max + 1)
    counts = zero_df['z0_layer'].value_counts().reindex(layers, fill_value=0)
    ax2.bar(layers, counts.to_numpy(), width=0.9, color='green', alpha = 0.5)
    ax2.set_xlabel('z voxel layer', fontsize = '15')
    ax2.set_ylabel(f'# of {angle_col} = 0 crossings', fontsize = '15')
    ax2.set_xticks(layers)

    fig.suptitle(f'{angle_col} vs. thickness position \n {title}', fontsize = '20')


    plt.tight_layout()
    plt.show()

    return grouped, zero_df, misc_df

bounds = [0,int(volume['shape'][0])]
grouped, zero_df, misc_df = plot_Bloch_angle_vs_z_m0(wall_df, bounds, bloch_axis = [0] , neel_axis = [-1, 1], title=f'{sample_name}', dz = voxel_nm)

### Bloch using a mask of bloch  = True

In [ ]:
def classify_pure_bloch_category(wall_df, method='normal', tol=0.01):
    """
    Classify wall vertices as 'bloch', 'neel', or 'unclassified' using one
    of two strict criteria.

    Parameters
    ----------
    wall_df : pd.DataFrame
        Must contain Mx, My, Mz, nx, ny, nz (and ix, iy, iz / x, y, z for
        location bookkeeping).
    method : 'normal' 
        'normal' -- classify using M_n = M.n_hat directly.
            Bloch: |M_n| < tol           (tangential, no normal component)
            Neel:  |M_n| = +/- 1
    tol : float
        Tolerance for the chosen method. Recommended: 0.0005 for 'normal'
        (dimensionless, dot-product units), 0.005 for 'psi' (radians).

    Returns
    -------
    df : pd.DataFrame
        Copy of wall_df with an added 'category' column
        ('bloch' / 'neel' / 'unclassified') and the raw comparison
        quantity used (M_n and mag, or psi_rad) as new columns.
    """
    df = wall_df.copy()
    category = np.full(len(df), 'unclassified', dtype=object)

    M_n = wall_df['M_n_n']
    is_bloch = np.abs(M_n) < tol
    category[is_bloch] = 'bloch'

    df['category'] = category
    n_bloch = (category == 'bloch').sum()

    is_bloch = (df['category'] == 'bloch').values
    is_other = ~is_bloch

    n_other = is_other.sum()


    return df, n_bloch, n_other


def extract_pure_locations(wall_df):
    """
    Pull out (x, y, z) physical coords and iz (true experimental voxel
    layer) for bloch.
    """
    cols = ['x', 'y', 'z', 'ix', 'iy', 'iz']
    bloch_locs = wall_df[wall_df['category'] == 'bloch'][cols].copy()
    return bloch_locs

# def plot_pure_bloch_3d(mesh_verts_xyz, wall_df, kappa=None,
#                             bloch_color='magenta', tol = 0.01,
#                             show_unclassified=True, point_size=6,
#                             kappa_lim=(-0.01, 0.01), other_opacity=0.6, ax=None):
#     """
#     3D PyVista scatter of classified wall vertices.

#     mesh_verts_xyz : (N, 3) array matching wall_df row order
#         (i.e. the same verts_xyz used to build wall_df originally).
#     kappa : (N,) array of local curvature in the same row order as mesh_verts_xyz.
#         If None, uses wall_df['kappa'] when present, otherwise falls back
#         to light gray.
#     kappa_lim : (vmin, vmax) for the coolwarm color scale (symmetric about 0
#         so white = flat).
#     """

#     plotter = pv.Plotter()

#     is_bloch = (wall_df['category'] == 'bloch').values
#     is_other = ~is_bloch

#     if kappa is None and 'kappa' in wall_df.columns:
#         kappa = wall_df['kappa'].to_numpy()

#     if show_unclassified and is_other.any():
#         other_pts = pv.PolyData(np.asarray(mesh_verts_xyz)[is_other])

#         if kappa is not None:
#             kappa = np.asarray(kappa, dtype=float)
#             assert len(kappa) == len(mesh_verts_xyz), \
#                 "kappa must have the same length/order as mesh_verts_xyz"
#             other_pts['kappa'] = kappa[is_other]
#             plotter.add_mesh(other_pts, scalars='kappa', cmap='coolwarm',
#                              clim=kappa_lim, style='points',
#                              point_size=point_size * 0.7,
#                              render_points_as_spheres=True,
#                              opacity=other_opacity,
#                              nan_color='lightgray', nan_opacity=0.15,
#                              scalar_bar_args={'title': 'local curvature κ'})
#         else:
#             plotter.add_mesh(other_pts, color='lightgray', style='points',
#                              point_size=point_size * 0.5, opacity=0.15,
#                              render_points_as_spheres=True)

#     if is_bloch.any():
#         plotter.add_points(np.asarray(mesh_verts_xyz)[is_bloch], color=bloch_color,
#                            point_size=point_size, render_points_as_spheres=True,
#                            label=f'Isolated Bloch Core with tol = {tol}')

#     plotter.add_legend(size=(0.2, 0.2)) 
#     plotter.add_axes()
#     plotter.show()


def plot_pure_bloch_3d(mesh_verts_xyz, wall_df, kappa=None,
                       bloch_color='magenta', tol=0.01,
                       show_unclassified=True, point_size=6,
                       kappa_lim=(-0.01, 0.01), other_opacity=0.6,
                       ax=None, window_size=(1000, 1000)):
    """
    PyVista scatter of classified wall vertices.
    If ax is given, the scene is rendered off-screen and drawn into ax as an image.
    If ax is None, an interactive PyVista window opens instead.
    """
    interactive = ax is None
    plotter = pv.Plotter(off_screen=not interactive, window_size=window_size)

    is_bloch = (wall_df['category'] == 'bloch').values
    is_other = ~is_bloch

    if kappa is None and 'kappa' in wall_df.columns:
        kappa = wall_df['kappa'].to_numpy()

    if show_unclassified and is_other.any():
        other_pts = pv.PolyData(np.asarray(mesh_verts_xyz)[is_other])

        if kappa is not None:
            kappa = np.asarray(kappa, dtype=float)
            assert len(kappa) == len(mesh_verts_xyz), \
                "kappa must have the same length/order as mesh_verts_xyz"
            other_pts['kappa'] = kappa[is_other]
            plotter.add_mesh(other_pts, scalars='kappa', cmap='coolwarm',
                             clim=kappa_lim, style='points',
                             point_size=point_size * 0.7,
                             render_points_as_spheres=True,
                             opacity=other_opacity,
                             nan_color='lightgray', nan_opacity=0.15,
                             scalar_bar_args={'title': 'local curvature κ'})
        else:
            plotter.add_mesh(other_pts, color='lightgray', style='points',
                             point_size=point_size * 0.5, opacity=0.15,
                             render_points_as_spheres=True)

    if is_bloch.any():
        plotter.add_points(np.asarray(mesh_verts_xyz)[is_bloch], color=bloch_color,
                           point_size=point_size, render_points_as_spheres=True)

    # plotter.add_legend(size=(0.2, 0.2))
    plotter.add_axes()

    if interactive:
        plotter.show()
        return None

    img = plotter.screenshot(return_img=True, scale=2) 
    plotter.close()

    ax.imshow(img)
    ax.axis('off')
    ax.set_title(f'Isolated Bloch Core with tol = +/- {tol}')
    return ax


In [ ]:


def _gaussian(x, A, mu, sigma):
    return A * np.exp(-(x - mu)**2 / (2 * sigma**2))



def plot_bloch_vs_thickness_histonly(wall_df, verts_xyz, mn_col='M_n_n', category='bloch',  #kappa = None, 
                            voxel_nm=None, tol = 0.1, color='magenta', title=None, show_fit=True):
    """
    Left:  mean +/- std of mn_col in each z-layer (Neel = +/-1, Bloch = 0).
    Right: fraction of the vertices in each z-layer classified as `category`
           (i.e. which layers the Bloch condition is met in).

    z-layers are derived from wall_df['iz'], so no bounds are needed.
    If voxel_nm is given, the x-axis is nm from the film center; otherwise
    it is the voxel layer number (in whatever frame iz is in).

    Parameters
    ----------
    wall_df : DataFrame with 'iz' and mn_col
    """
    df = wall_df.copy()
    df['z_layer'] = df['iz'].round().astype(int)

    # film geometry from the data itself
    z_min, z_max = df['z_layer'].min(), df['z_layer'].max()
    layers = np.arange(z_min, z_max + 1)
    z_center = 0.5 * (z_min + z_max)
    x_vals = (layers - z_center) * voxel_nm if voxel_nm else layers
    xlabel = ('distance from film center (nm)' if voxel_nm
              else 'z voxel layer')
    print(f"z layers {z_min}-{z_max} ({len(layers)} layers), center = {z_center}")

    # left panel: mean/std per layer
    grouped = (df.groupby('z_layer')[mn_col].agg(['mean', 'std', 'count'])
                 .reindex(layers))

    # right panel: fraction of each layer's vertices that are `category`
    df['is_cat'] = (wall_df['category'].reindex(df.index) == category)
    cat = df.groupby('z_layer')['is_cat'].agg(n_cat='sum', frac_cat='mean').reindex(layers)
    summary = grouped.join(cat)

    y = (summary['n_cat']).to_numpy(float)
    y = np.nan_to_num(y)
    width = (x_vals[1] - x_vals[0])

    # ---- empirical statistics of the histogram (weighted by bar height) ----
    stats = {}

    mean_e = np.sum(x_vals * y) / y.sum()
    var_e = np.sum(y * (x_vals - mean_e)**2) / y.sum()
    edges = np.concatenate([x_vals - width / 2, [x_vals[-1] + width / 2]])
    cum = np.concatenate([[0], np.cumsum(y)]) / y.sum()
    median_e = float(np.interp(0.5, cum, edges))   # interpolated within bins
    stats.update(mean=mean_e, median=median_e, variance=var_e, unit='nm')
 
    # ---- Gaussian fit ----
    if show_fit:
        p0 = [y.max(), stats['mean'], max(np.sqrt(stats['variance']), width)]
        popt, _ = curve_fit(_gaussian, x_vals, y, p0=p0,
                            bounds=([0, x_vals.min() - width, 1e-6],
                                    [np.inf, x_vals.max() + width, np.inf]))
        A, mu, sigma = popt
        sigma = abs(sigma)
        fwhm = 2 * np.sqrt(2 * np.log(2)) * sigma
        ss_res = np.sum((y - _gaussian(x_vals, *popt))**2)
        ss_tot = np.sum((y - y.mean())**2)
        r2 = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan
        stats.update(fit_amplitude=A, fit_mean=mu, fit_sigma=sigma,
                        fit_variance=sigma**2, fwhm=fwhm, r2=r2)



    
    fig = plt.figure(figsize=(7, 17))
    gs = GridSpec(2, 1, height_ratios=[3, 1], figure=fig)
    ax1 = fig.add_subplot(gs[0])
    ax2 = fig.add_subplot(gs[1])

    plot_pure_bloch_3d(verts_xyz, wall_df, kappa=wall_df['kappa'].to_numpy(),
                    tol=tol, ax=ax1)

    # Panel 3: counts where M_norm = 0 (previously ax2)
    ax2.bar(x_vals, y, width=width, color=color, alpha=0.5, edgecolor='gray')
    ax2.axvline(0, color='k', ls='--', label='Film center')

    if stats:
        ax2.axvline(stats['mean'], color='red', lw=1.5,
                    label=f"mean = {stats['mean']:.2f} nm")
        ax2.axvline(stats['median'], color='tab:blue', lw=1.5, ls='-.',
                    label=f"median = {stats['median']:.2f} nm")
        ax2.plot([], [], ' ', label=f"variance = {stats['variance']:.2f} nm^2")

    if show_fit:
        xs = np.linspace(x_vals.min() - width / 2, x_vals.max() + width / 2, 300)
        ax2.plot(xs, _gaussian(xs, A, mu, sigma), color='k', lw=2,
                 label=f"Gaussian fit (R² = {r2:.2f})")
        ax2.hlines(A / 2, mu - fwhm / 2, mu + fwhm / 2, color='k', lw=2.5, alpha = 0,
                   label=f"FWHM = {fwhm:.2f} nm")
        ax2.axvline(mu, color='darkorange', lw=1, ls=':')

    
    ax2.legend(loc = 'upper left', fontsize=15)
    ax2.set_xticks(x_vals)

    ax2.set_xlabel(xlabel)
    ax2.set_ylabel('M_norm')
    ax2.set_xticklabels([f'{v:g}' for v in x_vals], rotation=45)

    if title:
        fig.suptitle(title)
    plt.tight_layout()
    plt.show()
    return summary




def plot_bloch_vs_thickness(wall_df, verts_xyz, mn_col='M_n_n', category='bloch', kappa = None, 
                            voxel_nm=None, tol = 0.1, color='magenta', title=None, show_fit=True):
    """
    Left:  mean +/- std of mn_col in each z-layer (Neel = +/-1, Bloch = 0).
    Right: fraction of the vertices in each z-layer classified as `category`
           (i.e. which layers the Bloch condition is met in).

    z-layers are derived from wall_df['iz'], so no bounds are needed.
    If voxel_nm is given, the x-axis is nm from the film center; otherwise
    it is the voxel layer number (in whatever frame iz is in).

    Parameters
    ----------
    wall_df : DataFrame with 'iz' and mn_col
    """
    df = wall_df.copy()
    df['z_layer'] = df['iz'].round().astype(int)

    # film geometry from the data itself
    z_min, z_max = df['z_layer'].min(), df['z_layer'].max()
    layers = np.arange(z_min, z_max + 1)
    z_center = 0.5 * (z_min + z_max)
    x_vals = (layers - z_center) * voxel_nm if voxel_nm else layers
    xlabel = ('distance from film center (nm)' if voxel_nm
              else 'z voxel layer')
    print(f"z layers {z_min}-{z_max} ({len(layers)} layers), center = {z_center}")

    # left panel: mean/std per layer
    grouped = (df.groupby('z_layer')[mn_col].agg(['mean', 'std', 'count'])
                 .reindex(layers))

    # right panel: fraction of each layer's vertices that are `category`
    df['is_cat'] = (wall_df['category'].reindex(df.index) == category)
    cat = df.groupby('z_layer')['is_cat'].agg(n_cat='sum', frac_cat='mean').reindex(layers)
    summary = grouped.join(cat)

    y = (summary['n_cat']).to_numpy(float)
    y = np.nan_to_num(y)
    width = (x_vals[1] - x_vals[0])

    # ---- empirical statistics of the histogram (weighted by bar height) ----
    stats = {}

    mean_e = np.sum(x_vals * y) / y.sum()
    var_e = np.sum(y * (x_vals - mean_e)**2) / y.sum()
    edges = np.concatenate([x_vals - width / 2, [x_vals[-1] + width / 2]])
    cum = np.concatenate([[0], np.cumsum(y)]) / y.sum()
    median_e = float(np.interp(0.5, cum, edges))   # interpolated within bins
    stats.update(mean=mean_e, median=median_e, variance=var_e, unit='nm')
 
    # ---- Gaussian fit ----
    if show_fit:
        p0 = [y.max(), stats['mean'], max(np.sqrt(stats['variance']), width)]
        popt, _ = curve_fit(_gaussian, x_vals, y, p0=p0,
                            bounds=([0, x_vals.min() - width, 1e-6],
                                    [np.inf, x_vals.max() + width, np.inf]))
        A, mu, sigma = popt
        sigma = abs(sigma)
        fwhm = 2 * np.sqrt(2 * np.log(2)) * sigma
        ss_res = np.sum((y - _gaussian(x_vals, *popt))**2)
        ss_tot = np.sum((y - y.mean())**2)
        r2 = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan
        stats.update(fit_amplitude=A, fit_mean=mu, fit_sigma=sigma,
                        fit_variance=sigma**2, fwhm=fwhm, r2=r2)



    
    fig = plt.figure(figsize=(24, 7))
    gs = GridSpec(1, 2, width_ratios=[1, 1], figure=fig)
    # ax1 = fig.add_subplot(gs[0])
    ax2 = fig.add_subplot(gs[0])
    ax3 = fig.add_subplot(gs[1], sharex=ax2)

    # plot_pure_bloch_3d(verts_xyz, wall_df, kappa=wall_df['kappa'].to_numpy(),
    #                 tol=tol, ax=ax1)

    # Panel 2: M_norm profile (previously ax1)
    ax2.errorbar(x_vals, grouped['mean'], yerr=grouped['std'], fmt='o-',
                capsize=3, color=color)
    ax2.axhline(0, color='magenta', ls='--', label='Bloch (0)')
    ax2.axhline(1, color='cyan', ls='--', label='Neel (±1)')
    ax2.axhline(-1, color='cyan', ls='--')
    ax2.axvline(0, color='k', ls='--', label='Film center')
    ax2.set_ylim(-1.1, 1.1)
    ax2.set_xlabel(xlabel, fontsize = 15)
    ax2.set_ylabel('M_norm', fontsize = 15)
    ax2.legend(fontsize=15)

    # Panel 3: counts where M_norm = 0 (previously ax2)
    ax3.bar(x_vals, y, width=width, color=color, alpha=0.5, edgecolor='gray')
    ax3.axvline(0, color='k', ls='--', label='Film center')

    if stats:
        ax3.axvline(stats['mean'], color='red', lw=1.5,
                    label=f"mean = {stats['mean']:.2f} nm")
        ax3.axvline(stats['median'], color='tab:blue', lw=1.5, ls='-.',
                    label=f"median = {stats['median']:.2f} nm")
        ax3.plot([], [], ' ', label=f"variance = {stats['variance']:.2f} nm^2")

    if show_fit:
        xs = np.linspace(x_vals.min() - width / 2, x_vals.max() + width / 2, 300)
        ax3.plot(xs, _gaussian(xs, A, mu, sigma), color='k', lw=2,
                 label=f"Gaussian fit (R² = {r2:.2f})")
        ax3.hlines(A / 2, mu - fwhm / 2, mu + fwhm / 2, color='k', lw=2.5, alpha = 0,
                   label=f"FWHM = {fwhm:.2f} nm")
        ax3.axvline(mu, color='darkorange', lw=1, ls=':')

    
    ax3.legend(loc = 'upper left', fontsize=15)
    ax3.set_xlabel(xlabel, fontsize = 15)
    ax3.set_ylabel('# of Bloch points per z', fontsize = 15)
    ax3.set_xticks(x_vals)
    ax3.set_xticklabels([f'{v:g}' for v in x_vals], rotation=45)

    if title:
        fig.suptitle(title)
    plt.tight_layout()
    plt.show()
    return summary

In [ ]:
tol = 0.5

wall_df, n_bloch, n_other = classify_pure_bloch_category(wall_df, method='normal', tol=tol) #update wall_df with category! 

bloch_normal_locs = extract_pure_locations(wall_df)

print(f'number of bloch points found = {n_bloch} out of a total volume size of {n_other+n_bloch}')
print(f'% Bloch = {n_bloch/ (n_other+n_bloch) *100}. Should be about {1/5 *100}')

#plot_pure_bloch_3d(verts_xyz, wall_df, kappa=wall_df['kappa'].to_numpy(), tol = tol)


summary = plot_bloch_vs_thickness(wall_df, verts_xyz, mn_col='M_n_n', category='bloch', voxel_nm=voxel_nm, tol = tol, color = sample_color)
#summary = plot_bloch_vs_thickness_histonly(wall_df, verts_xyz, mn_col='M_n_n', category='bloch', voxel_nm=voxel_nm, tol = tol, color = sample_color)

Look at the effect of tolerance

In [ ]:
def bloch_z_stats(df_cat, category='bloch', voxel_nm=2.0, fit=True):
    """
    Histogram statistics of where `category` vertices fall through the
    thickness (one bin per z-layer, bar height = vertex count), plus an
    optional Gaussian fit for the FWHM. No plotting.
    """
    layer = df_cat['iz'].round().astype(int)
    layers = np.arange(layer.min(), layer.max() + 1)
    x_vals = (layers - 0.5 * (layers[0] + layers[-1])) * voxel_nm   # nm from film center
    width = voxel_nm

    y = ((df_cat['category'] == category).groupby(layer).sum()
         .reindex(layers).fillna(0).to_numpy(float))

    out = dict(n_cat=int(y.sum()), mean=np.nan, median=np.nan, variance=np.nan,
               fwhm=np.nan, r2=np.nan,
               var_uniform=width**2 * (len(layers)**2 - 1) / 12,  # variance if vertices were spread evenly over all layers
               thickness=len(layers) * width)
    if y.sum() == 0:
        return out

    mean = np.sum(x_vals * y) / y.sum()
    var = np.sum(y * (x_vals - mean)**2) / y.sum()
    edges = np.concatenate([x_vals - width / 2, [x_vals[-1] + width / 2]])
    cum = np.concatenate([[0], np.cumsum(y)]) / y.sum()
    out.update(mean=mean, median=float(np.interp(0.5, cum, edges)), variance=var)

    if fit:
        try:
            p0 = [y.max(), mean, max(np.sqrt(var), width)]
            popt, _ = curve_fit(_gaussian, x_vals, y, p0=p0,
                                bounds=([0, x_vals.min() - width, 1e-6],
                                        [np.inf, x_vals.max() + width, np.inf]))
            sigma = abs(popt[2])
            ss_res = np.sum((y - _gaussian(x_vals, *popt))**2)
            ss_tot = np.sum((y - y.mean())**2)
            out.update(fwhm=2 * np.sqrt(2 * np.log(2)) * sigma,
                       r2=1 - ss_res / ss_tot if ss_tot > 0 else np.nan)
        except Exception as e:
            print(f"  fit failed ({e})")
    return out


def tolerance_sweep(wall_df, tol_list, category='bloch', voxel_nm=2.0,
                    method='normal', color='magenta', title=None):
    rows = []
    for tol in tol_list:
        # copy so each tolerance starts from the same wall_df
        df_t, n_bloch, n_other = classify_pure_bloch_category(
            wall_df.copy(), method=method, tol=tol)
        s = bloch_z_stats(df_t, category=category, voxel_nm=voxel_nm)
        s.update(tol=tol, pct_bloch=100 * n_bloch / (n_bloch + n_other))
        rows.append(s)
        # print(f"tol={tol:.2f}: {n_bloch} {category} vertices ({s['pct_bloch']:.1f}%), "
        #       f"mean={s['mean']:.2f}, median={s['median']:.2f}, "
        #       f"var={s['variance']:.2f}, FWHM={s['fwhm']:.2f} nm")
    res = pd.DataFrame(rows).set_index('tol')

    fig, ax = plt.subplots(3,1, figsize=(5, 15), sharex=True)
    panels = [('mean', 'mean (nm from film center)'),
              ('median', 'median (nm from film center)'),
              ('fwhm', 'FWHM of Gaussian fit (nm)')]


    ax[0].plot(res.index, res['pct_bloch'], 'o-', color=color)
    ax[0].set_ylabel('% Bloch', fontsize=10)
    ax[0].set_xlabel('tolerance around M_norm = 0', fontsize=10)
    ax[0].axhline(20, color= 'k', linestyle = '--')

    ax[1].plot(res.index, res['mean'], 'o-', color=color, label = 'mean')
    ax[1].plot(res.index, res['median'], 'o-', color=color, linestyle = '--', label = 'median')
    ax[1].legend(fontsize = 15)
    ax[1].set_ylabel('position (nm)', fontsize=10)
    ax[1].set_xlabel('tolerance around M_norm = 0', fontsize=10)

    ax[2].plot(res.index, res['fwhm'], 'o-', color=color)
    ax[2].set_ylabel('FWHM', fontsize=10)
    ax[2].set_xlabel('tolerance around M_norm = 0', fontsize=10)

    if title:
        fig.suptitle(title, fontsize = 15)
    plt.tight_layout()
    plt.show()
    return res

In [ ]:
tol_list = np.linspace(0.1, 0.7, 9)   
res = tolerance_sweep(wall_df, tol_list, category='bloch', voxel_nm=2.0,
                      color=sample_color, title=sample_name)

#### Bimodal distribution

In [ ]:

from scipy.signal import find_peaks

FWHM_K = 2 * np.sqrt(2 * np.log(2))

def _gaussian(x, A, mu, sigma):
    return A * np.exp(-(x - mu)**2 / (2 * sigma**2))

def _bimodal(x, A1, mu1, s1, A2, mu2, s2):
    return _gaussian(x, A1, mu1, s1) + _gaussian(x, A2, mu2, s2)

def _aicc(y, yfit, k):
    """Least-squares AIC with small-sample correction (smaller = better)."""
    n = len(y)
    rss = max(np.sum((y - yfit)**2), 1e-12)
    aic = n * np.log(rss / n) + 2 * k
    return aic + 2 * k * (k + 1) / (n - k - 1) if n - k - 1 > 0 else np.nan


def bloch_z_histogram(df_cat, category='bloch', voxel_nm=2.0, y_mode='count'):
    """
    One bin per z-layer. Returns x (nm from film center), y, bin width.
    y_mode: 'count'    = number of `category` vertices per layer
            'fraction' = fraction of the layer's vertices that are `category`
                         (removes bias from uneven vertex counts per layer)
    """
    layer = df_cat['iz'].round().astype(int)
    layers = np.arange(layer.min(), layer.max() + 1)
    x = (layers - 0.5 * (layers[0] + layers[-1])) * voxel_nm
    is_cat = (df_cat['category'] == category)
    agg = is_cat.groupby(layer).sum() if y_mode == 'count' else is_cat.groupby(layer).mean()
    y = agg.reindex(layers).fillna(0).to_numpy(float)
    return x, y, voxel_nm


def fit_bloch_distribution(x, y, width, daicc_min=10.0, d_min=2.0):
    """
    Descriptive statistics + single-Gaussian fit + two-Gaussian (bimodal) fit.

    Bimodal output: mu1 < mu2, sigma/FWHM of each peak, weight of each peak
    (share of the area), separation, Ashman's D, and delta_aicc (single minus
    bimodal; positive favors bimodal).

    bimodal_supported = (delta_aicc >= daicc_min) and (ashman_d >= d_min).
    Both thresholds are conventions, not hard rules.
    """
    out = dict(mean=np.nan, median=np.nan, variance=np.nan,
               mu_single=np.nan, fwhm_single=np.nan, r2_single=np.nan,
               mu1=np.nan, mu2=np.nan, sigma1=np.nan, sigma2=np.nan,
               fwhm1=np.nan, fwhm2=np.nan, w1=np.nan, w2=np.nan,
               separation=np.nan, ashman_d=np.nan, r2_bimodal=np.nan,
               delta_aicc=np.nan, bimodal_supported=False)
    if y.sum() <= 0:
        return out, None, None

    # whole-distribution statistics (weighted by bar height)
    mean = np.sum(x * y) / y.sum()
    var = np.sum(y * (x - mean)**2) / y.sum()
    edges = np.concatenate([x - width / 2, [x[-1] + width / 2]])
    cum = np.concatenate([[0], np.cumsum(y)]) / y.sum()
    out.update(mean=mean, median=float(np.interp(0.5, cum, edges)), variance=var)

    lo, hi = x.min() - width, x.max() + width
    smin, smax = 0.5 * width, x.max() - x.min()   # sigma below half a bin is unresolvable
    ss_tot = np.sum((y - y.mean())**2)
    r2 = lambda yf: 1 - np.sum((y - yf)**2) / ss_tot if ss_tot > 0 else np.nan

    # ---- single Gaussian ----
    popt1 = None
    try:
        popt1, _ = curve_fit(_gaussian, x, y,
                             p0=[y.max(), mean, max(np.sqrt(var), width)],
                             bounds=([0, lo, smin], [np.inf, hi, smax]), maxfev=20000)
        y1 = _gaussian(x, *popt1)
        out.update(mu_single=popt1[1], fwhm_single=FWHM_K * popt1[2], r2_single=r2(y1))
        aicc1 = _aicc(y, y1, k=3)
    except Exception as e:
        print(f"  single fit failed ({e})")
        aicc1 = np.nan

    # ---- two Gaussians: several starting points, keep the best ----
    s0 = max(width, np.sqrt(var) / 2)
    starts = []
    pk, _ = find_peaks(y, prominence=0.05 * y.max())
    if len(pk) >= 2:
        top2 = np.sort(pk[np.argsort(y[pk])[-2:]])
        starts.append([y[top2[0]], x[top2[0]], s0, y[top2[1]], x[top2[1]], s0])
    left, right = x < mean, x >= mean
    if left.any() and right.any() and y[left].sum() > 0 and y[right].sum() > 0:
        starts.append([y[left].max(), np.average(x[left], weights=y[left]), s0,
                       y[right].max(), np.average(x[right], weights=y[right]), s0])
    q1, q3 = np.interp([0.25, 0.75], cum, edges)
    starts.append([y.max() / 2, q1, s0, y.max() / 2, q3, s0])

    best, best_rss = None, np.inf
    for p0 in starts:
        try:
            p, _ = curve_fit(_bimodal, x, y, p0=p0,
                             bounds=([0, lo, smin, 0, lo, smin],
                                     [np.inf, hi, smax, np.inf, hi, smax]), maxfev=20000)
            rss = np.sum((y - _bimodal(x, *p))**2)
            if rss < best_rss:
                best, best_rss = p, rss
        except Exception:
            continue

    popt2 = None
    if best is not None:
        A1, m1, s1, A2, m2, s2 = best
        if m1 > m2:                                   # order peaks left to right
            A1, m1, s1, A2, m2, s2 = A2, m2, s2, A1, m1, s1
        popt2 = np.array([A1, m1, s1, A2, m2, s2])
        a1, a2 = A1 * s1, A2 * s2                     # area ~ A * sigma
        y2 = _bimodal(x, *popt2)
        sep = m2 - m1
        D = np.sqrt(2) * sep / np.sqrt(s1**2 + s2**2)
        dA = aicc1 - _aicc(y, y2, k=6)
        out.update(mu1=m1, mu2=m2, sigma1=s1, sigma2=s2,
                   fwhm1=FWHM_K * s1, fwhm2=FWHM_K * s2,
                   w1=a1 / (a1 + a2), w2=a2 / (a1 + a2),
                   separation=sep, ashman_d=D, r2_bimodal=r2(y2), delta_aicc=dA,
                   bimodal_supported=bool(np.isfinite(dA) and dA >= daicc_min and D >= d_min))
    return out, popt1, popt2


def plot_bloch_vs_thickness(df_cat, category='bloch', voxel_nm=2.0, y_mode='count',
                            color='magenta', show_single=True, show_bimodal=True,
                            ax=None, title=None):
    """
    Histogram of where `category` vertices fall through the thickness, with
    single- and two-Gaussian fits. df_cat needs 'iz' and 'category'
    (e.g. wall_df after classify_pure_bloch_category). Pass ax to draw into
    an existing axis (e.g. your GridSpec layout).
    """
    x, y, width = bloch_z_histogram(df_cat, category, voxel_nm, y_mode)
    st, p1, p2 = fit_bloch_distribution(x, y, width)

    own_fig = ax is None
    if own_fig:
        fig, ax = plt.subplots(figsize=(7, 5))

    ax.bar(x, y, width=0.9 * width, color=color, alpha=0.45, edgecolor='gray',
           label=f'{category} histogram')
    ax.axvline(0, color='k', ls='--', lw=0.8, label='film center')
    xs = np.linspace(x.min() - width / 2, x.max() + width / 2, 400)

    if show_single and p1 is not None:
        ax.plot(xs, _gaussian(xs, *p1), color='gray', ls='--', lw=1.5,
                label=f"single: μ={st['mu_single']:.1f}, FWHM={st['fwhm_single']:.1f} nm "
                      f"(R²={st['r2_single']:.2f})")
    if show_bimodal and p2 is not None:
        ax.plot(xs, _bimodal(xs, *p2), color='k', lw=2,
                label=f"bimodal sum (R²={st['r2_bimodal']:.2f})")
        for i, c in enumerate(['tab:blue', 'tab:red']):
            A, mu, s = p2[3 * i: 3 * i + 3]
            ax.plot(xs, _gaussian(xs, A, mu, s), color=c, ls='-.', lw=1.5,
                    label=f"peak {i+1}: μ={mu:.1f} nm, FWHM={FWHM_K*s:.1f} nm, "
                          f"weight={st[f'w{i+1}']*100:.0f}%")
            ax.axvline(mu, color=c, lw=1, ls=':')
        verdict = 'supported' if st['bimodal_supported'] else 'NOT supported'
        ax.plot([], [], ' ', label=f"separation={st['separation']:.1f} nm, "
                                   f"Ashman D={st['ashman_d']:.2f}, "
                                   f"ΔAICc={st['delta_aicc']:.1f} → bimodal {verdict}")

    ax.set_xlabel('distance from film center (nm)')
    ax.set_ylabel('number of vertices' if y_mode == 'count' else 'fraction of layer vertices')
    ax.set_xticks(x[::max(1, len(x) // 8)])
    ax.legend(fontsize=8, loc='upper left')
    if title:
        ax.set_title(title)
    if own_fig:
        plt.tight_layout()
        plt.show()
    return st


def tolerance_sweep(wall_df, tol_list, category='bloch', voxel_nm=2.0, method='normal',
                    y_mode='count', color='magenta', title=None):
    rows = []
    for tol in tol_list:
        df_t, n_b, n_o = classify_pure_bloch_category(wall_df.copy(), method=method, tol=tol)
        x, y, width = bloch_z_histogram(df_t, category, voxel_nm, y_mode)
        st, _, _ = fit_bloch_distribution(x, y, width)
        st.update(tol=tol, n_cat=n_b, pct_bloch=100 * n_b / (n_b + n_o))
        rows.append(st)
        print(f"tol={tol:.2f}: n={n_b}, mu1={st['mu1']:.1f}, mu2={st['mu2']:.1f}, "
              f"FWHM1={st['fwhm1']:.1f}, FWHM2={st['fwhm2']:.1f}, w1={st['w1']:.2f}, "
              f"D={st['ashman_d']:.2f}, dAICc={st['delta_aicc']:.1f}, "
              f"bimodal={'yes' if st['bimodal_supported'] else 'no'}")
    res = pd.DataFrame(rows).set_index('tol')

    fig, axes = plt.subplots(2, 3, figsize=(15, 8), sharex=True)
    ok = res['bimodal_supported'].to_numpy()
    t = res.index.to_numpy()

    ax = axes[0, 0]   # peak positions; hollow markers where bimodal is not supported
    for mu, sg, c, lab in [('mu1', 'sigma1', 'tab:blue', 'peak 1'),
                           ('mu2', 'sigma2', 'tab:red', 'peak 2')]:
        ax.errorbar(t, res[mu], yerr=res[sg], fmt='-', color=c, capsize=3, label=f'{lab} ± σ')
        ax.plot(t[ok], res[mu][ok], 'o', color=c)
        ax.plot(t[~ok], res[mu][~ok], 'o', mfc='white', color=c)
    ax.plot(t, res['mean'], 'k--', lw=1, label='overall mean')
    ax.axhline(0, color='gray', ls=':')
    ax.set_ylabel('peak position (nm from center)')
    ax.legend(fontsize=8)

    ax = axes[0, 1]
    ax.plot(t, res['fwhm1'], 'o-', color='tab:blue', label='peak 1')
    ax.plot(t, res['fwhm2'], 'o-', color='tab:red', label='peak 2')
    ax.plot(t, res['fwhm_single'], 's--', color='gray', label='single Gaussian')
    ax.set_ylabel('FWHM (nm)')
    ax.legend(fontsize=8)

    ax = axes[0, 2]
    ax.plot(t, res['w1'], 'o-', color='tab:blue', label='peak 1')
    ax.plot(t, res['w2'], 'o-', color='tab:red', label='peak 2')
    ax.set_ylabel('weight (share of area)')
    ax.set_ylim(0, 1)
    ax.legend(fontsize=8)

    ax = axes[1, 0]
    ax.plot(t, res['separation'], 'o-', color=color)
    ax.set_ylabel('peak separation (nm)')

    ax = axes[1, 1]
    ax.plot(t, res['ashman_d'], 'o-', color=color)
    ax.axhline(2, color='gray', ls=':', label='D = 2 (clean separation)')
    ax.set_ylabel("Ashman's D")
    ax.legend(fontsize=8)

    ax = axes[1, 2]
    ax.plot(t, res['delta_aicc'], 'o-', color=color)
    ax.axhline(10, color='gray', ls=':', label='ΔAICc = 10')
    ax.axhline(0, color='k', lw=0.8)
    ax.set_ylabel('ΔAICc (single − bimodal)\n> 0 favors bimodal')
    ax.legend(fontsize=8)

    for a in axes.ravel():
        a.grid(alpha=0.3)
    for a in axes[1]:
        a.set_xlabel('tolerance on |M_n_n|')
    if title:
        fig.suptitle(title)
    plt.tight_layout()
    plt.show()
    return res

In [ ]:
# one tolerance, histogram with both fits
wall_df, n_bloch, n_other = classify_pure_bloch_category(wall_df, method='normal', tol=0.2)
st = plot_bloch_vs_thickness(wall_df, category='bloch', voxel_nm=2.0, title=sample_name)

# sweep
tol_list = np.linspace(0.1, 0.5, 9)
res = tolerance_sweep(wall_df, tol_list, category='bloch', voxel_nm=2.0,
                      color=sample_color, title=sample_name)

In [ ]:
stop

#### Most complicated, AI reccomended analysis

In [ ]:
# def vertex_areas(verts_xyz, faces):
#     """Area of mesh attributed to each vertex (1/3 of each adjacent triangle)."""
#     v = np.asarray(verts_xyz, float)
#     tri = v[faces]                                    # (M, 3, 3)
#     tri_area = 0.5 * np.linalg.norm(
#         np.cross(tri[:, 1] - tri[:, 0], tri[:, 2] - tri[:, 0]), axis=1)
#     area = np.zeros(len(v))
#     for k in range(3):
#         np.add.at(area, faces[:, k], tri_area / 3.0)
#     return area



# def _binned_cluster_stats(bin_idx, z, w, cl, n_bins, n_boot, rng):
#     """
#     Weighted mean of z per kappa bin, with cluster-bootstrap uncertainty.
#     bin_idx : int bin index per point; z : value; w : weight; cl : compact int cluster id
#     Returns mean (n_bins,), err (n_bins,), n_points (n_bins,), n_clusters (n_bins,), boots
#     """
#     n_cl = cl.max() + 1
#     Szw = np.zeros((n_cl, n_bins))
#     Sw = np.zeros((n_cl, n_bins))
#     np.add.at(Szw, (cl, bin_idx), w * z)
#     np.add.at(Sw, (cl, bin_idx), w)

#     with np.errstate(invalid='ignore', divide='ignore'):
#         mean = Szw.sum(0) / Sw.sum(0)
#         boots = np.empty((n_boot, n_bins))
#         for i in range(n_boot):
#             idx = rng.integers(0, n_cl, n_cl)
#             boots[i] = Szw[idx].sum(0) / Sw[idx].sum(0)
#     err = np.nanstd(boots, axis=0)
#     n_pts = np.bincount(bin_idx, minlength=n_bins)
#     n_clusters = (Sw > 0).sum(0)
#     return mean, err, n_pts, n_clusters, boots

# def plot_bloch_v_thickness_fancy(wall_df,
#                            category='bloch', mn_col='M_n_n', area_col='v_area',
#                            z_source='z', voxel_nm=2.0, z_bottom_nm=None,
#                            exclude_boundary=True, edge_tol=1.0,
#                            n_zbins=None, min_count=10, n_boot=500, seed=0,
#                            show_points=True, max_points=20000,
#                            color='magenta', title=None):
#     """
#     Parameters
#     ----------
#     wall_df : DataFrame
#         All wall vertices. Needs 'x', 'y', the z column(s) described below,
#         `mn_col` (the M component along the wall normal) and `area_col`.
#     classified_df : DataFrame
#         Output of classify_pure_bloch(...); needs a 'category' column and an
#         index that matches wall_df's.
#     thickness_nm : float, optional
#         Film thickness in nm. Default: range of z in wall_df.
#     category : str, default 'bloch'
#         Which category of classified_df marks "M_normal = 0 is met".
#     mn_col : str, default 'M_normal'
#         Column holding M_normal. Looked up in wall_df first, then classified_df.
#     area_col : str or None, default 'v_area'
#         Per-vertex area weights. None -> every vertex weighted equally.
#     z_source : {'z', 'iz'}, default 'z'
#         'z'  : use wall_df['z'] (already in nm).
#         'iz' : use wall_df['iz'] * voxel_nm.
#     voxel_nm : float, default 2.0
#         Voxel spacing in nm (also sets the default z-bin width and the (x, y)
#         column size used for the cluster bootstrap).
#     z_bottom_nm : float, optional
#         z (nm) of the bottom film surface. Default: the minimum z in wall_df
#         (same convention as plot_bloch_v_curvature). Position 0 is then
#         z_bottom_nm + thickness_nm / 2.
#     exclude_boundary, edge_tol : drop vertices within edge_tol (nm) of the
#         lateral (x, y) edges of the volume. z edges are deliberately KEPT:
#         the film surfaces are where the Neel caps live, and cutting them
#         would hide the ends of the M_normal profile.
#     n_zbins : int, optional
#         Number of z bins (both panels). Default: one per voxel.
#     min_count : int
#         Left panel: bins with fewer than this many vertices are not drawn.
#     n_boot, seed : cluster-bootstrap repetitions and RNG seed.
#     show_points, max_points : draw (up to max_points of) the raw vertices.
#     color : color of the binned-mean line.
#     title : str, optional
 
#     Returns
#     -------
#     profile_summary : DataFrame, per z-bin M_normal mean/error/counts (left panel)
#     zero_summary    : dict, weighted mean/std/error (nm, relative to film center),
#                       n points and n columns of the M_normal = 0 locations
#     zero_df         : DataFrame, the vertices used for the histogram
#                       (columns include 'zc' = nm from film center, 'w' = weight)
#     """
#     rng = np.random.default_rng(seed)
 
#     # ---------- gather columns ----------
#     def _get(col):
#         if col in wall_df.columns:
#             return wall_df[col]
#         if col in wall_df.columns:
#             return wall_df[col].reindex(wall_df.index)
#         raise KeyError(f"column '{col}' not found in wall_df or classified_df. "
#                        f"wall_df columns: {list(wall_df.columns)}")
 

#     ## Define wall thickness 
#     # The "0" of the x axis is z_bottom + thickness/2, so it is only as good as
#     # z_bottom. Print the numbers so a mismatch is visible, not silent.
#     z_nm = wall_df['z'].astype(float)
#     z_bottom = z_nm.min()
#     print(f'z_bottom = {z_bottom}')

#     thickness_nm = float(z_nm.max() - z_bottom)
#     print(f'thickness_nm = {thickness_nm}')

#     span = float(z_nm.max() - z_nm.min())
#     print(f"wall_df z range: [{z_nm.min():.2f}, {z_nm.max():.2f}] nm (span {span:.2f} nm); "
#           f"thickness_nm = {thickness_nm:.2f}; x = 0 placed at z = {z_bottom + thickness_nm / 2:.2f} nm")
    
#     ## create a Wall data frame 
#     W = pd.DataFrame({'x': wall_df['x'], 'y': wall_df['y'],
#                       'zc': z_nm - z_bottom - thickness_nm / 2.0,   # 0 = film center, zc is z relative to film center
#                       'mn': _get(mn_col)})
#     W['w'] = _get(area_col) if area_col else 1.0


#     print(np.mean(W['y']))
#     print(np.mean(W['y']/ voxel_nm))


#     print(f' zc = {np.mean(W['zc'])}')
#     # ---------- clean to remove the edges  ----------
#     n0 = len(W)
#     if exclude_boundary:
#         on_edge = np.zeros(len(W), dtype=bool)
#         for c in ('x', 'y'):
#             lo, hi = wall_df[c].min(), wall_df[c].max()
#             on_edge |= ((W[c] <= lo + edge_tol) | (W[c] >= hi - edge_tol)).to_numpy()
#         W = W[~on_edge]
#         print(f"excluded {n0 - len(W)} / {n0} wall vertices on the lateral (x, y) volume boundary")
 
#     W = W.dropna(subset=['mn', 'zc', 'w'])
#     W = W[W['w'] > 0].copy()
#     W['cx'] = np.rint(W['x'] / voxel_nm).astype(int)
#     W['cy'] = np.rint(W['y'] / voxel_nm).astype(int)
#     W['col_id'] = W.groupby(['cx', 'cy']).ngroup()
 
#     # ---------- left panel: M_normal vs z ----------
#     if n_zbins is None:
#         n_zbins = max(int(round(thickness_nm / voxel_nm)), 1)
#         print((f'n_zbins = {n_zbins}'))

#     zedges = np.linspace(-thickness_nm / 2.0, thickness_nm / 2.0, n_zbins + 1)
#     zcent = 0.5 * (zedges[:-1] + zedges[1:])
 
#     zc, mn = W['zc'].to_numpy(), W['mn'].to_numpy() # z and Mn values
#     w, cl = W['w'].to_numpy(), W['col_id'].to_numpy() # area weigth (decided as best way to handle the vertex spacing) and cl as column id (x,y)
#     bi = np.floor((zc - zedges[0]) / (zedges[-1] - zedges[0]) * n_zbins).astype(int)
#     bi[zc == zedges[-1]] = n_zbins - 1
#     ok = (bi >= 0) & (bi < n_zbins)
 
#     mean, err, n_pts, n_cl, _ = _binned_cluster_stats(
#         bi[ok], mn[ok], w[ok], cl[ok], n_zbins, n_boot, rng)
#     keep = n_pts >= min_count
#     profile_summary = pd.DataFrame({'z_center_nm': zcent, 'mn_mean': mean, 'mn_err': err,
#                                     'n_points': n_pts, 'n_columns': n_cl, 'shown': keep})
 
#     # ---------- right panel: where M_normal = 0 ----------
#     zero_idx = wall_df.index[wall_df['category'] == category]
#     Z = W.loc[W.index.intersection(zero_idx)]
#     if len(Z) == 0:
#         raise ValueError(f"no '{category}' vertices left after cleaning -- "
#                          "check `category` / `tol` / boundary exclusion.")
#     zb, wb, clb = Z['zc'].to_numpy(), Z['w'].to_numpy(), Z['col_id'].to_numpy()
#     m, e, n_z, ncl_z, _ = _binned_cluster_stats(
#         np.zeros(len(zb), dtype=int), zb, wb, clb, 1, n_boot, rng)
#     z_mean, z_err = float(m[0]), float(e[0])
#     z_std = float(np.sqrt(np.sum(wb * (zb - z_mean) ** 2) / np.sum(wb)))
#     zero_summary = {'mean_nm': z_mean, 'err_nm': z_err, 'std_nm': z_std,
#                     'n_points': int(n_z[0]), 'n_columns': int(ncl_z[0])}
 
#     # ---------- plot ----------
#     fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5), sharex=True)
 
#     if show_points:
#         sel = rng.choice(len(zc), size=min(max_points, len(zc)), replace=False)
#         ax1.scatter(zc[sel], mn[sel], s=3, color='gray', alpha=0.08,
#                     rasterized=True, zorder=0, label='wall vertices')
#     ax1.errorbar(zcent[keep], mean[keep], yerr=err[keep], fmt='o-', capsize=3,
#                  color=color, label='area-weighted mean ± bootstrap')
#     ax1.axhline(0, color='k', ls=':', lw=0.8, label=f'{mn_col} = 0')
#     ax1.axvline(0, color='gray', ls='--', lw=0.8, label='film center')
#     ax1.axvline(z_mean, color=color, ls='-', lw=0.8, alpha=0.4)
#     ax1.set_xlabel('position through thickness (nm), 0 = film center')
#     ax1.set_ylabel(mn_col)
#     ax1.set_title(f'{mn_col} vs. thickness')
#     ax1.legend(fontsize=8)
 
#     ax2.hist(zb, bins=zedges, weights=wb, density=True,
#              color=color, alpha=0.6, edgecolor='k', lw=0.5)
#     ax2.axvline(0, color='gray', ls='--', lw=0.8, label='film center')
#     ax2.axvline(z_mean, color='k', lw=1.5,
#                 label=f'mean = {z_mean:.2f} ± {z_err:.2f} nm')
#     ax2.set_xlabel('position through thickness (nm), 0 = film center')
#     ax2.set_ylabel('area-weighted density')
#     ax2.set_title(f'where {mn_col} = 0 is met ({category} vertices)')
#     ax2.legend(fontsize=8)
 
#     ax1.set_xlim(zedges[0], zedges[-1])
#     if title:
#         fig.suptitle(title)
#     plt.tight_layout()
#     plt.show()
 
#     # ---------- printed summary ----------
#     print(f"\n{category} core position (area-weighted, cluster-bootstrap error):")
#     print(f"  mean = {z_mean:.3f} ± {z_err:.3f} nm from film center "
#           f"(+ = toward larger z),  std of distribution = {z_std:.3f} nm")
#     print(f"  = {z_mean / thickness_nm + 0.5:.3f} of thickness from the bottom surface")
#     print(f"  {zero_summary['n_points']} vertices in {zero_summary['n_columns']} (x, y) columns")
 
#     return profile_summary, zero_summary, Z

In [ ]:
# wall_df['v_area'] = vertex_areas(verts_xyz, faces)

# prof, zero_summary, zero_df = plot_bloch_v_thickness_fancy(
#     wall_df, 
#     mn_col='M_n_n',      # whatever your M_normal column is called
#     z_source='z',           # or 'iz' (multiplied by voxel_nm)
#     voxel_nm=voxel_nm, 
#     color = sample_color)

## Segment by curvature


In [ ]:
print('run')

### Overlay spline kappa onto isosurfac --- Does not look good

In [ ]:
plt.imshow(curvature['kappa_map'][2], cmap = 'coolwarm', vmin = -0.02, vmax = 0.02)
plt.colorbar()

plt.show()


### compute Kappa on iso-surface centerline and compare to spline kappa

In [ ]:
def vertex_areas(verts_xyz, faces):
    """Area of mesh attributed to each vertex (1/3 of each adjacent triangle)."""
    v = np.asarray(verts_xyz, float)
    tri = v[faces]                                    # (M, 3, 3)
    tri_area = 0.5 * np.linalg.norm(
        np.cross(tri[:, 1] - tri[:, 0], tri[:, 2] - tri[:, 0]), axis=1)
    area = np.zeros(len(v))
    for k in range(3):
        np.add.at(area, faces[:, k], tri_area / 3.0)
    return area


wall_df['v_area'] = vertex_areas(verts_xyz, faces)

#### scratch work

In [ ]:
# import numpy as np
# import matplotlib.pyplot as plt
# from scipy.ndimage import gaussian_filter

pixel_nm = voxel_nm         # in-plane pixel size
Mz = volume['Mz']       # assumed shape (z, y, x)

# average each z column; optionally restrict to the film layers, e.g. slice(z_lo, z_hi + 1)
z_slice = slice(None)
Mz_2d = Mz[z_slice].mean(axis=0)

# light smoothing so the contour (and later the curvature) isn't pixel-jagged
sigma_pix = 3.0 #2.0 #### CHANGE TO 3 IF TOO MUCH NOISE 
Mz_2d_s = gaussian_filter(Mz_2d, sigma_pix)

ny, nx = Mz_2d_s.shape
extent = [0, nx * pixel_nm, 0, ny * pixel_nm]

fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(Mz_2d_s, cmap='magma', origin='lower', extent=extent)
ax.contour(np.arange(nx) * pixel_nm, np.arange(ny) * pixel_nm, Mz_2d_s,
           levels=[0], colors='cyan', linewidths=1.5)
ax.set_xlabel('x (nm)')
ax.set_ylabel('y (nm)')
ax.set_title('z-averaged Mz with zero contour')
plt.show()

print("2D shape (y, x):", Mz_2d.shape)

In [ ]:
from skimage import measure
from scipy.ndimage import gaussian_filter1d

ds_nm = 2.0             # spacing of the resampled points along the wall
smooth_nm =  12 #6.0         # Gaussian smoothing length along the wall #################### CHANGE TO 12 IF TOO MUCH NOISE!!!
min_length_nm = 50.0    # discard contours shorter than this (tiny spurious loops)

# 1) find the zero contour (returns pixel coordinates as (row, col))
raw_contours = measure.find_contours(Mz_2d_s, level=0.0)

contours = []
for c in raw_contours:
    y_raw, x_raw = c[:, 0] * pixel_nm, c[:, 1] * pixel_nm     # to nm
    closed = np.allclose(c[0], c[-1])                         # loop vs. line hitting the edge

    # 2) arc length along the raw contour
    seg = np.hypot(np.diff(x_raw), np.diff(y_raw))
    s_raw = np.concatenate([[0], np.cumsum(seg)])
    L = s_raw[-1]
    if L < min_length_nm:
        continue

    # 3) resample to evenly spaced points
    n = int(round(L / ds_nm))
    if closed:
        s_new = np.linspace(0, L, n, endpoint=False)          # don't repeat the first point
    else:
        s_new = np.linspace(0, L, n + 1)
    x_r = np.interp(s_new, s_raw, x_raw)
    y_r = np.interp(s_new, s_raw, y_raw)

    # 4) smooth along the wall ('wrap' for loops so there's no seam)
    mode = 'wrap' if closed else 'nearest'
    sigma_pts = smooth_nm / (L / n)
    x_s = gaussian_filter1d(x_r, sigma_pts, mode=mode)
    y_s = gaussian_filter1d(y_r, sigma_pts, mode=mode)

    contours.append({'x': x_s, 'y': y_s, 'x_raw': x_r, 'y_raw': y_r,
                     'ds': L / n, 'closed': closed, 'length_nm': L})

print(f"{len(contours)} contours kept (of {len(raw_contours)} found)")
for i, c in enumerate(contours):
    print(f"  contour {i}: {len(c['x'])} points, {c['length_nm']:.0f} nm, "
          f"{'closed loop' if c['closed'] else 'open (touches edge)'}")

# plot: raw (white) vs. smoothed (cyan), numbered
fig, ax = plt.subplots(figsize=(7, 7))
ax.imshow(Mz_2d_s, cmap='magma', origin='lower',
          extent=[-0.5 * pixel_nm, (nx - 0.5) * pixel_nm,
                  -0.5 * pixel_nm, (ny - 0.5) * pixel_nm])
for i, c in enumerate(contours):
    ax.plot(c['x_raw'], c['y_raw'], color='white', lw=0.8, alpha=0.8)
    ax.plot(c['x'], c['y'], color='cyan', lw=1.5)
    ax.text(c['x'][0], c['y'][0], str(i), color='yellow', fontsize=9)
ax.set_xlabel('x (nm)')
ax.set_ylabel('y (nm)')
ax.set_title('Wall centerline: raw (white) vs. smoothed (cyan)')
plt.show()

In [ ]:
from scipy.ndimage import map_coordinates

kappa_clim = 0.02          # color limits for the plot (1/nm)
edge_trim_nm = 10.0        # open contours: drop kappa this close to each end
positive_side = 'high'     # 'high': kappa > 0 means the center of curvature is in the Mz > 0 domain
                           # 'low' : the opposite

def d_ds(f, ds, closed):
    """Central-difference derivative along the contour (wraps around for closed loops)."""
    if closed:
        return (np.roll(f, -1) - np.roll(f, 1)) / (2 * ds)
    return np.gradient(f, ds)

def signed_curvature(x, y, ds, closed):
    xp, yp = d_ds(x, ds, closed), d_ds(y, ds, closed)
    xpp, ypp = d_ds(xp, ds, closed), d_ds(yp, ds, closed)
    return (xp * ypp - yp * xpp) / (xp**2 + yp**2) ** 1.5

for i, c in enumerate(contours):
    x, y, ds, closed = c['x'], c['y'], c['ds'], c['closed']

    # --- fix the travel direction so the sign of kappa has a definite meaning ---
    # Put the chosen domain on the LEFT of the direction of travel. A positive kappa
    # then means the contour curves toward that domain.
    tx, ty = d_ds(x, ds, closed), d_ds(y, ds, closed) #### comput the unit tanget T
    nrm = np.hypot(tx, ty)
    tx, ty = tx / nrm, ty / nrm
    off = 2 * pixel_nm
    xl, yl = x - ty * off, y + tx * off                     # point just left of the wall
    left_Mz = map_coordinates(Mz_2d_s, [yl / pixel_nm, xl / pixel_nm],
                              order=1, mode='nearest')
    want_positive = (positive_side == 'high')
    if (np.mean(left_Mz) > 0) != want_positive:
        x, y = x[::-1].copy(), y[::-1].copy()
        left_Mz = -left_Mz                                  # only for the consistency print
    frac_ok = np.mean((left_Mz > 0) == want_positive)

    # --- curvature ---
    k = signed_curvature(x, y, ds, closed)

    if not closed:                                          # edge effects on open walls
        n_trim = int(round(edge_trim_nm / ds))
        k[:n_trim] = np.nan
        k[-n_trim:] = np.nan

    c['x'], c['y'], c['kappa'] = x, y, k
    print(f"contour {i}: {np.sum(np.isfinite(k))} valid points, "
          f"direction consistent at {frac_ok:.0%} of points, "
          f"kappa range [{np.nanmin(k):.4f}, {np.nanmax(k):.4f}] 1/nm")

# --- plot like the reference image ---
fig, ax = plt.subplots(figsize=(7, 6))
ax.imshow(Mz_2d_s, cmap='viridis', origin='lower', alpha=0.6,
          extent=[-0.5 * pixel_nm, (nx - 0.5) * pixel_nm,
                  -0.5 * pixel_nm, (ny - 0.5) * pixel_nm])
all_x = np.concatenate([c['x'] for c in contours])
all_y = np.concatenate([c['y'] for c in contours])
all_k = np.concatenate([c['kappa'] for c in contours])
sc = ax.scatter(all_x, all_y, c=all_k, cmap='coolwarm', s=6,
                vmin=-kappa_clim, vmax=kappa_clim)
plt.colorbar(sc, label='κ (1/nm)')
ax.set_xlabel('x (nm)')
ax.set_ylabel('y (nm)')
ax.set_title('Center line curvature')
ax.set_aspect('equal')
plt.show()

print("\nkappa summary (1/nm):")
print(pd.Series(all_k).describe().to_string())

In [ ]:
max_dist_nm = 10 * voxel_nm    # pixels farther than this from the contour stay NaN

valid = np.isfinite(all_k)
pts = np.column_stack([all_x[valid], all_y[valid]])
kap = all_k[valid]

# nearest contour point for every pixel (pixel centers in nm)
yy, xx = np.mgrid[0:ny, 0:nx]
grid_xy = np.column_stack([xx.ravel() * pixel_nm, yy.ravel() * pixel_nm])
dist, j = cKDTree(pts).query(grid_xy)

kappa_map = kap[j].reshape(ny, nx)
dist_map = dist.reshape(ny, nx)
kappa_map[dist_map > max_dist_nm] = np.nan

# plt.imshow(kappa_map, cmap='coolwarm', origin='lower', vmin=-kappa_clim, vmax=kappa_clim)
# plt.colorbar(label='κ (1/nm)')
# plt.title('rasterized kappa_map')
# plt.show()

In [ ]:
nz = volume['Mz'].shape[0]

kappa_vol = np.repeat(np.nan_to_num(kappa_map)[None], nz, axis=0)             # NaN -> 0
valid_vol = np.repeat(np.isfinite(kappa_map)[None].astype(float), nz, axis=0)
dist_vol  = np.repeat(dist_map[None], nz, axis=0)

num = sample_volume_at_points(kappa_vol, verts_idx)
den = sample_volume_at_points(valid_vol, verts_idx)

if 'kappa' in wall_df.columns:
    wall_df['kappa_old'] = wall_df['kappa']

wall_df['kappa'] = np.where(den > 0.5, num / np.maximum(den, 1e-9), np.nan)
wall_df['kappa_dist_nm'] = sample_volume_at_points(dist_vol, verts_idx)

print(f"kappa is NaN for {wall_df['kappa'].isna().mean():.1%} of vertices")

# top-down check
fig, ax = plt.subplots(figsize=(7, 6))
sc = ax.scatter(wall_df['x'], wall_df['y'], c=wall_df['kappa'], cmap='coolwarm',
                s=1, vmin=-kappa_clim, vmax=kappa_clim)
plt.colorbar(sc, label='κ (1/nm)')
ax.set_aspect('equal')
ax.set_title('Wall vertices colored by sampled κ (top-down)')
plt.show()

### Plot on 3D

In [ ]:
sargs = dict(
    position_x=0.4,  # Move towards the right edge
    position_y=0.85,   # Move near the bottom
    height=0.05,       # Make it take up 50% of window height
    width=0.5,        # Width of the bar
    vertical=False,    # Orientation
    title = f'{sample_name} \n Kappa'
)

psi_plot_dict = dict(
    view= wall_df['kappa'].values, 
    bounds = [-0.02, 0.02],
    color_map = 'coolwarm', 
    zoom = 1
)

plot_map(volume, verts_xyz, faces, psi_plot_dict, sargs, show_grid = True, field = 'Mz', axis = 'z')

### Map Bloch mask to curvature

In [ ]:
plot_pure_bloch_3d(verts_xyz, wall_df, kappa=wall_df['kappa'].to_numpy(), tol=0.005, ax=None)

segment with curvature

In [ ]:
# def plot_bloch_v_curvature(wall_df, thickness_nm,
#                            category='bloch', category_col='category',
#                            curv_col='kappa', area_col='v_area',
#                            voxel_nm=2.0, color = 'green', exclude_boundary=False, edge_tol=1.0,
#                            max_dist_nm=None, dist_col='kappa_dist_nm',
#                            n_bins=10, kappa_range=None, kappa_tol=0.002,
#                            min_count=10, cluster_by='column', segment_nm=20.0, ds_nm=2.0,
#                            n_boot=500, seed=0, ylim=None, title=None):
#     """
#     Area-weighted mean z (fraction of thickness) vs. centerline curvature kappa,
#     with cluster-bootstrap error bars, plus a three-class summary (kappa < -tol,
#     |kappa| <= tol, kappa > tol).

#     category_col : column holding the classification ('category', or 'category_normal'
#                    if you reloaded the saved wall_df).
#     kappa_range  : (lo, hi) for the bins. None = symmetric, set by the 98th percentile of |kappa|.
#     max_dist_nm  : if given, drop vertices whose nearest contour point (dist_col) is farther
#                    than this. Lets you tighten the cutoff from step 3b without recomputing.
#     cluster_by   : 'column' resamples lateral (x, y) pixel columns.
#                    'contour_segment' resamples ~segment_nm stretches of centerline
#                    (needs wall_df['contour_pt'], see note below). This is more honest now,
#                    because many neighboring columns share the same contour point.
#     ds_nm        : contour point spacing from step 2 (used to convert segment_nm to points).
#     Returns (summary, class_summary, d); requires _binned_cluster_stats from before.
#     """
#     rng = np.random.default_rng(seed)
#     z_min = wall_df['z'].min()

#     cols = ['x', 'y', 'z', curv_col, area_col]
#     if cluster_by == 'contour_segment':
#         cols.append('contour_pt')
#     if max_dist_nm is not None:
#         cols.append(dist_col)
#     d = wall_df.loc[wall_df[category_col] == category, cols].copy()

#     n0 = len(d)
#     if exclude_boundary:
#         on_edge = np.zeros(len(d), dtype=bool)
#         for c in ['x', 'y', 'z']:
#             lo, hi = wall_df[c].min(), wall_df[c].max()
#             on_edge |= (d[c] <= lo + edge_tol) | (d[c] >= hi - edge_tol)
#         d = d[~on_edge]
#         print(f"excluded {n0 - len(d)} / {n0} {category} vertices on the volume boundary")

#     n1 = len(d)
#     d = d.dropna(subset=[curv_col, 'z', area_col])
#     d = d[d[area_col] > 0]
#     if max_dist_nm is not None:
#         d = d[d[dist_col] <= max_dist_nm]
#     d = d.copy()
#     print(f"dropped {n1 - len(d)} more vertices with no valid kappa / area / distance; "
#           f"{len(d)} remain")

#     d['z_frac'] = (d['z'] - z_min) / thickness_nm

#     # ---- cluster ids for the bootstrap ----
#     if cluster_by == 'column':
#         d['cx'] = np.rint(d['x'] / voxel_nm).astype(int)
#         d['cy'] = np.rint(d['y'] / voxel_nm).astype(int)
#         d['col_id'] = d.groupby(['cx', 'cy']).ngroup()
#     elif cluster_by == 'contour_segment':
#         step = max(int(round(segment_nm / ds_nm)), 1)
#         d['col_id'] = pd.factorize(d['contour_pt'] // step)[0]
#     else:
#         raise ValueError("cluster_by must be 'column' or 'contour_segment'")
#     print(f"{d['col_id'].nunique()} bootstrap clusters (cluster_by='{cluster_by}')")

#     z = d['z_frac'].to_numpy()
#     w = d[area_col].to_numpy()
#     k = d[curv_col].to_numpy()
#     cl = d['col_id'].to_numpy()

#     # ---------- binned mean vs kappa ----------
#     if kappa_range is None:
#         lim = np.percentile(np.abs(k), 98)
#         kappa_range = (-lim, lim)
#     print(f"kappa bins span {kappa_range[0]:.4f} to {kappa_range[1]:.4f} 1/nm")

#     edges = np.linspace(*kappa_range, n_bins + 1)
#     centers = 0.5 * (edges[:-1] + edges[1:])
#     vb = pd.cut(pd.Series(k), edges, labels=False).to_numpy()
#     vm = ~np.isnan(vb)
#     mean, err, n_pts, n_cl, _ = _binned_cluster_stats(
#         vb[vm].astype(int), z[vm], w[vm], cl[vm], n_bins, n_boot, rng)
#     keep = n_pts >= min_count
#     summary = pd.DataFrame({'kappa_center': centers, 'z_frac_mean': mean, 'z_frac_err': err,
#                             'z_nm_mean': mean * thickness_nm, 'n_points': n_pts,
#                             'n_columns': n_cl, 'shown': keep})

#     # ---------- three kappa classes (summary only) ----------
#     cls = np.where(k < -kappa_tol, 0, np.where(k > kappa_tol, 2, 1))
#     names = [f'κ < -{kappa_tol}', f'|κ| ≤ {kappa_tol}', f'κ > {kappa_tol}']
#     c_mean, c_err, c_n, c_ncl, c_boots = _binned_cluster_stats(cls, z, w, cl, 3, n_boot, rng)
#     class_summary = pd.DataFrame({'class': names, 'z_frac_mean': c_mean, 'z_frac_err': c_err,
#                                   'z_nm_mean': c_mean * thickness_nm,
#                                   'n_points': c_n, 'n_clusters': c_ncl})

#     # ---------- plot ----------
#     fig, ax = plt.subplots(figsize=(8, 5))
#     ax.errorbar(centers[keep], mean[keep], yerr=err[keep], fmt='o-', capsize=3,
#                 color=color)
#     ax.axvline(0, color='k', ls=':', lw=0.8, label='kappa = 0')
#     ax.axhline(0.5, color='gray', ls='--', lw=0.8, label='film center')
#     # ax.axvspan(-kappa_tol, kappa_tol, color='gray', alpha=0.12, label='"κ ≈ 0" range')
#     ax.set_xlabel('centerline curvature κ (1/nm)')
#     ax.set_ylabel(f'{category} position (fraction of thickness)')
#     ax.set_title(f'Mean {category} z vs. curvature \n {title}')
#     ax.legend(fontsize=8)
#     if ylim is not None:
#         ax.set_ylim(*ylim)
#     plt.tight_layout()
#     plt.show()

#     # ---------- printed summary ----------
#     print("\nClass summary (area-weighted, cluster-bootstrap error):")
#     print(class_summary.to_string(index=False))
#     diff = c_boots[:, 2] - c_boots[:, 0]
#     print(f"\nmean z_frac(κ>tol) - mean z_frac(κ<-tol) = {np.nanmean(diff):.3f} ± {np.nanstd(diff):.3f}"
#           f"  ({np.nanmean(diff) * thickness_nm:.2f} nm)")
#     return summary, class_summary, d

In [ ]:
def _binned_cluster_stats(bin_idx, z, w, cl, n_bins, n_boot, rng):
    """
    Weighted mean of z per kappa bin, with cluster-bootstrap uncertainty.
    bin_idx : int bin index per point; z : value; w : weight; cl : compact int cluster id
    Returns mean (n_bins,), err (n_bins,), n_points (n_bins,), n_clusters (n_bins,), boots
    """
    n_cl = cl.max() + 1
    Szw = np.zeros((n_cl, n_bins))
    Sw = np.zeros((n_cl, n_bins))
    np.add.at(Szw, (cl, bin_idx), w * z) #### numerator weighted sum
    np.add.at(Sw, (cl, bin_idx), w) #### denominator weighted sum

    with np.errstate(invalid='ignore', divide='ignore'):
        mean = Szw.sum(0) / Sw.sum(0)
        boots = np.empty((n_boot, n_bins))
        for i in range(n_boot):
            idx = rng.integers(0, n_cl, n_cl)
            boots[i] = Szw[idx].sum(0) / Sw[idx].sum(0)
    err = np.nanstd(boots, axis=0)
    n_pts = np.bincount(bin_idx, minlength=n_bins)
    n_clusters = (Sw > 0).sum(0)
    return mean, err, n_pts, n_clusters, boots



def film_geometry(wall_df, voxel_nm=2.0, z_min=None, z_max=None, thickness_nm=None):
    """
    z_min, z_max : first / last vertex (voxel-center) positions. Default: from the data.
    thickness_nm : known film thickness. None = (z_max - z_min) + voxel_nm, i.e. N layers x voxel.
    Center = midpoint of z_min and z_max (unchanged); z = 0 in every plot is this center.
    z_bottom / z_top are the physical film faces (half a voxel beyond the outer layers,
    or +/- thickness/2 about the center if thickness_nm is given).
    """
    print(f'input thickness= {thickness_nm}')
    z_min = wall_df['z'].min() if z_min is None else z_min
    z_max = wall_df['z'].max() if z_max is None else z_max
    z_center = 0.5 * (z_min + z_max)
    if thickness_nm is None:
        thickness_nm = (z_max - z_min) + voxel_nm
    layers = np.arange(int(round(z_min / voxel_nm)), int(round(z_max / voxel_nm)) + 1)
    geom = dict(z_min=z_min, z_max=z_max, z_center=z_center,
                thickness=thickness_nm,
                z_bottom=z_center - thickness_nm / 2, z_top=z_center + thickness_nm / 2,
                voxel_nm=voxel_nm, layers=layers, x_vals=layers * voxel_nm - z_center)
    print(f"film: vertices span {z_min:.1f} to {z_max:.1f} nm, {len(layers)} layers, "
          f"thickness {thickness_nm:.1f} nm, center {z_center:.1f} nm")
    return geom


def _binned_raw_stats(bin_idx, z, w, cl, n_bins, k=None):
    """
    Non-bootstrap companion to _binned_cluster_stats, for checking its assumptions.

    bin_idx : int bin index per point; z : value; w : weight; cl : compact int cluster id
    k       : optional raw kappa per point; if given, also fits a weighted linear trend z ~ k

    Returns mean, err, n_pts, n_clusters, extra
      mean : weighted mean per bin (should match the bootstrap point estimate exactly)
      err  : naive SEM assuming independent points (std / sqrt(n_eff))
      extra (dict):
        std         : raw (weighted) standard deviation of z within each bin
        n_eff       : Kish effective sample size, (sum w)^2 / sum w^2
        sem_cluster : analytic cluster-robust SEM (ratio-estimator linearization),
                      the no-resampling analogue of the bootstrap err
        deff        : (sem_cluster / err)^2, design effect; ~1 means points act independent,
                      >>1 means within-cluster correlation is inflating the true error
        trend       : dict(slope, intercept, se_naive, se_cluster) if k is given
    """
    bin_idx = np.asarray(bin_idx)
    n_cl = cl.max() + 1

    # point-level per-bin sums
    Sw = np.bincount(bin_idx, weights=w, minlength=n_bins)
    Sw2 = np.bincount(bin_idx, weights=w**2, minlength=n_bins)
    Szw = np.bincount(bin_idx, weights=w * z, minlength=n_bins)
    n_pts = np.bincount(bin_idx, minlength=n_bins)

    # per-cluster, per-bin weight sums (for n_clusters and cluster-robust SE)
    Sw_cb = np.zeros((n_cl, n_bins))
    Szw_cb = np.zeros((n_cl, n_bins))
    np.add.at(Sw_cb, (cl, bin_idx), w)
    np.add.at(Szw_cb, (cl, bin_idx), w * z)
    n_clusters = (Sw_cb > 0).sum(0)

    with np.errstate(invalid='ignore', divide='ignore'):
        mean = Szw / Sw

        # two-pass weighted variance, with reliability-weight small-sample correction
        dev2 = (z - mean[bin_idx]) ** 2
        var_pop = np.bincount(bin_idx, weights=w * dev2, minlength=n_bins) / Sw
        n_eff = Sw**2 / Sw2
        std = np.sqrt(var_pop * n_eff / (n_eff - 1))
        err = std / np.sqrt(n_eff)

        # cluster-robust SEM: residual of each cluster's contribution to the ratio estimator
        resid = Szw_cb - mean * Sw_cb
        sem_cluster = np.sqrt(n_cl / (n_cl - 1) * (resid**2).sum(0)) / Sw
        deff = (sem_cluster / err) ** 2

    extra = dict(std=std, n_eff=n_eff, sem_cluster=sem_cluster, deff=deff)

    if k is not None:
        k = np.asarray(k, dtype=float)
        W = w.sum()
        kbar = (w * k).sum() / W
        zbar = (w * z).sum() / W
        kc = k - kbar
        Sxx = (w * kc**2).sum()
        slope = (w * kc * (z - zbar)).sum() / Sxx
        intercept = zbar - slope * kbar
        e = z - (intercept + slope * k)
        n = len(z)
        se_naive = np.sqrt((w * e**2).sum() / (n - 2) / Sxx)
        score_c = np.bincount(cl, weights=w * kc * e, minlength=n_cl)  # per-cluster score
        se_cluster = np.sqrt(n_cl / (n_cl - 1) * (score_c**2).sum()) / Sxx
        extra['trend'] = dict(slope=slope, intercept=intercept,
                              se_naive=se_naive, se_cluster=se_cluster)

    return mean, err, n_pts, n_clusters, extra


if __name__ == '__main__':
    # synthetic test: spatially correlated blocks => cluster effects, known slope
    rng = np.random.default_rng(0)
    n_cl, per = 60, 80
    cl = np.repeat(np.arange(n_cl), per)
    k = rng.normal(0, 1, n_cl * per)
    z = 0.5 * k + rng.normal(0, 1, n_cl)[cl] + rng.normal(0, 1, n_cl * per)  # block offset + noise
    w = np.ones_like(z)
    n_bins = 8
    edges = np.linspace(-2, 2, n_bins + 1)
    vb = np.digitize(k, edges) - 1
    m = (vb >= 0) & (vb < n_bins)

    mean, err, n_pts, n_c, ex = _binned_raw_stats(vb[m], z[m], w[m], cl[m], n_bins, k=k[m])

    # reference: cluster bootstrap on the same data
    Szw = np.zeros((n_cl, n_bins)); Sw = np.zeros((n_cl, n_bins))
    np.add.at(Szw, (cl[m], vb[m]), z[m]); np.add.at(Sw, (cl[m], vb[m]), 1.0)
    boots = np.array([Szw[i].sum(0) / Sw[i].sum(0)
                      for i in (rng.integers(0, n_cl, n_cl) for _ in range(2000))])
    err_boot = boots.std(0)

    print('mean matches plain mean:', np.allclose(mean[0], z[m][vb[m] == 0].mean()))
    print('bin  n_pts  mean    naive   cluster  boot    deff')
    for b in range(n_bins):
        print(f'{b:3d} {n_pts[b]:6d} {mean[b]:7.3f} {err[b]:7.3f} {ex["sem_cluster"][b]:8.3f} '
              f'{err_boot[b]:7.3f} {ex["deff"][b]:6.2f}')
    print('trend:', {a: round(float(v), 4) for a, v in ex['trend'].items()})


def plot_bloch_v_curvature(wall_df, geom=None, kappa_tol=0.002, n_bins=10, thickness = None, 
                           category='bloch', category_col='category',
                           curv_col='kappa', area_col='v_area',
                           voxel_nm=2.0, color='green', exclude_boundary=True, edge_tol=1.0,
                           max_dist_nm=None, dist_col='kappa_dist_nm',
                           kappa_range=None, 
                           min_count=10, cluster_by='column', segment_nm=20.0, ds_nm=2.0,
                           n_boot=500, seed=0, ylim=None, title=None):
    """
    Area-weighted mean z (nm relative to the film center) vs. centerline curvature kappa,
    with cluster-bootstrap error bars, plus a three-class summary (kappa < -tol,
    |kappa| <= tol, kappa > tol).

    thickness_nm : film thickness in nm. Only used to locate the film center
                   (and for the z_frac columns in the outputs).
    z_center_nm  : z of the film center in the same units as wall_df['z'].
                   None = wall_df['z'].min() + thickness_nm / 2.
    ylim         : (lo, hi) in nm relative to the film center, e.g. (-10, 5).
    category_col : column holding the classification ('category', or 'category_normal'
                   if you reloaded the saved wall_df).
    kappa_range  : (lo, hi) for the bins. None = symmetric, set by the 98th percentile of |kappa|.
    max_dist_nm  : if given, drop vertices whose nearest contour point (dist_col) is farther
                   than this.
    cluster_by   : 'column' resamples lateral (x, y) pixel columns.
                   'contour_segment' resamples ~segment_nm stretches of centerline
                   (needs wall_df['contour_pt']).
    ds_nm        : contour point spacing from step 2 (used to convert segment_nm to points).
    Returns (summary, class_summary, d); requires _binned_cluster_stats from before.
    """

    ds_nm = voxel_nm
    print(f'ds_nm = {ds_nm}')
    rng = np.random.default_rng(seed)
    if geom is None:
        geom = film_geometry(wall_df, voxel_nm, thickness_nm = thickness)
    z_min, thickness_nm, z_center_nm = geom['z_bottom'], geom['thickness'], geom['z_center']

    cols = ['x', 'y', 'z', curv_col, area_col]
    if cluster_by == 'contour_segment':
        cols.append('contour_pt')
    if max_dist_nm is not None:
        cols.append(dist_col)
    d = wall_df.loc[wall_df[category_col] == category, cols].copy()

    n0 = len(d)
    if exclude_boundary:
        on_edge = np.zeros(len(d), dtype=bool)
        for c in ['x', 'y', 'z']:
            lo, hi = wall_df[c].min(), wall_df[c].max()
            on_edge |= (d[c] <= lo + edge_tol) | (d[c] >= hi - edge_tol)
        d = d[~on_edge]
        print(f"excluded {n0 - len(d)} / {n0} {category} vertices on the volume boundary")

    n1 = len(d)
    d = d.dropna(subset=[curv_col, 'z', area_col])
    d = d[d[area_col] > 0]
    if max_dist_nm is not None:
        d = d[d[dist_col] <= max_dist_nm]
    d = d.copy()
    print(f"dropped {n1 - len(d)} more vertices with no valid kappa / area / distance; "
          f"{len(d)} remain")

    d['z_rel_nm'] = d['z'] - z_center_nm                       # nm from film center
    d['z_frac'] = (d['z'] - z_min) / thickness_nm              # kept for reference

    # ---- cluster ids for the bootstrap ----
    if cluster_by == 'column':
        d['cx'] = np.rint(d['x'] / voxel_nm).astype(int)
        d['cy'] = np.rint(d['y'] / voxel_nm).astype(int)
        d['col_id'] = d.groupby(['cx', 'cy']).ngroup()
    elif cluster_by == 'contour_segment':
        step = max(int(round(segment_nm / ds_nm)), 1)
        d['col_id'] = pd.factorize(d['contour_pt'] // step)[0]
    else:
        raise ValueError("cluster_by must be 'column' or 'contour_segment'")
    print(f"{d['col_id'].nunique()} bootstrap clusters (cluster_by='{cluster_by}')")

    z = d['z_rel_nm'].to_numpy()
    w = d[area_col].to_numpy()
    k = d[curv_col].to_numpy()
    cl = d['col_id'].to_numpy()

    # ---------- binned mean vs kappa ----------
    if kappa_range is None:
        lim = np.percentile(np.abs(k), 98)
        kappa_range = (-lim, lim)
    print(f"kappa bins span {kappa_range[0]:.4f} to {kappa_range[1]:.4f} 1/nm")

    edges = np.linspace(*kappa_range, n_bins + 1)
    centers = 0.5 * (edges[:-1] + edges[1:])
    vb = pd.cut(pd.Series(k), edges, labels=False).to_numpy() #### assign each value of kappa to a bin and return the bin number 
    vm = ~np.isnan(vb) ## boolean mask to ensure that we are only looking at points that get a binned curvature. only look at points in this true range 
    mean, err, n_pts, n_cl, _ = _binned_cluster_stats(
        vb[vm].astype(int), z[vm], w[vm], cl[vm], n_bins, n_boot, rng)
    keep = n_pts >= min_count
    summary = pd.DataFrame({'kappa_center': centers,
                            'z_nm_mean': mean, 'z_nm_err': err,          # nm from film center
                            'z_frac_mean': (mean + z_center_nm - z_min) / thickness_nm,
                            'z_frac_err': err / thickness_nm,
                            'n_points': n_pts, 'n_columns': n_cl, 'shown': keep})

    # ---------- three kappa classes (summary only) ----------
    cls = np.where(k < -kappa_tol, 0, np.where(k > kappa_tol, 2, 1))
    names = [f'κ < -{kappa_tol}', f'|κ| ≤ {kappa_tol}', f'κ > {kappa_tol}']
    c_mean, c_err, c_n, c_ncl, c_boots = _binned_cluster_stats(cls, z, w, cl, 3, n_boot, rng)
    class_summary = pd.DataFrame({'class': names,
                                  'z_nm_mean': c_mean, 'z_nm_err': c_err,
                                  'z_frac_mean': (c_mean + z_center_nm - z_min) / thickness_nm,
                                  'z_frac_err': c_err / thickness_nm,
                                  'n_points': c_n, 'n_clusters': c_ncl})

    # ---------- plot ----------
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.errorbar(centers[keep], mean[keep], yerr=err[keep], fmt='o-', capsize=3,
                color=color)
    ax.axvline(0, color='k', ls=':', lw=0.8, label='kappa = 0')
    ax.axhline(0, color='gray', ls='--', lw=0.8, label='film center')
    ax.set_xlabel('centerline curvature κ (1/nm)')
    ax.set_ylabel(f'{category} position relative to film center (nm)')
    ax.set_title(f'Mean {category} z vs. curvature with kappa_tol = {kappa_tol} \n {title}')
    ax.legend(fontsize=8)
    if ylim is not None:
        ax.set_ylim(*ylim)
    plt.tight_layout()
    plt.show()

    # ---------- printed summary ----------
    print("\nClass summary (area-weighted, cluster-bootstrap error; z in nm from film center):")
    print(class_summary.to_string(index=False))
    diff = c_boots[:, 2] - c_boots[:, 0]
    print(f"\nmean z(κ>tol) - mean z(κ<-tol) = {np.nanmean(diff):.2f} ± {np.nanstd(diff):.2f} nm")
    return summary, class_summary, d


from matplotlib.lines import Line2D



def plot_class_summary(summaries, xlim=None, title=None, default_label='class summary'):
    """
    summaries : a single class_summary DataFrame, or a dict {label: class_summary DataFrame}.
    Each DataFrame needs columns: class, z_nm_mean, z_nm_err.
    """
    if isinstance(summaries, pd.DataFrame):
        summaries = {default_label: summaries}

    markers = ['o', 's', 'D', '^']
    colors = ['tab:blue', 'tab:gray', 'tab:red']
    n = len(summaries)
    offs = np.linspace(-0.18, 0.18, n) if n > 1 else np.zeros(1)
    first = next(iter(summaries.values()))

    fig, ax = plt.subplots(figsize=(7, 3.5))
    for m, (name, cs) in enumerate(summaries.items()):
        for i in range(len(cs)):
            ax.errorbar(cs['z_nm_mean'].iloc[i], i + offs[m], xerr=cs['z_nm_err'].iloc[i],
                        fmt=markers[m], color=colors[i], capsize=3, ms=7,
                        mfc=colors[i] if m == 0 else 'white')
    ax.axvline(0, color='k', ls=':', lw=0.8)
    ax.set_yticks(range(len(first)))
    # ax.set_yticklabels('first['class']')
    ax.invert_yaxis()
    ax.set_xlabel('mean z relative to film center (nm)')
    if xlim is not None:
        ax.set_xlim(*xlim)
    handles = [Line2D([], [], marker=markers[m], color='k', ls='',
                      mfc='k' if m == 0 else 'white', label=lab)
               for m, lab in enumerate(summaries)]
    ax.legend(handles=handles, fontsize=8)
    ax.set_title(title)
    plt.tight_layout()
    plt.show()

In [ ]:
kappa_tol = 0.005

n_bins = 10
summary, class_summary, pts = plot_bloch_v_curvature(
    wall_df,  kappa_tol = kappa_tol, n_bins = n_bins, thickness = None, voxel_nm = voxel_nm, color = sample_color,  ylim=(-10, 10), title='sample 452')

plot_class_summary(class_summary, xlim=(-10, 10), title='sample 452')
# n_bins = 5
# summary, class_summary, pts = plot_bloch_v_curvature(
#     wall_df,kappa_tol = kappa_tol,  n_bins = n_bins, color = sample_color,  ylim=(-5, 5), title='sample 452')

# # if you reloaded the saved file, the category column is named differently
# summary, class_summary, pts = plot_bloch_v_curvature(
#     wall_df, thickness_nm=thickness, color = sample_color, category_col='category_normal', title='sample 452')

In [ ]:
# def plot_crossings_by_curvature_class(wall_df, kappa_tol=0.002, voxel_nm=2.0, normalize=False,
#                                       category='bloch', category_col='category',
#                                       curv_col='kappa', area_col='v_area',
#                                       weight_by='area', exclude_boundary=False, edge_tol=1.0):
#     """
#     Per-z-layer distribution of Bloch-classified vertices, split into
#     kappa < -tol, |kappa| <= tol, kappa > tol. x axis is z relative to the film center (nm).

#     weight_by        : 'area' weights each vertex by wall_df[area_col]; 'vertex' counts each equally.
#     normalize        : True shows the fraction within each class (each class sums to 1).
#     exclude_boundary : drop vertices within edge_tol nm of the volume faces, where the crop clips
#                        the isosurface (same as plot_bloch_v_curvature).
#     Returns a DataFrame of per-layer values for each class.
#     """
#     # film geometry from the full wall_df, so the axis doesn't depend on the selection
#     layer_all = (wall_df['z'] / voxel_nm).round().astype(int)
#     l_min, l_max = layer_all.min(), layer_all.max()
#     layers = np.arange(l_min, l_max + 1)
#     x_vals = (layers - 0.5 * (l_min + l_max)) * voxel_nm        # nm from film center

#     # same Bloch selection as plot_bloch_v_curvature
#     cols = ['x', 'y', 'z', curv_col] + ([area_col] if weight_by == 'area' else [])
#     df = wall_df.loc[wall_df[category_col] == category, cols].copy()

#     n0 = len(df)
#     if exclude_boundary:
#         on_edge = np.zeros(len(df), dtype=bool)
#         for c in ['x', 'y', 'z']:
#             lo, hi = wall_df[c].min(), wall_df[c].max()
#             on_edge |= (df[c] <= lo + edge_tol) | (df[c] >= hi - edge_tol)
#         df = df[~on_edge]
#         print(f"excluded {n0 - len(df)} / {n0} {category} vertices on the volume boundary")

#     df = df.dropna(subset=[curv_col, 'z'])
#     if weight_by == 'area':
#         df = df.dropna(subset=[area_col])
#         df = df[df[area_col] > 0]
#         weights = df[area_col].to_numpy()
#     else:
#         weights = np.ones(len(df))

#     k = df[curv_col].to_numpy()
#     idx = (df['z'] / voxel_nm).round().astype(int).to_numpy() - l_min

#     masks = {
#         f'κ < -{kappa_tol:g}': k < -kappa_tol,
#         f'|κ| ≤ {kappa_tol:g}': np.abs(k) <= kappa_tol,
#         f'κ > {kappa_tol:g}': k > kappa_tol,
#     }
#     colors = ['tab:blue', 'tab:gray', 'tab:red']

#     fig, ax = plt.subplots(figsize=(8, 4))
#     w = 0.27 * voxel_nm
#     table = {'z_from_center_nm': x_vals}
#     for j, ((name, m), c) in enumerate(zip(masks.items(), colors)):
#         vals = np.bincount(idx[m], weights=weights[m], minlength=len(layers))[:len(layers)]
#         if normalize and vals.sum() > 0:
#             vals = vals / vals.sum()
#         table[name] = vals
#         ax.bar(x_vals + (j - 1) * w, vals, width=w, color=c, label=f'{name} (n={m.sum()})')

#     ax.axvline(0, color='k', ls='--', lw=1)                    # film center
#     ax.set_xlabel('z position relative to film center (nm)')
#     unit = 'area' if weight_by == 'area' else 'vertices'
#     ax.set_ylabel(f'fraction of {category} {unit}' if normalize else f'{category} {unit}')
#     ax.set_xticks(x_vals)
#     ax.set_xticklabels([f'{v:g}' for v in x_vals], rotation=45)
#     ax.legend()
#     plt.tight_layout()
#     plt.show()
#     return pd.DataFrame(table)


def plot_crossings_by_curvature_class(wall_df, kappa_tol=0.002, voxel_nm=2.0, normalize=False,
                                      category='bloch', category_col='category',
                                      curv_col='kappa', area_col='v_area',
                                      weight_by='area', exclude_boundary=True, edge_tol=1.0,
                                      geom=None):
    if geom is None:
        geom = film_geometry(wall_df, voxel_nm)
    layers, x_vals, z_center = geom['layers'], geom['x_vals'], geom['z_center']

    cols = ['x', 'y', 'z', curv_col] + ([area_col] if weight_by == 'area' else [])
    df = wall_df.loc[wall_df[category_col] == category, cols].copy()

    n0 = len(df)
    if exclude_boundary:
        on_edge = np.zeros(len(df), dtype=bool)
        for c in ['x', 'y', 'z']:
            lo, hi = wall_df[c].min(), wall_df[c].max()
            on_edge |= (df[c] <= lo + edge_tol) | (df[c] >= hi - edge_tol)
        df = df[~on_edge]
        print(f"excluded {n0 - len(df)} / {n0} {category} vertices on the volume boundary")

    df = df.dropna(subset=[curv_col, 'z'])
    if weight_by == 'area':
        df = df.dropna(subset=[area_col])
        df = df[df[area_col] > 0]
        weights = df[area_col].to_numpy()
    else:
        weights = np.ones(len(df))

    k = df[curv_col].to_numpy()
    z_rel = df['z'].to_numpy() - z_center
    idx = np.rint(df['z'].to_numpy() / voxel_nm).astype(int) - layers[0]
    in_range = (idx >= 0) & (idx < len(layers))

    masks = {f'κ < -{kappa_tol:g}': k < -kappa_tol,
             f'|κ| ≤ {kappa_tol:g}': np.abs(k) <= kappa_tol,
             f'κ > {kappa_tol:g}': k > kappa_tol}
    colors = ['tab:blue', 'tab:gray', 'tab:red']

    fig, ax = plt.subplots(figsize=(8, 4))
    w = 0.27 * voxel_nm
    table = {'z_from_center_nm': x_vals}
    for j, ((name, m), c) in enumerate(zip(masks.items(), colors)):
        mm = m & in_range
        vals = np.bincount(idx[mm], weights=weights[mm], minlength=len(layers))[:len(layers)]
        if normalize and vals.sum() > 0:
            vals = vals / vals.sum()
        table[name] = vals
        ax.bar(x_vals + (j - 1) * w, vals, width=w, color=c, label=f'{name} (n={m.sum()})')
        if m.any():
            mean_z = np.average(z_rel[m], weights=weights[m])
            ax.axvline(mean_z, color=c, ls='--', lw=1.5)
            print(f"{name:14s} weighted mean z = {mean_z:+.2f} nm from center")

    ax.axvline(0, color='k', ls=':', lw=0.8)
    ax.set_xlabel('z position relative to film center (nm)')
    unit = 'area' if weight_by == 'area' else 'vertices'
    ax.set_ylabel(f'fraction of {category} {unit}' if normalize else f'{category} {unit}')
    ax.set_xticks(x_vals)
    ax.set_xticklabels([f'{v:g}' for v in x_vals], rotation=45)
    ax.legend()
    plt.tight_layout()
    plt.show()
    return pd.DataFrame(table)


In [ ]:
kappa_tol = 0.005
tbl = plot_crossings_by_curvature_class(wall_df, kappa_tol=kappa_tol, voxel_nm=voxel_nm)

## Map Interpolated zero crossings to curvature

In [ ]:
def find_M0_crossings_mesh(wall_df, faces, geom, angle_col='M_n_n', curv_col='kappa',
                           voxel_nm=2.0, cluster_gap_nm=4.0, drop_degenerate=True):
    """
    Exact M_normal = 0 points along mesh edges, grouped by lateral column.
    Returns
      z0_df     : one row per column with a single crossing
                  (col_x, col_y, x, y, z0_nm, z0_rel_nm, kappa, n_cross)
      misc_df   : columns with status 'multiple' or 'none' (z0_list_rel_nm for multiple)
      crossings : every edge crossing (x, y, z, z_rel_nm, kappa, col_x, col_y, status)
    z0_rel_nm is relative to the film center from geom, like the other plots.
    """
    assert (wall_df.index.to_numpy() == np.arange(len(wall_df))).all(), \
        "wall_df index must be 0..N-1 to match face indices (use reset_index(drop=True))"
    faces = np.asarray(faces, dtype=int)
    P = wall_df[['x', 'y', 'z']].to_numpy(float)
    m = wall_df[angle_col].to_numpy(float).copy()
    k = wall_df[curv_col].to_numpy(float)
    if drop_degenerate and 'degenerate' in wall_df.columns:
        m[wall_df['degenerate'].to_numpy(bool)] = np.nan
    m[m == 0] = 1e-12                      # exact zeros count as positive

    # unique mesh edges whose endpoints have opposite sign
    edges = np.vstack([faces[:, [0, 1]], faces[:, [1, 2]], faces[:, [2, 0]]])
    edges = np.unique(np.sort(edges, axis=1), axis=0)
    a, b = edges[:, 0], edges[:, 1]
    with np.errstate(invalid='ignore'):
        cross = (m[a] * m[b]) < 0          # NaN -> False
    a, b = a[cross], b[cross]
    t = m[a] / (m[a] - m[b])               # fraction of the way from a to b
    pts = P[a] + t[:, None] * (P[b] - P[a])
    kap = k[a] + t * (k[b] - k[a])         # NaN if either end has no kappa

    lo, hi = P.min(axis=0), P.max(axis=0)
    vert_ok = np.ones(len(P), bool)

    zc = geom['z_center']
    cr = pd.DataFrame({'x': pts[:, 0], 'y': pts[:, 1], 'z': pts[:, 2],
                       'z_rel_nm': pts[:, 2] - zc, 'kappa': kap,
                       'col_x': np.rint(pts[:, 0] / voxel_nm).astype(int),
                       'col_y': np.rint(pts[:, 1] / voxel_nm).astype(int)})

    singles, multis = [], []
    for (cx, cy), g in cr.groupby(['col_x', 'col_y']):
        z = np.sort(g['z'].to_numpy())
        groups = np.split(z, np.where(np.diff(z) > cluster_gap_nm)[0] + 1)
        if len(groups) == 1:
            singles.append({'col_x': cx, 'col_y': cy, 'x': g['x'].mean(), 'y': g['y'].mean(),
                            'z0_nm': z.mean(), 'z0_rel_nm': z.mean() - zc,
                            'kappa': g['kappa'].mean(), 'n_cross': len(z)})
        else:
            multis.append({'col_x': cx, 'col_y': cy, 'status': 'multiple',
                           'z0_list_rel_nm': [gz.mean() - zc for gz in groups]})

    z0_df = pd.DataFrame(singles, columns=['col_x', 'col_y', 'x', 'y', 'z0_nm', 'z0_rel_nm',
                                           'kappa', 'n_cross'])
    multi_df = pd.DataFrame(multis, columns=['col_x', 'col_y', 'status', 'z0_list_rel_nm'])

    # columns that contain wall vertices but no crossing
    vcols = pd.DataFrame({'col_x': np.rint(P[vert_ok, 0] / voxel_nm).astype(int),
                          'col_y': np.rint(P[vert_ok, 1] / voxel_nm).astype(int)}).drop_duplicates()
    have = pd.concat([z0_df[['col_x', 'col_y']], multi_df[['col_x', 'col_y']]])
    none_df = vcols.merge(have, on=['col_x', 'col_y'], how='left', indicator=True)
    none_df = none_df[none_df['_merge'] == 'left_only'][['col_x', 'col_y']].copy()
    none_df['status'] = 'none'
    none_df['z0_list_rel_nm'] = [[] for _ in range(len(none_df))]
    misc_df = pd.concat([multi_df, none_df], ignore_index=True)

    multi_keys = set(zip(multi_df['col_x'], multi_df['col_y']))
    cr['status'] = np.where([(i, j) in multi_keys for i, j in zip(cr['col_x'], cr['col_y'])],
                            'multiple', 'single')

    n_tot = len(z0_df) + len(misc_df)
    print(f"{len(cr)} edge crossings; {n_tot} columns total")
    print(f"  single   : {len(z0_df)} ({len(z0_df) / n_tot:.0%})")
    print(f"  multiple : {len(multi_df)} ({len(multi_df) / n_tot:.0%})")
    print(f"  none     : {len(none_df)} ({len(none_df) / n_tot:.0%})  "
          f"(approximate: the crossing may fall in a neighboring column)")
    print(f"  kappa valid for {z0_df['kappa'].notna().sum()} of the single columns")
    return z0_df, misc_df, cr

In [ ]:
def plot_z0_vs_curvature(z0_df, geom, n_bins=10, kappa_range=None, kappa_tol=0.002,
                         min_count=10, cluster_nm=40.0, normalize=False,
                         n_boot=500, seed=0, color='green', ylim=None, title=None,
                         bin_nm=8.0):
    """
    Left : mean z0 (nm from film center) vs. kappa, one value per column, standard errors.
    Right: distribution of z0 for kappa < -tol, |kappa| <= tol, kappa > tol (dashed = class mean).
    Returns (summary, class_summary, d), with the same columns as plot_bloch_v_curvature.
    """
    rng = np.random.default_rng(seed)
    d = z0_df.dropna(subset=['kappa']).copy()
    z = d['z0_rel_nm'].to_numpy()
    k = d['kappa'].to_numpy()
    w = np.ones(len(d))
    d['bx'] = np.floor(d['x'] / cluster_nm).astype(int)
    d['by'] = np.floor(d['y'] / cluster_nm).astype(int)
    cl = d.groupby(['bx', 'by']).ngroup().to_numpy()
    print(f"{len(d)} columns with valid kappa, {cl.max() + 1} bootstrap blocks of {cluster_nm:g} nm")

    if kappa_range is None:
        lim = np.percentile(np.abs(k), 98)
        kappa_range = (-lim, lim)
    print(f"kappa bins span {kappa_range[0]:.4f} to {kappa_range[1]:.4f} 1/nm")

    edges = np.linspace(*kappa_range, n_bins + 1)
    centers = 0.3 * (edges[:-1] + edges[1:])
    vb = pd.cut(pd.Series(k), edges, labels=False).to_numpy()
    vm = ~np.isnan(vb)
    # mean, err, n_pts, n_cl, _ = _binned_cluster_stats(
    #     vb[vm].astype(int), z[vm], w[vm], pd.factorize(cl[vm])[0], n_bins, n_boot, rng)

    mean, err, n_pts, n_cl, ex = _binned_raw_stats(vb[vm].astype(int), z[vm], w[vm], pd.factorize(cl[vm])[0], n_bins, k=k[vm])

    keep = n_pts >= min_count
    summary = pd.DataFrame({'kappa_center': centers, 'z_nm_mean': mean, 'z_nm_err': err,
                            'n_points': n_pts, 'n_clusters': n_cl, 'shown': keep})

    cls = np.where(k < -kappa_tol, 0, np.where(k > kappa_tol, 2, 1))
    names = [f'κ < -{kappa_tol:g}', f'|κ| ≤ {kappa_tol:g}', f'κ > {kappa_tol:g}']
    colors = ['tab:blue', 'tab:gray', 'tab:red']
    # c_mean, c_err, c_n, c_ncl, c_boots = _binned_cluster_stats(cls, z, w, cl, 3, n_boot, rng)
    c_mean, c_err, c_n, c_ncl, c_ex = _binned_raw_stats(cls, z, w, cl, 3, k = None)


    class_summary = pd.DataFrame({'class': names, 'z_nm_mean': c_mean, 'z_nm_err': c_err,
                                  'n_points': c_n, 'n_clusters': c_ncl})

    fig, (ax, axh) = plt.subplots(1, 2, figsize=(13, 4.5))
    ax.errorbar(centers[keep], mean[keep], yerr=err[keep], fmt='o-', capsize=3, color=color)
    ax.axvline(0, color='k', ls=':', lw=0.8, label='kappa = 0')
    ax.axhline(0, color='gray', ls='--', lw=0.8, label='film center')
    
    for j, c in enumerate(colors):
        ax.axhline(c_mean[j], color=c, ls='--', lw=1.5)



    # ax.axvspan(-kappa_tol, kappa_tol, color='gray', alpha=0.12, label=f'|κ| ≤ {kappa_tol:g}')
    ax.set_xlabel('dw curvature (1/nm)')
    ax.set_ylabel('M_normal = 0 (Bloch) position relative to film center (nm)')
    ax.set_title(f'Mean Bloch position (z) vs. curvature \n {title}')
    ax.legend(fontsize=8)
    if ylim is not None:
        ax.set_ylim(*ylim)

    # x_vals, vox = geom['x_vals'], geom['voxel_nm']
    # zedges = np.concatenate([x_vals - vox / 2, [x_vals[-1] + vox / 2]])
    # wb = 0.27 * vox
    # for j, c in enumerate(colors):
    #     m = cls == j
    #     counts, _ = np.histogram(z[m], bins=zedges)
    #     vals = counts / counts.sum() if normalize and counts.sum() > 0 else counts
    #     axh.bar(x_vals + (j - 1) * wb, vals, width=wb, color=c, label=f'{names[j]} (n={m.sum()})')
    #     axh.axvline(c_mean[j], color=c, ls='--', lw=1.5)
    # axh.axvline(0, color='k', ls=':', lw=0.8)
    # axh.set_xlabel('z relative to film center (nm)')
    # axh.set_ylabel('fraction of columns' if normalize else '# columns')
    # axh.set_xticks(x_vals)
    # axh.set_xticklabels([f'{v:g}' for v in x_vals], rotation=45)

    x_vals, vox = geom['x_vals'], geom['voxel_nm']
    z_lo = x_vals.min() - vox / 2
    z_hi = x_vals.max() + vox / 2
    z_edges = np.arange(z_lo, z_hi + bin_nm, bin_nm)
    z_centers = 0.5 * (z_edges[:-1] + z_edges[1:])
    wb = 0.3 * bin_nm                      # width of each of the 3 grouped bars

    for j, c in enumerate(colors):
        m = cls == j
        counts, _ = np.histogram(z[m], bins=z_edges)
        vals = counts / counts.sum() if normalize and counts.sum() > 0 else counts
        axh.bar(z_centers + (j - 1) * wb, vals, width=wb, color=c,
                label=f'{names[j]} (n={m.sum()})')
        axh.axvline(c_mean[j], color=c, ls='--', lw=1.5)
    axh.axvline(0, color='k', ls=':', lw=0.8)
    axh.set_xlabel('z relative to film center (nm)')
    axh.set_ylabel('fraction of layers' if normalize else 'counts')
    axh.set_xticks(z_centers)
    axh.set_xticklabels([f'{v:g}' for v in z_centers], rotation=45)

    axh.set_title('Bloch position distribution by curvature')
    axh.legend(fontsize=8)
    plt.tight_layout()
    plt.show()

    print("\nClass summary (one value per column, block-bootstrap error; z in nm from center):")
    print(class_summary.to_string(index=False))
    # diff = c_boots[:, 2] - c_boots[:, 0]
    # print(f"\nmean z0(κ>tol) - mean z0(κ<-tol) = {np.nanmean(diff):.2f} ± {np.nanstd(diff):.2f} nm")
    return summary, class_summary, d

In [ ]:
def plot_z0_map_topdown(z0_df, misc_df=None, voxel_nm=2.0, z0_clim=4.0, s=8, title=None):
    """Top-down map: each single-crossing column colored by z0 (nm from center).
    Dropped columns are marked: 'multiple' = black x, 'none' = light gray x."""
    fig, ax = plt.subplots(figsize=(7, 6))
    if misc_df is not None and len(misc_df):
        for status, c in [('none', 'lightgray'), ('multiple', 'black')]:
            mm = misc_df[misc_df['status'] == status]
            ax.scatter(mm['col_x'] * voxel_nm, mm['col_y'] * voxel_nm, s=s, color=c,
                       marker='x', lw=0.6, label=f'{status} (n={len(mm)})')
    sc = ax.scatter(z0_df['x'], z0_df['y'], c=z0_df['z0_rel_nm'], cmap='coolwarm',
                    vmin=-z0_clim, vmax=z0_clim, s=s)
    plt.colorbar(sc, label='z0 relative to film center (nm)')
    ax.set_aspect('equal')
    ax.set_xlabel('x (nm)')
    ax.set_ylabel('y (nm)')
    ax.set_title(f'Assigned z0 (top-down)\n {title}')
    ax.legend(fontsize=8)
    plt.show()


def plot_z0_on_isosurface(wall_df, faces, crossings, z0_df, voxel_nm=2.0,
                          window=None, z_scale=1.0, m_clim=None, z0_clim=4.0, point_size=6):
    """
    Two linked PyVista panels of the isosurface:
      left : surface colored by M_normal; black dots = single-column crossings,
             magenta = crossings in columns dropped as 'multiple'. Dots should sit on the
             pale (M_normal ~ 0) band.
      right: surface colored by the z0 assigned to each vertex's lateral column
             (nm from center; gray = column has no single z0), black dots = crossings.
    window  : (xmin, xmax, ymin, ymax) in nm. Strongly recommended: the film is ~30 nm
              thick and ~700 nm wide, so zoom to something like 150 x 150 nm.
    z_scale : vertical exaggeration (the film is thin, so z0 differences are invisible at 1).
    m_clim  : half-range of the M_normal color scale (None = 95th percentile of |M_normal|).
    """



    P = wall_df[['x', 'y', 'z']].to_numpy(float)
    faces = np.asarray(faces, dtype=int)
    cr = crossings
    if window is not None:
        xmin, xmax, ymin, ymax = window
        inwin = (P[:, 0] >= xmin) & (P[:, 0] <= xmax) & (P[:, 1] >= ymin) & (P[:, 1] <= ymax)
        faces = faces[inwin[faces].all(axis=1)]
        cr = cr[(cr['x'] >= xmin) & (cr['x'] <= xmax) & (cr['y'] >= ymin) & (cr['y'] <= ymax)]

    mesh = pv.PolyData(P, np.hstack([np.full((len(faces), 1), 3), faces]).ravel())
    Mn = wall_df['M_n_n'].to_numpy(float)
    mesh.point_data['M_n_n'] = Mn
    key = pd.MultiIndex.from_arrays([np.rint(P[:, 0] / voxel_nm).astype(int),
                                     np.rint(P[:, 1] / voxel_nm).astype(int)])
    lut = z0_df.set_index(['col_x', 'col_y'])['z0_rel_nm']
    mesh.point_data['z0_rel_nm'] = lut.reindex(key).to_numpy()

    if m_clim is None:
        m_clim = np.nanpercentile(np.abs(Mn), 95)
    single = cr[cr['status'] == 'single']
    multi = cr[cr['status'] == 'multiple']

    pl = pv.Plotter()

    pl.add_mesh(mesh, scalars='M_n_n', cmap='coolwarm', clim=(-m_clim, m_clim),
                nan_color='lightgray', scalar_bar_args={'title': 'M_n_n'})
    if len(single):
        pl.add_points(single[['x', 'y', 'z']].to_numpy(), color='magenta',
                      point_size=point_size, render_points_as_spheres=True)
    if len(multi):
        pl.add_points(multi[['x', 'y', 'z']].to_numpy(), color='magenta',
                      point_size=point_size, render_points_as_spheres=True)

    pl.set_scale(zscale=z_scale)

    pl.show()



 

In [ ]:
geom = film_geometry(wall_df, voxel_nm=8.0)        # or z_min=0, z_max=30

# 3) zero-crossing version
z0_df, misc_df, crossings = find_M0_crossings_mesh(wall_df, faces, geom, angle_col='M_n_n',voxel_nm=8.0)
z0_summary, z0_class_summary, z0_pts = plot_z0_vs_curvature(
    z0_df, geom, n_bins = 5,  kappa_tol=0.005, normalize=False, color=sample_color, title='sample 452', bin_nm=8.0)

z0_summary, z0_class_summary, z0_pts = plot_z0_vs_curvature(
    z0_df, geom, n_bins = 5,  kappa_tol=0.005, normalize=True, color=sample_color, title='sample 452', bin_nm=8.0)

# # 2) compare the two definitions
# plot_class_summary({'band |M_normal|<tol': class_summary, 'M_normal = 0 crossing': z0_class_summary}, xlim=(-10, 10))

In [ ]:
ADD A SCATTER PLOT

In [ ]:
# # 4) visual check
# plot_z0_map_topdown(z0_df, misc_df, title='sample 452')


plot_z0_on_isosurface(wall_df, faces, crossings, z0_df,voxel_nm=voxel_nm)



print(f'================= TEST ASSUMPTIONS ==============')
# 1) faces and wall_df are consistent
print(faces.shape, faces.max(), len(wall_df))          # faces.max() should be len(wall_df) - 1

# 2) mesh welded? (should be close to len(wall_df))
print(len(np.unique(wall_df[['x', 'y', 'z']].round(4).to_numpy(), axis=0)), len(wall_df))

# 3) even crossing counts in 'single' columns (likely zig-zag noise)
print((z0_df['n_cross'] % 2 == 0).mean(), z0_df['n_cross'].describe())

# 4) how many crossings sit on the clipped boundary
b = (crossings['z'] <= wall_df['z'].min() + 1) | (crossings['z'] >= wall_df['z'].max() - 1)
print(b.mean())

cr = crossings

# 1) z spread inside 'single' columns: should be small if it is one clean curve
span = (cr[cr['status'] == 'single'].groupby(['col_x', 'col_y'])['z']
          .agg(lambda s: s.max() - s.min()))
print(span.describe())
print("fraction of singles with z span > 4 nm:", (span > 4).mean())

# 2) 'multiple' columns: largest z gap between consecutive crossings
def largest_gap(s):
    return np.diff(np.sort(s.to_numpy())).max()
gap = (cr[cr['status'] == 'multiple'].groupby(['col_x', 'col_y'])['z']
         .agg(largest_gap))
print(gap.describe())
print(np.histogram(gap, bins=[4, 5, 6, 8, 10, 15, 30]))

total_len = sum(c['length_nm'] for c in contours)
print(f"contour length {total_len:.0f} nm -> ~{total_len / 2.0:.0f} columns expected; "
      f"singles = {len(z0_df)}, multiple = {(misc_df['status'] == 'multiple').sum()}, "
      f"none = {(misc_df['status'] == 'none').sum()}")


# Thickness and fit histogram with bimodal?

In [ ]:
# def plot_Bloch_angle_vs_z_m0(wall_df, faces, geom=None, angle_col='M_n_n', curv_col='kappa',
#                              bloch_axis=None, neel_axis=None, title=None, voxel_nm=2.0,
#                              cluster_gap_nm=4.0, drop_degenerate=False,
#                              exclude_boundary=True, edge_tol=1.0):
#     """
#     Left : mean angle_col per z layer (± std), x axis = z relative to film center (nm).
#     Right: number of interpolated angle_col = 0 crossings (one per lateral column)
#            in each z layer, from find_M0_crossings_mesh.

#     faces          : (M, 3) array of vertex indices into wall_df rows (wall_df index must be 0..N-1)
#     geom           : output of film_geometry(); None = computed from wall_df
#     cluster_gap_nm : crossings in one column closer than this in z are merged into one z0;
#                      farther apart, the column is 'multiple'. Replaces the old max_gap.
#     Returns
#     -------
#     grouped   : DataFrame indexed by z layer (mean, std, count of angle_col, plus z_rel_nm)
#     z0_df     : one row per column with a single crossing
#                 (col_x, col_y, x, y, z0_nm, z0_rel_nm, kappa, n_cross, z0_layer)
#     misc_df   : columns with status 'multiple' or 'none'
#     crossings : every edge crossing, for plot_z0_on_isosurface
#     """
#     if geom is None:
#         geom = film_geometry(wall_df, voxel_nm)
#     voxel_nm = geom['voxel_nm']
#     layers, x_vals, z_center = geom['layers'], geom['x_vals'], geom['z_center']

#     xlabel = ('distance from film center (nm)' if voxel_nm
#               else 'z voxel layer')
#     print(f"z layers ({len(layers)} layers), center = {z_center}")



#     # the crossing finder needs a curvature column; fill with NaN if it doesn't exist yet
#     wdf = wall_df if curv_col in wall_df.columns else wall_df.assign(**{curv_col: np.nan})

#     z0_df, misc_df, crossings = find_M0_crossings_mesh(
#         wdf, faces, geom, angle_col=angle_col, curv_col=curv_col, voxel_nm=voxel_nm,
#         cluster_gap_nm=cluster_gap_nm, drop_degenerate=drop_degenerate,
#         exclude_boundary=exclude_boundary, edge_tol=edge_tol)
#     z0_df['z0_layer'] = np.rint(z0_df['z0_nm'] / voxel_nm).astype(int)

#     # ---- mean profile of angle_col per z layer ----
#     d = wall_df[['z', angle_col]].copy()
#     if drop_degenerate and 'degenerate' in wall_df.columns:
#         d = d[~wall_df['degenerate'].astype(bool)]
#     d = d.dropna(subset=[angle_col])
#     d['z_layer'] = np.rint(d['z'] / voxel_nm).astype(int)
#     d = d[d['z_layer'].between(layers[0], layers[-1])]
#     grouped = d.groupby('z_layer')[angle_col].agg(['mean', 'std', 'count'])
#     grouped['z_rel_nm'] = grouped.index.to_numpy() * voxel_nm - z_center

#     # ---- crossing counts per z layer (nm bins centered on the layers) ----
#     zedges = np.concatenate([x_vals - voxel_nm / 2, [x_vals[-1] + voxel_nm / 2]])
#     counts, _ = np.histogram(z0_df['z0_rel_nm'], bins=zedges)
#     n_out = len(z0_df) - counts.sum()
#     if n_out:
#         print(f"{n_out} crossings fall outside the film layers and are not shown")



#     # ---- plots ----
#     fig, (ax, ax2) = plt.subplots(1, 2, figsize=(18, 6), sharex=True)

#     ax.errorbar(grouped['z_rel_nm'], grouped['mean'], yerr=grouped['std'],
#                 fmt='o-', capsize=3)
#     ax.axhline(0, color='k', lw=0.8, ls=':')
#     if bloch_axis is not None:
#         for i, deg in enumerate(bloch_axis):
#             ax.axhline(deg, color='magenta', label='Bloch Wall' if i == 0 else None)
#     if neel_axis is not None:
#         for i, deg in enumerate(neel_axis):
#             ax.axhline(deg, color='cyan', label='Neel Wall' if i == 0 else None)
#     if bloch_axis is not None or neel_axis is not None:
#         ax.legend(fontsize=11)
#     ax.axvline(0, color='gray', ls='--', lw=0.8)
#     ax.set_ylabel(angle_col, fontsize=15)
#     ax.set_xlabel('z relative to film center (nm)', fontsize=15)

#     ax2.bar(x_vals, counts, width=0.9 * voxel_nm, color='green', alpha=0.5)
#     ax2.axvline(0, color='gray', ls='--', lw=0.8)
#     ax2.set_xlabel('z relative to film center (nm)', fontsize=15)
#     ax2.set_ylabel(f'# of {angle_col} = 0 crossings (one per column)', fontsize=15)
#     ax2.set_xticks(x_vals)
#     ax2.set_xticklabels([f'{v:g}' for v in x_vals], rotation=45)

#     fig.suptitle(f'{angle_col} vs. thickness position \n {title}', fontsize=20)
#     plt.tight_layout()
#     plt.show()

#     return grouped, z0_df, misc_df, crossings




In [ ]:
# def plot_Bloch_angle_vs_z_m0(wall_df, faces, geom=None, angle_col='M_n_n', curv_col='kappa',
#                              color='green', title=None, voxel_nm=2.0,
#                              cluster_gap_nm=4.0, drop_degenerate=False,
#                              show_fit=True, ylim=(-1.1, 1.1), figsize=(24, 7)):
#     """
#     Left : mean angle_col per z layer (± std) vs. z relative to film center (nm).
#     Right: number of interpolated angle_col = 0 crossings (one per lateral column) per z layer,
#            with mean / median / variance of z0 and an optional Gaussian fit (R², FWHM).

#     Statistics:
#       mean, median, variance (ddof=1), std : computed from the individual z0 values
#                                              (not the binned counts), nm from film center
#       Gaussian fit                         : A*exp(-(z-mu)^2/(2 sigma^2)) fit to the per-layer counts;
#                                              FWHM = 2*sqrt(2 ln 2)*|sigma|; R² on the binned counts

#     Returns grouped, z0_df, misc_df, crossings, stats
#       stats : dict(mean, median, variance, std, n, and if the fit succeeded: A, mu, sigma, fwhm, r2)
#     """
#     print(voxel_nm)
#     if geom is None:
#         geom = film_geometry(wall_df, voxel_nm)
#     #voxel_nm = geom['voxel_nm']
#     layers, x_vals, z_center = geom['layers'], geom['x_vals'], geom['z_center']

#     wdf = wall_df if curv_col in wall_df.columns else wall_df.assign(**{curv_col: np.nan})
#     z0_df, misc_df, crossings = find_M0_crossings_mesh(
#         wdf, faces, geom, angle_col=angle_col, curv_col=curv_col, voxel_nm=voxel_nm,
#         cluster_gap_nm=cluster_gap_nm, drop_degenerate=drop_degenerate)
#     z0_df['z0_layer'] = np.rint(z0_df['z0_nm'] / voxel_nm).astype(int)

#     # ---- mean profile of angle_col per z layer (reindexed so it matches x_vals) ----
#     d = wall_df[['z', angle_col]].copy()
#     if drop_degenerate and 'degenerate' in wall_df.columns:
#         d = d[~wall_df['degenerate'].astype(bool)]
#     d = d.dropna(subset=[angle_col])
#     d['z_layer'] = np.rint(d['z'] / voxel_nm).astype(int)
#     grouped = d.groupby('z_layer')[angle_col].agg(['mean', 'std', 'count', 'sem']).reindex(layers)
#     grouped['z_rel_nm'] = x_vals

#     # ---- crossing counts per z layer ----
#     zedges = np.concatenate([x_vals - voxel_nm / 2, [x_vals[-1] + voxel_nm / 2]])
#     y, _ = np.histogram(z0_df['z0_rel_nm'], bins=zedges)
#     width = 0.9 * voxel_nm
#     n_out = len(z0_df) - y.sum()
#     if n_out:
#         print(f"{n_out} crossings fall outside the film layers and are not shown")

#     # ---- statistics from the individual z0 values (inside the film layers) ----
#     z0 = z0_df['z0_rel_nm'].to_numpy()
#     z0 = z0[(z0 >= zedges[0]) & (z0 <= zedges[-1])]
#     stats = {'n': len(z0)}
#     if len(z0) > 1:
#         stats.update(mean=float(np.mean(z0)), median=float(np.median(z0)),
#                      variance=float(np.var(z0, ddof=1)), std=float(np.std(z0, ddof=1)))
#     else:
#         stats = None

#     # ---- Gaussian fit to the per-layer counts ----
#     A = mu = sigma = fwhm = r2 = None
#     if show_fit and stats is not None and y.sum() > 0:
#         try:
#             p0 = [y.max(), stats['mean'], max(stats['std'], voxel_nm)]
#             (A, mu, sigma), _ = curve_fit(_gaussian, x_vals, y, p0=p0, maxfev=10000)
#             sigma = abs(sigma)
#             fwhm = 2 * np.sqrt(2 * np.log(2)) * sigma
#             resid = y - _gaussian(x_vals, A, mu, sigma)
#             ss_tot = np.sum((y - y.mean()) ** 2)
#             r2 = 1 - np.sum(resid ** 2) / ss_tot if ss_tot > 0 else np.nan
#             stats.update(A=float(A), mu=float(mu), sigma=float(sigma),
#                          fwhm=float(fwhm), r2=float(r2))
#         except RuntimeError:
#             print("Gaussian fit did not converge; plotting without it")
#             show_fit = False
#     else:
#         show_fit = False

#     if stats is not None:
#         print(f"z0 (nm from center): n = {stats['n']}, mean = {stats['mean']:.2f}, "
#               f"median = {stats['median']:.2f}, variance = {stats['variance']:.2f} nm^2")
#         if show_fit:
#             print(f"Gaussian fit: mu = {mu:.2f} nm, sigma = {sigma:.2f} nm, "
#                   f"FWHM = {fwhm:.2f} nm, R² = {r2:.2f}")

#     # ---- plots ----
#     xlabel = 'z relative to film center (nm)'
#     fig = plt.figure(figsize=figsize)
#     gs = GridSpec(1, 2, width_ratios=[1, 1], figure=fig)
#     ax2 = fig.add_subplot(gs[0])
#     ax3 = fig.add_subplot(gs[1], sharex=ax2)

#     # Panel 1: M_norm profile
#     ax2.errorbar(x_vals, grouped['mean'], yerr=grouped['sem'], fmt='o-',
#                  capsize=3, color=color)
#     ax2.axhline(0, color='magenta', ls='--', label='Bloch (0)')
#     ax2.axhline(1, color='cyan', ls='--', label='Neel (±1)')
#     ax2.axhline(-1, color='cyan', ls='--')
#     ax2.axvline(0, color='k', ls='--', label='Film center')
#     if ylim is not None:
#         ax2.set_ylim(*ylim)

#     if stats:
#         ax2.axvline(stats['mean'], color='red', lw=1.5,
#                     label=f"mean = {stats['mean']:.2f} nm")
#         ax2.axvline(stats['median'], color='tab:blue', lw=1.5, ls='-.',
#                     label=f"median = {stats['median']:.2f} nm")
        
#     ax2.set_xlabel(xlabel, fontsize=25)
#     ax2.set_ylabel(angle_col, fontsize=25)
#     ax2.legend(fontsize=20)

#     # Panel 2: counts where M_norm = 0
#     ax3.bar(x_vals, y, width=width, color=color, alpha=0.5, edgecolor='gray')
#     ax3.axvline(0, color='k', ls='--', label='Film center')

#     if stats:
#         ax3.axvline(stats['mean'], color='red', lw=1.5,
#                     label=f"mean = {stats['mean']:.2f} nm")
#         ax3.axvline(stats['median'], color='tab:blue', lw=1.5, ls='-.',
#                     label=f"median = {stats['median']:.2f} nm")
#         ax3.plot([], [], ' ', label=f"variance = {stats['variance']:.2f} nm^2")

#     if show_fit:
#         xs = np.linspace(x_vals.min() - width / 2, x_vals.max() + width / 2, 300)
#         ax3.plot(xs, _gaussian(xs, A, mu, sigma), color='k', lw=2,
#                  label=f"Gaussian fit (R² = {r2:.2f})")
#         ax3.hlines(A / 2, mu - fwhm / 2, mu + fwhm / 2, color='k', lw=2.5, alpha=0,
#                    label=f"FWHM = {fwhm:.2f} nm")
#         ax3.axvline(mu, color='darkorange', lw=1, ls=':')

#     ax3.legend(loc='upper left', fontsize=20)
#     ax3.set_xlabel(xlabel, fontsize=25)
#     ax3.set_ylabel('# of Bloch points per z', fontsize=25)
#     ax3.set_xticks(x_vals)
#     ax3.set_xticklabels([f'{v:g}' for v in x_vals], rotation=45)

#     fig.suptitle(f'{angle_col} vs. thickness position \n {title}', fontsize=30)
#     plt.tight_layout()
#     plt.show()

#     return grouped, z0_df, misc_df, crossings, stats


def plot_Bloch_angle_vs_z_m0(wall_df, faces, geom=None, angle_col='M_n_n', curv_col='kappa',
                             color='green', title=None, voxel_nm=2.0, bin_nm=None,
                             cluster_gap_nm=4.0, drop_degenerate=False,
                             show_fit=True, ylim=(-1.1, 1.1), figsize=(24, 7)):
    """
    bin_nm : z bin size (nm) used for the profile, the crossing counts and the Gaussian fit.
             Defaults to voxel_nm. Set it to the real resolution, not the interpolated voxel size.
    """
    if bin_nm is None:
        bin_nm = voxel_nm
    if geom is None:
        geom = film_geometry(wall_df, voxel_nm)
    layers, x_vals, z_center = geom['layers'], geom['x_vals'], geom['z_center']

    wdf = wall_df if curv_col in wall_df.columns else wall_df.assign(**{curv_col: np.nan})
    z0_df, misc_df, crossings = find_M0_crossings_mesh(
        wdf, faces, geom, angle_col=angle_col, curv_col=curv_col, voxel_nm=voxel_nm,
        cluster_gap_nm=cluster_gap_nm, drop_degenerate=drop_degenerate)
    z0_df['z0_layer'] = np.rint(z0_df['z0_nm'] / voxel_nm).astype(int)

    # ---- per-layer profile (fine, interpolated grid): this is what gets returned ----
    d = wall_df[['z', angle_col]].copy()
    if drop_degenerate and 'degenerate' in wall_df.columns:
        d = d[~wall_df['degenerate'].astype(bool)]
    d = d.dropna(subset=[angle_col])
    d['z_layer'] = np.rint(d['z'] / voxel_nm).astype(int)
    grouped = d.groupby('z_layer')[angle_col].agg(['mean', 'std', 'count', 'sem']).reindex(layers)
    grouped['z_rel_nm'] = x_vals

    # ---- coarse z bins at the real resolution, centered on the film center ----
    ext = max(abs(x_vals.min()), abs(x_vals.max())) + voxel_nm / 2
    n_half = int(np.ceil((ext - bin_nm / 2) / bin_nm))
    zedges = (np.arange(-n_half, n_half + 2) - 0.5) * bin_nm      # one bin centered on z = 0
    centers = 0.5 * (zedges[:-1] + zedges[1:])
    nb = len(centers)
    width = 0.9 * bin_nm
    print(f"{nb} z bins of {bin_nm:g} nm (interpolated voxel = {voxel_nm:g} nm)")

    # profile of angle_col on the coarse bins
    d['z_rel'] = d['z'] - z_center
    d['bin'] = np.digitize(d['z_rel'], zedges) - 1
    d = d[(d['bin'] >= 0) & (d['bin'] < nb)]
    grouped_b = d.groupby('bin')[angle_col].agg(['mean', 'std', 'count', 'sem']).reindex(range(nb))

    # ---- crossing counts per coarse bin ----
    y, _ = np.histogram(z0_df['z0_rel_nm'], bins=zedges)
    n_out = len(z0_df) - y.sum()
    if n_out:
        print(f"{n_out} crossings fall outside the film layers and are not shown")

    # ---- statistics from the individual z0 values (independent of bin size) ----
    z0 = z0_df['z0_rel_nm'].to_numpy()
    z0 = z0[(z0 >= zedges[0]) & (z0 <= zedges[-1])]
    stats = {'n': len(z0)}
    if len(z0) > 1:
        stats.update(mean=float(np.mean(z0)), median=float(np.median(z0)),
                     variance=float(np.var(z0, ddof=1)), std=float(np.std(z0, ddof=1)))
    else:
        stats = None

    # ---- Gaussian fit to the binned counts ----
    A = mu = sigma = fwhm = r2 = None
    if show_fit and stats is not None and y.sum() > 0 and nb >= 4:
        try:
            p0 = [y.max(), stats['mean'], max(stats['std'], bin_nm)]
            (A, mu, sigma), _ = curve_fit(_gaussian, centers, y, p0=p0, maxfev=10000)
            sigma = abs(sigma)
            fwhm = 2 * np.sqrt(2 * np.log(2)) * sigma
            resid = y - _gaussian(centers, A, mu, sigma)
            ss_tot = np.sum((y - y.mean()) ** 2)
            r2 = 1 - np.sum(resid ** 2) / ss_tot if ss_tot > 0 else np.nan
            stats.update(A=float(A), mu=float(mu), sigma=float(sigma),
                         fwhm=float(fwhm), r2=float(r2))
        except RuntimeError:
            print("Gaussian fit did not converge; plotting without it")
            show_fit = False
    else:
        show_fit = False

    if stats is not None:
        print(f"z0 (nm from center): n = {stats['n']}, mean = {stats['mean']:.2f}, "
              f"median = {stats['median']:.2f}, variance = {stats['variance']:.2f} nm^2")
        if show_fit:
            print(f"Gaussian fit: mu = {mu:.2f} nm, sigma = {sigma:.2f} nm, "
                  f"FWHM = {fwhm:.2f} nm, R² = {r2:.2f}")

    # ---- plots ----
    xlabel = 'z relative to film center (nm)'
    fig = plt.figure(figsize=figsize)
    gs = GridSpec(1, 2, width_ratios=[1, 1], figure=fig)
    ax2 = fig.add_subplot(gs[0])
    ax3 = fig.add_subplot(gs[1], sharex=ax2)

    # Panel 1: profile on the coarse bins
    ax2.errorbar(centers, grouped_b['mean'], yerr=grouped_b['sem'], fmt='o-',
                 capsize=3, color=color)
    ax2.axhline(0, color='magenta', ls='--', label='Bloch (0)')
    ax2.axhline(1, color='cyan', ls='--', label='Neel (±1)')
    ax2.axhline(-1, color='cyan', ls='--')
    ax2.axvline(0, color='k', ls='--', label='Film center')
    if ylim is not None:
        ax2.set_ylim(*ylim)
    if stats:
        ax2.axvline(stats['mean'], color='red', lw=1.5, label=f"mean = {stats['mean']:.2f} nm")
        ax2.axvline(stats['median'], color='tab:blue', lw=1.5, ls='-.',
                    label=f"median = {stats['median']:.2f} nm")
    ax2.set_xlabel(xlabel, fontsize=25)
    ax2.set_ylabel(angle_col, fontsize=25)
    ax2.legend(fontsize=20)

    # Panel 2: counts where M_norm = 0
    ax3.bar(centers, y, width=width, color=color, alpha=0.5, edgecolor='gray')
    ax3.axvline(0, color='k', ls='--', label='Film center')
    if stats:
        ax3.axvline(stats['mean'], color='red', lw=1.5, label=f"mean = {stats['mean']:.2f} nm")
        ax3.axvline(stats['median'], color='tab:blue', lw=1.5, ls='-.',
                    label=f"median = {stats['median']:.2f} nm")
        ax3.plot([], [], ' ', label=f"variance = {stats['variance']:.2f} nm^2")
    if show_fit:
        xs = np.linspace(zedges[0], zedges[-1], 300)
        ax3.plot(xs, _gaussian(xs, A, mu, sigma), color='k', lw=2,
                 label=f"Gaussian fit (R² = {r2:.2f})")
        ax3.hlines(A / 2, mu - fwhm / 2, mu + fwhm / 2, color='k', lw=2.5, alpha=0,
                   label=f"FWHM = {fwhm:.2f} nm")
        ax3.axvline(mu, color='darkorange', lw=1, ls=':')

    ax3.legend(loc='upper left', fontsize=20)
    ax3.set_xlabel(xlabel, fontsize=25)
    ax3.set_ylabel(f'# of Bloch points per {bin_nm:g} nm', fontsize=25)
    ax3.set_xticks(centers)
    ax3.set_xticklabels([f'{v:g}' for v in centers], rotation=45)

    fig.suptitle(f'{angle_col} vs. thickness position ({bin_nm:g} nm bins) \n {title}', fontsize=30)
    plt.tight_layout()
    plt.show()

    return grouped, z0_df, misc_df, crossings, stats

In [ ]:
# geom = film_geometry(wall_df, voxel_nm=2.0)      # same geom as the other plots
# grouped, z0_df, misc_df, crossings = plot_Bloch_angle_vs_z_m0(
#     wall_df, faces, geom=geom, bloch_axis=[0], neel_axis=[-1, 1], title=f'{sample_name}')

geom = film_geometry(wall_df, voxel_nm=8.0)
grouped, z0_df, misc_df, crossings, stats = plot_Bloch_angle_vs_z_m0(
    wall_df, faces, geom=geom, voxel_nm=voxel_nm, bin_nm=8.0, color=sample_color, title=f'{sample_name}')

# grouped, z0_df, misc_df, crossings, stats = plot_Bloch_angle_vs_z_m0(
#     wall_df, faces, geom=geom, color=sample_color, voxel_nm=8.0, title=f'{sample_name}')

In [ ]:
# def compare_z0_outputs(z0_df, misc_df, crossings, geom, grouped=None,
#                        color='green', title=None, figsize=(18, 10)):
#     """
#     Compare what each output of find_M0_crossings_mesh contains, on the x_vals axis
#     (z relative to film center, nm; bins identical to plot_Bloch_angle_vs_z_m0).

#     Sets compared
#       z0_df (columns)      : one z0 per 'single' column            <- what your stats use
#       multiple (columns)   : cluster-mean z of each cluster in 'multiple' columns (dropped from z0_df)
#       all edges            : every mesh-edge crossing in `crossings`
#       edges, single cols   : edge crossings belonging to the columns that became z0_df
#       edges, multiple cols : edge crossings belonging to the dropped 'multiple' columns
#       none columns         : have no z at all, so they can only be counted, not histogrammed
#       grouped (optional)   : per-layer vertex count (where the mesh has data at all)

#     Panels
#       (a) column level: z0_df + multiple-column clusters, stacked
#       (b) edge level  : crossings in single vs. multiple columns, stacked (total = all crossings)
#       (c) shapes      : all of the above normalized to unit area, dashed lines = means
#       (d) per-layer fraction of crossings lost to 'multiple' columns (edge and column level)

#     Returns (table, stats): per-layer counts DataFrame and a summary DataFrame.
#     """
#     x_vals, vox = np.asarray(geom['x_vals']), geom['voxel_nm']
#     zedges = np.concatenate([x_vals - vox / 2, [x_vals[-1] + vox / 2]])

#     # ---- gather z values for each set ----
#     mult = misc_df[misc_df['status'] == 'multiple']
#     z_mult = (np.concatenate([np.asarray(l, float) for l in mult['z0_list_rel_nm']])
#               if len(mult) else np.array([]))
#     n_none = int((misc_df['status'] == 'none').sum())

#     sets = {
#         'z0_df (columns)':       z0_df['z0_rel_nm'].to_numpy(float),
#         'multiple (clusters)':   z_mult,
#         'all edge crossings':    crossings['z_rel_nm'].to_numpy(float),
#         'edges, single cols':    crossings.loc[crossings['status'] == 'single', 'z_rel_nm'].to_numpy(float),
#         'edges, multiple cols':  crossings.loc[crossings['status'] == 'multiple', 'z_rel_nm'].to_numpy(float),
#     }

#     counts, rows = {}, []
#     for name, v in sets.items():
#         v = v[np.isfinite(v)]
#         c, _ = np.histogram(v, bins=zedges)
#         counts[name] = c
#         rows.append({'set': name, 'n': len(v), 'n_outside_layers': len(v) - int(c.sum()),
#                      'mean_nm': v.mean() if len(v) else np.nan,
#                      'median_nm': np.median(v) if len(v) else np.nan,
#                      'std_nm': v.std(ddof=1) if len(v) > 1 else np.nan})
#     stats = pd.DataFrame(rows)
#     table = pd.DataFrame(counts, index=x_vals)
#     table.index.name = 'z_rel_nm'

#     vcount = None
#     if grouped is not None:
#         vcount = grouped['count'].fillna(0).to_numpy(float)
#         if len(vcount) != len(x_vals):
#             print("grouped has a different number of layers than geom['x_vals']; skipping vertex profile")
#             vcount = None

#     def norm(c):
#         c = np.asarray(c, float)
#         return c / c.sum() if c.sum() > 0 else c

#     # ---- printed summary ----
#     n_single, n_mult = len(z0_df), len(mult)
#     n_cols = n_single + n_mult + n_none
#     print(f"columns: single {n_single} ({n_single / n_cols:.1%}), multiple {n_mult} "
#           f"({n_mult / n_cols:.1%}), none {n_none} ({n_none / n_cols:.1%})  "
#           f"[none has no z and is not in any histogram]")
#     print(f"edge crossings: {len(crossings)} total, "
#           f"{len(sets['edges, single cols'])} in single columns, "
#           f"{len(sets['edges, multiple cols'])} in multiple columns")
#     print(stats.round(3).to_string(index=False))

#     # ---- plots ----
#     fig, axs = plt.subplots(2, 2, figsize=figsize, sharex=True)
#     (axa, axb), (axc, axd) = axs
#     w = 0.9 * vox
#     ticks = dict(ticks=x_vals, labels=[f'{v:g}' for v in x_vals], rotation=45)

#     # (a) column level
#     axa.bar(x_vals, counts['z0_df (columns)'], width=w, color=color, alpha=0.6,
#             edgecolor='gray', label=f"z0_df, single columns (n={n_single})")
#     axa.bar(x_vals, counts['multiple (clusters)'], width=w, color='tab:purple', alpha=0.9,
#             edgecolor='k', bottom=counts['z0_df (columns)'],
#             label=f"multiple-column clusters (n={len(z_mult)} from {n_mult} columns)")
#     axa.set_ylabel('# of columns / clusters', fontsize=14)
#     axa.set_title(f'(a) column level (+ {n_none} "none" columns with no z)', fontsize=14)
#     axa.legend(fontsize=11)

#     # (b) edge level
#     axb.bar(x_vals, counts['edges, single cols'], width=w, color=color, alpha=0.6,
#             edgecolor='gray', label=f"edges in single columns (n={len(sets['edges, single cols'])})")
#     axb.bar(x_vals, counts['edges, multiple cols'], width=w, color='tab:purple', alpha=0.9,
#             edgecolor='k', bottom=counts['edges, single cols'],
#             label=f"edges in multiple columns (n={len(sets['edges, multiple cols'])})")
#     axb.set_ylabel('# of mesh-edge crossings', fontsize=14)
#     axb.set_title('(b) edge level (stack total = all crossings)', fontsize=14)
#     axb.legend(fontsize=11)

#     # (c) normalized shapes
#     style = {'z0_df (columns)': dict(color=color, lw=3),
#              'all edge crossings': dict(color='k', lw=2, ls='--'),
#              'multiple (clusters)': dict(color='tab:purple', lw=2)}
#     for name, kw in style.items():
#         if counts[name].sum() > 0:
#             axc.stairs(norm(counts[name]), zedges, label=name, **kw)
#             m = stats.loc[stats['set'] == name, 'mean_nm'].iloc[0]
#             axc.axvline(m, color=kw['color'], ls=':', lw=1.5)
#     if vcount is not None:
#         axc.stairs(norm(vcount), zedges, color='tab:orange', lw=2, alpha=0.8,
#                    label='mesh vertices per layer (grouped)')
#     axc.set_ylabel('fraction (each set sums to 1)', fontsize=14)
#     axc.set_title('(c) shapes, dotted = mean', fontsize=14)
#     axc.legend(fontsize=11)

#     # (d) fraction lost to multiple columns, per layer
#     with np.errstate(invalid='ignore', divide='ignore'):
#         f_edge = counts['edges, multiple cols'] / counts['all edge crossings']
#         f_col = counts['multiple (clusters)'] / (counts['multiple (clusters)'] + counts['z0_df (columns)'])
#     axd.plot(x_vals, f_edge, 'o-', color='tab:purple', label='edge level')
#     axd.plot(x_vals, f_col, 's--', color='k', label='column level')
#     axd.set_ylabel('fraction in "multiple" columns', fontsize=14)
#     axd.set_title('(d) where in z the dropped columns sit', fontsize=14)
#     axd.legend(fontsize=11)

#     for a in axs.ravel():
#         a.axvline(0, color='k', ls='--', lw=0.8)
#     for a in (axc, axd):
#         a.set_xlabel('z relative to film center (nm)', fontsize=14)
#         a.set_xticks(**{k: v for k, v in ticks.items() if k == 'ticks'})
#         a.set_xticklabels(ticks['labels'], rotation=45)
#     fig.suptitle(f'Comparison of z0_df / misc_df / crossings\n{title}', fontsize=18)
#     plt.tight_layout()
#     plt.show()
#     return table, stats

# table, cmp_stats = compare_z0_outputs(z0_df, misc_df, crossings, geom,
#                                       grouped=grouped, color=sample_color, title=sample_name)

In [ ]:
print(grouped.keys())
print(z0_df.keys())
# print(misc_df.keys()) ###################################### I need to check out misc_df
print(crossings.keys())
# print(stats.keys())

In [ ]:
from scipy.signal import find_peaks

def _gaussian(x, A, mu, sigma):
    return A * np.exp(-(x - mu)**2 / (2 * sigma**2))

def _bimodal(x, A1, mu1, s1, A2, mu2, s2):
    return _gaussian(x, A1, mu1, s1) + _gaussian(x, A2, mu2, s2)


def fit_bloch_distribution(grouped, z0_df, stats, color = 'green', voxel_nm = 2.0, bin_nm = 8.0):

    # x_vals = grouped['z_rel_nm'].to_numpy()
    # zedges = np.concatenate([x_vals - voxel_nm / 2, [x_vals[-1] + voxel_nm / 2]])
    # y, _ = np.histogram(z0_df['z0_rel_nm'], bins=zedges)
    # width = 0.9 * voxel_nm
    xlabel = 'z relative to film center (nm)'

    x_vals = grouped['z_rel_nm'].to_numpy()
    # zedges = np.concatenate([x_vals - voxel_nm / 2, [x_vals[-1] + voxel_nm / 2]])

    bin_nm = bin_nm

    lo = x_vals.min() - voxel_nm / 2
    hi = x_vals.max() + voxel_nm / 2
    zedges = np.arange(lo, hi + bin_nm, bin_nm)       # + bin_nm so the last edge covers hi
    centers = 0.5 * (zedges[:-1] + zedges[1:])


    y, _ = np.histogram(z0_df['z0_rel_nm'], bins=zedges)
    xlabel = 'z relative to film center (nm)'

    # ---- single Gaussian fit (baseline for comparison) ----
    p0_1 = [y.max(), stats['mean'], max(stats['std'], bin_nm)]
    popt1, _ = curve_fit(_gaussian, centers, y, p0=p0_1, maxfev=10000)
    r2_1 = 1 - np.sum((y - _gaussian(centers, *popt1))**2) / np.sum((y - y.mean())**2)
    print(f"1 Gaussian: mu = {popt1[1]:.2f}, sigma = {abs(popt1[2]):.2f}, R² = {r2_1:.3f}")


    # # ---- single Gaussian fit (baseline for comparison) ----
    # p0_1 = [y.max(), stats['mean'], max(stats['std'], voxel_nm)]
    # popt1, _ = curve_fit(_gaussian, x_vals, y, p0=p0_1, maxfev=10000)
    # r2_1 = 1 - np.sum((y - _gaussian(x_vals, *popt1))**2) / np.sum((y - y.mean())**2)
    # print(f"1 Gaussian: mu = {popt1[1]:.2f}, sigma = {abs(popt1[2]):.2f}, R² = {r2_1:.3f}")

    # ---- bimodal fit ----
    # Initial guess: split the data at the mean, one peak in each half

    pk, _ = find_peaks(y, prominence=0.05 * y.max())
    print(f'pk = {pk} with length {len(pk)}')
    if len(pk) >= 2:
        left  = pk[0]
        right = pk[1]

        print(f'left = {left} right =  {right}')

        s0 = 2 * voxel_nm   # starting width, change this if the fit struggles

        p0_2 = [y[left].max(),  x_vals[left],  s0,
                y[right].max(), x_vals[right], s0]

        print(f'p0_2 = {p0_2}')

        # Keep sigmas positive and amplitudes non-negative
        lower = [0, x_vals.min(), 0.1, 0, x_vals.min(), 0.1]
        upper = [np.inf, x_vals.max(), np.inf, np.inf, x_vals.max(), np.inf]

        popt2, _ = curve_fit(_bimodal, x_vals, y, p0=p0_2, bounds=(lower, upper), maxfev=20000)
        A1, mu1, s1, A2, mu2, s2 = popt2

        # make sure peak 1 is the left one
        if mu1 > mu2:
            A1, mu1, s1, A2, mu2, s2 = A2, mu2, s2, A1, mu1, s1

        r2_2 = 1 - np.sum((y - _bimodal(x_vals, A1, mu1, s1, A2, mu2, s2))**2) / np.sum((y - y.mean())**2)
        print(f"2 Gaussians: mu1 = {mu1:.2f}, sigma1 = {s1:.2f} | "
            f"mu2 = {mu2:.2f}, sigma2 = {s2:.2f} | R² = {r2_2:.3f}")

    # ---- plot ----
    # fig, ax = plt.subplots(figsize=(10, 5))

    # ---- plot ----
    fig, ax = plt.subplots(1, 2, figsize=(20, 7))


    # Panel 1: M_norm profile
    ax[0].errorbar(x_vals, grouped['mean'], yerr=grouped['sem'], fmt='o-',
                 capsize=3, color=color)
    ax[0].axhline(0, color='magenta', ls='--', label='Bloch (0)')
    ax[0].axhline(1, color='cyan', ls='--', label='Neel (±1)')
    ax[0].axhline(-1, color='cyan', ls='--')
    ax[0].axvline(0, color='k', ls='--', label='Film center')

    if len(pk) >= 2:
        ax[0].axvline(mu1, color = 'red', linestyle = '--', label = f'mu1 = {mu1:.2f}')
        ax[0].axvline(mu2, color = 'blue', linestyle = '--', label = f'mu2 = {mu2:.2f}')
    ax[0].axvline((popt1[1]), color = 'tab:gray', linestyle = ':', label = f'mu = {popt1[1]:.2f}')
        
    ax[0].set_xlabel(xlabel, fontsize=15)
    ax[0].set_ylabel('M_normal value', fontsize=15)
    ax[0].legend(fontsize=14)

    # Panel 1: histogram and fit
    xs = np.linspace(x_vals.min(), x_vals.max(), 400)

    ax[1].bar(centers, y, width=0.9 * bin_nm, color=color, alpha=0.5, edgecolor='gray')
    ax[1].axvline(0, color='k', ls='--', label='Film center')
    ax[1].plot(centers, _gaussian(centers, *popt1), 'tab:gray', lw=2, label=f'1 Gaussian (R² = {r2_1:.2f})')
    ax[1].axvline((popt1[1]), color = 'tab:gray', linestyle = ':', label = f'mu = {popt1[1]:.2f}')
    if len(pk) >= 2:
        ax[1].plot(centers, _bimodal(centers, A1, mu1, s1, A2, mu2, s2), color = 'purple', lw=2,
                label=f'2 Gaussians (R² = {r2_2:.2f})')
        ax[1].axvline(mu1, color = 'red', linestyle = '--', label = f'mu1 = {mu1:.2f}')
        ax[1].axvline(mu2, color = 'blue', linestyle = '--', label = f'mu2 = {mu2:.2f}')
        ax[1].axvline(((mu1 + mu2)/2), color = 'purple', linestyle = ':', label = f'mu_avg = {((mu1 + mu2)/2):.2f}')

    ax[1].legend(loc='upper left', fontsize=14)
    ax[1].set_xlabel('z relative to film center (nm)', fontsize=15)
    ax[1].set_ylabel('# of Bloch points per z', fontsize=15)
    ax[1].set_xticks(centers)
    plt.tight_layout()
    plt.show()

    return popt1 , np.array([A1, mu1, s1, A2, mu2, s2])


#popt1, popt2 = fit_bloch_distribution(grouped, z0_df, stats, color=sample_color, voxel_nm=voxel_nm)

## Investigate the bimodal distribution 

In [ ]:
def Investigate_bimodal_distribution (grouped, z0_df, stats, color = 'green', voxel_nm = 2.0, scatter_key = 'kappa', lag = 1 ):

    x_vals = grouped['z_rel_nm'].to_numpy()
    # zedges = np.concatenate([x_vals - voxel_nm / 2, [x_vals[-1] + voxel_nm / 2]])

    bin_nm = 8.0

    lo = x_vals.min() - voxel_nm / 2
    hi = x_vals.max() + voxel_nm / 2
    zedges = np.arange(lo, hi + bin_nm, bin_nm)       # + bin_nm so the last edge covers hi
    centers = 0.5 * (zedges[:-1] + zedges[1:])


    y, _ = np.histogram(z0_df['z0_rel_nm'], bins=zedges)
    width = 0.9 * voxel_nm
    xlabel = 'z relative to film center (nm)'

    # ---- single Gaussian fit (baseline for comparison) ----
    p0_1 = [y.max(), stats['mean'], max(stats['std'], voxel_nm)]
    popt1, _ = curve_fit(_gaussian, centers, y, p0=p0_1, maxfev=10000)
    r2_1 = 1 - np.sum((y - _gaussian(centers, *popt1))**2) / np.sum((y - y.mean())**2)
    print(f"1 Gaussian: mu = {popt1[1]:.2f}, sigma = {abs(popt1[2]):.2f}, R² = {r2_1:.3f}")

    # ---- bimodal fit ----
    # Initial guess: split the data at the mean, one peak in each half

    pk, _ = find_peaks(y, prominence=0.05 * y.max())
    if len(pk) >= 2:

        print(f'pk = {pk} with length {len(pk)}')
        left  = pk[0]
        right = pk[1]
        print(f'left = {left} right =  {right}')
        s0 = 2 * voxel_nm   # starting width, change this if the fit struggles
        p0_2 = [y[left].max(),  x_vals[left],  s0,
                y[right].max(), x_vals[right], s0]
        print(f'p0_2 = {p0_2}')
        # Keep sigmas positive and amplitudes non-negative
        lower = [0, x_vals.min(), 0.1, 0, x_vals.min(), 0.1]
        upper = [np.inf, x_vals.max(), np.inf, np.inf, x_vals.max(), np.inf]
        popt2, _ = curve_fit(_bimodal, x_vals, y, p0=p0_2, bounds=(lower, upper), maxfev=20000)
        A1, mu1, s1, A2, mu2, s2 = popt2
        # make sure peak 1 is the left one
        if mu1 > mu2:
            A1, mu1, s1, A2, mu2, s2 = A2, mu2, s2, A1, mu1, s1
        r2_2 = 1 - np.sum((y - _bimodal(x_vals, A1, mu1, s1, A2, mu2, s2))**2) / np.sum((y - y.mean())**2)
        print(f"2 Gaussians: mu1 = {mu1:.2f}, sigma1 = {s1:.2f} | "
            f"mu2 = {mu2:.2f}, sigma2 = {s2:.2f} | R² = {r2_2:.3f}")


    # ---- plot ----
    fig, ax = plt.subplots(2, 2, figsize=(20, 10))

    # Panel 1: histogram and fit
    xs = np.linspace(x_vals.min(), x_vals.max(), 400)

    ax[0,0].bar(centers, y, width=0.9 * bin_nm, color=color, alpha=0.5, edgecolor='gray')
    ax[0,0].axvline(0, color='k', ls='--', label='Film center')
    ax[0,0].plot(xs, _gaussian(xs, *popt1), 'tab:gray', lw=2, label=f'1 Gaussian (R² = {r2_1:.2f})')
    ax[0,0].axvline((popt1[1]), color = 'tab:gray', linestyle = ':', label = f'mu = {popt1[1]:.2f}')

    if len(pk) >= 2:
        ax[0,0].plot(xs, _bimodal(xs, A1, mu1, s1, A2, mu2, s2), color = 'purple', lw=2,
                label=f'2 Gaussians (R² = {r2_2:.2f})')
        ax[0,0].axvline(mu1, color = 'red', linestyle = '--', label = f'mu1 = {mu1:.2f}')
        ax[0,0].axvline(mu2, color = 'blue', linestyle = '--', label = f'mu2 = {mu2:.2f}')
        ax[0,0].axvline(((mu1 + mu2)/2), color = 'purple', linestyle = ':', label = f'mu_avg = {((mu1 + mu2)/2):.2f}')

    ax[0,0].legend(loc='upper left', fontsize=14)
    ax[0,0].set_xlabel('z relative to film center (nm)', fontsize=15)
    ax[0,0].set_ylabel('# of Bloch points per z', fontsize=15)
    ax[0,0].set_xticks(centers)
    ax[0,0].set_xticklabels([f'{v:g}' for v in centers], rotation=45)
    ax[0,0].set_title('Histogram')


#############################################################################################################################
    # ---- Run-sequence plot ----

    z = z0_df['z0_rel_nm'].to_numpy()
    idx = np.arange(len(z))

    ax[0,1].scatter(idx, z, c=color, s=15, alpha=0.6)
    ax[0,1].set_xlabel('idx', fontsize=15)
    ax[0,1].set_ylabel('z relative to film center (nm)', fontsize=15)
    ax[0,1].axhline(stats['mean'], color='red', ls='--', label=f"mean = {stats['mean']:.2f}")
    ax[0,1].axhline(0, color='k', ls=':', label='Film center')
    ax[0,1].set_title('Run-sequence plot')
    ax[0,1].legend()

    # ---- 2. Scatter plot: z vs. kappa ----

    if scatter_key == 'x':
        # # position along the wall: project onto the main direction of the (x, y) cloud
        # xy = z0_df[['x', 'y']].to_numpy()

        # xy_c = xy - xy.mean(axis=0)
        # _, _, vt = np.linalg.svd(xy_c, full_matrices=False)
        # s = xy_c @ vt[0]                      # coordinate along the wall (nm)
        # ax[1,0].scatter(s, z, c=color, s=15, alpha=0.6)   # <-- change 'x' to your column
        lim = np.abs(z).max()
        sc = ax[1,0].scatter(z0_df['x'], z0_df['y'], c=z, cmap='coolwarm', s=15,
                            vmin=-lim, vmax=lim)
        plt.colorbar(sc, ax=ax[1,0], label='z0 rel. to center (nm)')
        ax[1,0].set_xlabel('x (nm)'); ax[1,0].set_ylabel('y (nm)')
        ax[1,0].set_aspect('equal'); ax[1,0].set_title('z0 map')

    else:

        cluster_nm = 100
        w = np.ones(len(z0_df))
        z0_df['bx'] = np.floor(z0_df['x'] / cluster_nm).astype(int)   # was d['x']; use z0_df to be safe
        z0_df['by'] = np.floor(z0_df['y'] / cluster_nm).astype(int)
        cl = z0_df.groupby(['bx', 'by']).ngroup().to_numpy()
        n_cl = cl.max() + 1
        print(f'n_cl = {n_cl}')

        # k = z0_df[scatter_key].to_numpy()
        # kappa_tol = 0.005
        # cls = np.where(k < -kappa_tol, 0, np.where(k > kappa_tol, 2, 1))
        # colors = ['tab:blue', 'tab:gray', 'tab:red']
        # names = ['kappa < -tol', '|kappa| <= tol', 'kappa > tol']   # reuse your existing names if defined

        # for j, c in enumerate(colors):
        #     m = cls == j
        #     ax[1,0].scatter(k[m], z[m], c=c, s=15, alpha=0.6, label=f'{names[j]} (n={m.sum()})')

        # ax[1,0].axhline(0, color='k', ls=':')
        # ax[1,0].axhline(stats['mean'], color='red', ls='--', label=f"mean = {stats['mean']:.2f}")
        # ax[1,0].set_xlabel(scatter_key, fontsize=14)
        # ax[1,0].set_ylabel('z relative to film center (nm)', fontsize=14)
        # ax[1,0].set_title(f'z vs. {scatter_key}')
        # ax[1,0].set_ylim(np.min(xs), np.max(xs))
        # ax[1,0].legend(fontsize=10)


        ax[1,0].scatter(z0_df[scatter_key], abs(z), c=color,  s=15, alpha=0.6)   # <-- change 'x' to your column
        # ax[1,0].axhline(0, color='k', ls=':')
        # ax[1,0].axhline(stats['mean'], color='red', ls='--', label=f"mean = {stats['mean']:.2f}")
        ax[1,0].set_xlabel(scatter_key, fontsize=14)
        ax[1,0].set_ylabel('z relative to film center (nm)', fontsize=14)
        ax[1,0].set_title(f'z vs. {scatter_key}')
        # ax[1,0].set_ylim(np.min(xs), np.max(xs) )


        # # shuffle the label -> color mapping so neighboring tiles get different colors
        # rng = np.random.default_rng(0)
        # cl_color = rng.permutation(n_cl)[cl] % cluster_nm     # tab20 has 20 distinct colors

        # sc = ax[1,0].scatter(z0_df[scatter_key], z, c=cl_color, cmap='bwr',
        #                     s=15, alpha=0.8)
        # ax[1,0].axhline(0, color='k', ls=':')
        # ax[1,0].axhline(stats['mean'], color='red', ls='--', label=f"mean = {stats['mean']:.2f}")
        # ax[1,0].set_xlabel(scatter_key, fontsize=14)
        # ax[1,0].set_ylabel('z relative to film center (nm)', fontsize=14)
        # ax[1,0].set_title(f'z vs. {scatter_key} (colored by {20}-nm spatial cluster)')
        # ax[1,0].set_ylim(np.min(xs), np.max(xs))

    # ---- 3. Lag plot: each point vs. the previous one ----


    k = z0_df[scatter_key].to_numpy()
    kappa_tol = 0.005
    cls = np.where(k < -kappa_tol, 0, np.where(k > kappa_tol, 2, 1))
    colors = ['tab:blue', 'tab:gray', 'tab:red']
    names = ['kappa < -tol', '|kappa| <= tol', 'kappa > tol']   # reuse your existing names if defined

    for j, c in enumerate(colors):
        m = cls == j
        series = pd.Series(z[m])
        pd.plotting.lag_plot(series, lag=lag, ax=ax[1,1], c=c, s=15, alpha=0.6, label = f'{names[j]}')


    # series = pd.Series(z)
    # pd.plotting.lag_plot(series, lag=lag, ax=ax[1,1], c=color, s=15, alpha=0.6, label = f'lag = {lag}')
    ax[1,1].set_xlabel('z[i] (nm)', fontsize=14)
    ax[1,1].set_ylabel('z[i+1] (nm)', fontsize=14)
    ax[1,1].set_title('Lag plot')
    ax[1,1].legend(loc = 'upper right')

    #ADD both scatter plots and an autocorrelation plot
#############################################################################################################################


    plt.tight_layout()
    plt.show()

    return popt1 , np.array([A1, mu1, s1, A2, mu2, s2])



print(grouped.keys())
print(z0_df.keys())
# print(misc_df.keys()) ###################################### I need to check out misc_df
#print(crossings.keys())
# print(stats.keys())

popt1, popt2 = Investigate_bimodal_distribution(grouped, z0_df, stats, color=sample_color, voxel_nm=voxel_nm, scatter_key = 'kappa', lag = 1 )

# popt1, popt2 = Investigate_bimodal_distribution(grouped, z0_df, stats, color=sample_color, voxel_nm=voxel_nm, scatter_key = 'x', lag = 2 )


# popt1, popt2 = Investigate_bimodal_distribution(grouped, z0_df, stats, color=sample_color, voxel_nm=voxel_nm, scatter_key = 'x', lag = 1 )


In [ ]:
def plot_z0_vs_inplane(z0_df, color='green'):
    z = z0_df['z0_rel_nm'].to_numpy()
    xy = z0_df[['x', 'y']].to_numpy()

    # position along the wall: project onto the main direction of the (x, y) cloud
    xy_c = xy - xy.mean(axis=0)
    _, _, vt = np.linalg.svd(xy_c, full_matrices=False)
    s = xy_c @ vt[0]                      # coordinate along the wall (nm)

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # 1. Spatial map: where in the wall do high/low z0 live?
    sc = axes[0].scatter(z0_df['x'], z0_df['y'], c=z, cmap='coolwarm', s=15)
    plt.colorbar(sc, ax=axes[0], label='z0 rel. to center (nm)')
    axes[0].set_xlabel('x (nm)'); axes[0].set_ylabel('y (nm)')
    axes[0].set_aspect('equal'); axes[0].set_title('z0 map')

    # 2. z0 vs position along the wall
    axes[1].scatter(s, z, c=color, s=15, alpha=0.6)
    axes[1].set_xlabel('position along wall (nm)')
    axes[1].set_ylabel('z0 rel. to center (nm)')
    axes[1].set_title('z0 vs. position along wall')

    # 3. z0 vs curvature
    ok = z0_df['kappa'].notna()
    axes[2].scatter(z0_df.loc[ok, 'kappa'], z[ok], c=color, s=15, alpha=0.6)
    axes[2].set_xlabel('kappa'); axes[2].set_ylabel('z0 rel. to center (nm)')
    axes[2].set_title('z0 vs. curvature')

    for ax in axes[1:]:
        ax.axhline(0, color='k', ls=':')
    plt.tight_layout()
    plt.show()

plot_z0_vs_inplane(z0_df, color=sample_color)

In [ ]:
stop

### Compare abs value of curvature

In [ ]:


def _abs_class_info(thresholds):
    t = list(thresholds)
    names = ([f'|κ| ≤ {t[0]:g}']
             + [f'{a:g} < |κ| ≤ {b:g}' for a, b in zip(t[:-1], t[1:])]
             + [f'|κ| > {t[-1]:g}'])
    colors = ['tab:gray'] + list(plt.cm.YlOrRd(np.linspace(0.45, 0.9, len(names) - 1)))
    return names, colors


def _abs_kappa_stats(z, k, w, cl, thresholds, n_bins, kappa_max, n_boot, seed):
    """Binned weighted mean of z vs |kappa| and per-class means, with cluster-bootstrap errors."""
    rng = np.random.default_rng(seed)
    ka = np.abs(k)
    cl = pd.factorize(cl)[0]

    edges = np.linspace(0, kappa_max, n_bins + 1)
    centers = 0.5 * (edges[:-1] + edges[1:])
    vb = pd.cut(pd.Series(ka), edges, labels=False, include_lowest=True).to_numpy()
    vm = ~np.isnan(vb)
    mean, err, n_pts, n_cl, _ = _binned_cluster_stats(
        vb[vm].astype(int), z[vm], w[vm], pd.factorize(cl[vm])[0], n_bins, n_boot, rng)

    names, colors = _abs_class_info(thresholds)
    cls = np.digitize(ka, thresholds, right=True)        # 0: |k| <= t0, 1: t0 < |k| <= t1, ...
    c_mean, c_err, c_n, c_ncl, c_boots = _binned_cluster_stats(
        cls, z, w, cl, len(names), n_boot, rng)

    summary = pd.DataFrame({'kappa_abs_center': centers, 'z_nm_mean': mean, 'z_nm_err': err,
                            'n_points': n_pts, 'n_clusters': n_cl})
    class_summary = pd.DataFrame({'class': names, 'z_nm_mean': c_mean, 'z_nm_err': c_err,
                                  'n_points': c_n, 'n_clusters': c_ncl})
    return summary, class_summary, cls, c_boots, colors


def _plot_abs_panels(summary, class_summary, cls, c_boots, colors, z, w, geom, thresholds,
                     min_count, normalize, color, ylim, ylabel, hist_ylabel, title):
    summary['shown'] = summary['n_points'] >= min_count
    keep = summary['shown'].to_numpy()
    names = class_summary['class'].tolist()
    n_cls = len(names)

    fig, (ax, axh) = plt.subplots(1, 2, figsize=(13, 4.5))
    ax.errorbar(summary['kappa_abs_center'][keep], summary['z_nm_mean'][keep],
                yerr=summary['z_nm_err'][keep], fmt='o-', capsize=3, color=color)
    ax.axhline(0, color='gray', ls='--', lw=0.8, label='film center')
    for i, t in enumerate(thresholds):
        ax.axvline(t, color='k', ls=':', lw=0.8, label='class threshold' if i == 0 else None)
    ax.set_xlabel('|centerline curvature| |κ| (1/nm)')
    ax.set_ylabel(ylabel)
    ax.set_title(f'Mean z vs. |κ|\n {title}')
    ax.legend(fontsize=8)
    if ylim is not None:
        ax.set_ylim(*ylim)

    x_vals, vox = geom['x_vals'], geom['voxel_nm']
    zedges = np.concatenate([x_vals - vox / 2, [x_vals[-1] + vox / 2]])
    wb = 0.8 * vox / n_cls
    for j in range(n_cls):
        m = cls == j
        counts, _ = np.histogram(z[m], bins=zedges, weights=w[m])
        vals = counts / counts.sum() if normalize and counts.sum() > 0 else counts
        axh.bar(x_vals + (j - (n_cls - 1) / 2) * wb, vals, width=wb, color=colors[j],
                label=f'{names[j]} (n={m.sum()})')
        axh.axvline(class_summary['z_nm_mean'].iloc[j], color=colors[j], ls='--', lw=1.5)
    axh.axvline(0, color='k', ls=':', lw=0.8)
    axh.set_xlabel('z relative to film center (nm)')
    axh.set_ylabel(hist_ylabel if not normalize else f'fraction of {hist_ylabel}')
    axh.set_xticks(x_vals)
    axh.set_xticklabels([f'{v:g}' for v in x_vals], rotation=45)
    axh.set_title('z distribution by |κ| class\n(dashed = class mean)')
    axh.legend(fontsize=8)
    plt.tight_layout()
    plt.show()

    print("\nClass summary (z in nm from film center, cluster-bootstrap error):")
    print(class_summary.to_string(index=False))
    diff = c_boots[:, -1] - c_boots[:, 0]
    print(f"\nmean z({names[-1]}) - mean z({names[0]}) = "
          f"{np.nanmean(diff):.2f} ± {np.nanstd(diff):.2f} nm")

In [ ]:
def plot_z0_vs_abs_curvature(z0_df, geom, thresholds=(0.005,), n_bins=10, kappa_max=None,
                             min_count=10, cluster_nm=20.0, normalize=True,
                             n_boot=500, seed=0, color='green', ylim=None, title=None):
    """One z0 per lateral column. Bootstrap resamples cluster_nm x cluster_nm blocks of wall."""
    d = z0_df.dropna(subset=['kappa']).copy()
    z = d['z0_rel_nm'].to_numpy()
    k = d['kappa'].to_numpy()
    w = np.ones(len(d))
    cl = d.groupby([np.floor(d['x'] / cluster_nm).astype(int),
                    np.floor(d['y'] / cluster_nm).astype(int)]).ngroup().to_numpy()
    if kappa_max is None:
        kappa_max = np.percentile(np.abs(k), 98)
    print(f"{len(d)} columns with valid kappa, {cl.max() + 1} bootstrap blocks; "
          f"|kappa| bins span 0 to {kappa_max:.4f} 1/nm")

    summary, class_summary, cls, c_boots, colors = _abs_kappa_stats(
        z, k, w, cl, thresholds, n_bins, kappa_max, n_boot, seed)
    _plot_abs_panels(summary, class_summary, cls, c_boots, colors, z, w, geom, thresholds,
                     min_count, normalize, color, ylim,
                     'M_normal = 0 position relative to film center (nm)', 'columns',
                     f'z0 (interpolated zero crossings)\n{title}')
    return summary, class_summary, d

In [ ]:
def plot_bloch_v_abs_curvature(wall_df, geom, thresholds=(0.005,), n_bins=10, kappa_max=None,
                               category='bloch', category_col='category',
                               curv_col='kappa', area_col='v_area',
                               exclude_boundary=True, edge_tol=1.0, min_count=10,
                               normalize=True, n_boot=500, seed=0,
                               color='green', ylim=None, title=None):
    voxel_nm, z_center = geom['voxel_nm'], geom['z_center']
    d = wall_df.loc[wall_df[category_col] == category, ['x', 'y', 'z', curv_col, area_col]].copy()

    if exclude_boundary:
        on_edge = np.zeros(len(d), dtype=bool)
        for c in ['x', 'y', 'z']:
            lo, hi = wall_df[c].min(), wall_df[c].max()
            on_edge |= (d[c] <= lo + edge_tol) | (d[c] >= hi - edge_tol)
        print(f"excluded {on_edge.sum()} / {len(d)} {category} vertices on the volume boundary")
        d = d[~on_edge]
    d = d.dropna(subset=[curv_col, 'z', area_col])
    d = d[d[area_col] > 0].copy()

    z = (d['z'] - z_center).to_numpy()
    k = d[curv_col].to_numpy()
    w = d[area_col].to_numpy()
    cl = d.groupby([np.rint(d['x'] / voxel_nm).astype(int),
                    np.rint(d['y'] / voxel_nm).astype(int)]).ngroup().to_numpy()
    if kappa_max is None:
        kappa_max = np.percentile(np.abs(k), 98)
    print(f"{len(d)} vertices in {cl.max() + 1} columns; "
          f"|kappa| bins span 0 to {kappa_max:.4f} 1/nm")

    summary, class_summary, cls, c_boots, colors = _abs_kappa_stats(
        z, k, w, cl, thresholds, n_bins, kappa_max, n_boot, seed)
    _plot_abs_panels(summary, class_summary, cls, c_boots, colors, z, w, geom, thresholds,
                     min_count, normalize, color, ylim,
                     f'{category} position relative to film center (nm)', f'{category} area',
                     f'{category} band (|M_normal| < tol, area-weighted)\n{title}')
    return summary, class_summary, d

In [ ]:
def plot_class_summary(summaries, xlim=None, title=None, default_label='class summary',
                       absolute=False):
    if isinstance(summaries, pd.DataFrame):
        summaries = {default_label: summaries}
    markers = ['o', 's', 'D', '^']
    first = next(iter(summaries.values()))
    n_rows = len(first)
    if absolute:
        _, colors = _abs_class_info([0] * (n_rows - 1))      # only the colors are used
    else:
        colors = ['tab:blue', 'tab:gray', 'tab:red']
    n = len(summaries)
    offs = np.linspace(-0.18, 0.18, n) if n > 1 else np.zeros(1)

    fig, ax = plt.subplots(figsize=(7, 1.2 + 0.9 * n_rows))
    for m, (name, cs) in enumerate(summaries.items()):
        for i in range(len(cs)):
            ax.errorbar(cs['z_nm_mean'].iloc[i], i + offs[m], xerr=cs['z_nm_err'].iloc[i],
                        fmt=markers[m], color=colors[i], capsize=3, ms=7,
                        mfc=colors[i] if m == 0 else 'white')
    ax.axvline(0, color='k', ls=':', lw=0.8)
    ax.set_yticks(range(n_rows))
    ax.set_yticklabels(first['class'])
    ax.invert_yaxis()
    ax.set_xlabel('mean z relative to film center (nm)')
    if xlim is not None:
        ax.set_xlim(*xlim)
    handles = [Line2D([], [], marker=markers[m], color='k', ls='',
                      mfc='k' if m == 0 else 'white', label=lab)
               for m, lab in enumerate(summaries)]
    ax.legend(handles=handles, fontsize=8)
    ax.set_title(title)
    plt.tight_layout()
    plt.show()

In [ ]:
geom = film_geometry(wall_df, voxel_nm=voxel_nm)
abs_thr = (0.005,)          # or (0.005, 0.01) for flat / moderate / tight

# zero-crossing version
z0_df, misc_df, crossings = find_M0_crossings_mesh(wall_df, faces, geom, angle_col='M_n_n',
                                                   voxel_nm=voxel_nm)
z0_abs_summary, z0_abs_class, z0_abs_pts = plot_z0_vs_abs_curvature(
    z0_df, geom, thresholds=abs_thr, color=sample_color, title='sample 452')

# band version
band_abs_summary, band_abs_class, band_abs_pts = plot_bloch_v_abs_curvature(
    wall_df, geom, thresholds=abs_thr, color=sample_color, title='sample 452')

# compare the two definitions
plot_class_summary({'band |M_normal|<tol': band_abs_class,
                    'M_normal = 0 crossing': z0_abs_class},
                   xlim=(-10, 10), absolute=True, title='sample 452')

In [ ]:
stop

# old

In [ ]:
def plot_bloch_z_vs_curvature(wall_df, classified_df, thickness_nm=None,
                              curv_col='kappa', category='bloch',
                              exclude_boundary=False, edge_tol=1.0,
                              per_column=False, voxel_nm=2.0,
                              n_bins=10, kappa_range=(-0.01, 0.01),
                              min_count=10, title=None):
    """
    Mean z of classified (Bloch) vertices vs. local curvature, as a fraction of
    film thickness (right axis in nm), plus a z-vs-kappa 2D histogram that is
    normalized within each kappa bin.

    thickness_nm     : film thickness. Default = wall_df['z'].max() - wall_df['z'].min().
    exclude_boundary : drop vertices within edge_tol (nm) of the volume faces
                       (x, y, z min/max), where the isosurface is clipped by the crop.
    per_column       : average z and kappa within each lateral voxel column first, so
                       regions with denser mesh don't get more weight.
    Returns (binned, d): the binned summary and the per-point data actually plotted.
    """
    d = classified_df[classified_df['category'] == category].copy()
    if curv_col not in d.columns:
        d[curv_col] = wall_df.loc[d.index, curv_col]   # assumes shared index

    z_min, z_max = wall_df['z'].min(), wall_df['z'].max()
    if thickness_nm is None:
        thickness_nm = z_max - z_min
    print(f"film thickness used: {thickness_nm:.1f} nm  (z from {z_min:.1f} to {z_max:.1f})")

    n0 = len(d)
    if exclude_boundary:
        on_edge = np.zeros(len(d), dtype=bool)
        for c in ['x', 'y', 'z']:
            lo, hi = wall_df[c].min(), wall_df[c].max()
            on_edge |= (d[c] <= lo + edge_tol) | (d[c] >= hi - edge_tol)
        d = d[~on_edge]
        print(f"excluded {n0 - len(d)} / {n0} {category} vertices on the volume boundary")

    d = d.dropna(subset=[curv_col, 'z'])

    if per_column:
        d['cx'] = np.rint(d['x'] / voxel_nm).astype(int)
        d['cy'] = np.rint(d['y'] / voxel_nm).astype(int)
        d = (d.groupby(['cx', 'cy'])
               .agg(z=('z', 'mean'), **{curv_col: (curv_col, 'mean')})
               .reset_index())
        print(f"{len(d)} lateral columns after averaging")

    d['z_frac'] = (d['z'] - z_min) / thickness_nm

    edges = np.linspace(*kappa_range, n_bins + 1)
    centers = 0.5 * (edges[:-1] + edges[1:])
    d['k_bin'] = pd.cut(d[curv_col], bins=edges)
    g = d.groupby('k_bin', observed=False)
    binned = pd.DataFrame({
        'z_frac_mean': g['z_frac'].mean(),
        'z_frac_median': g['z_frac'].median(),
        'z_frac_std': g['z_frac'].std(),
        'count': g['z_frac'].count(),
    })
    binned['z_frac_sem'] = binned['z_frac_std'] / np.sqrt(binned['count'])
    binned['z_nm_mean'] = binned['z_frac_mean'] * thickness_nm
    binned['kappa_center'] = centers
    keep = (binned['count'] >= min_count).to_numpy()

    fig = plt.figure(figsize=(12, 6))
    gs = fig.add_gridspec(2, 2, height_ratios=[3, 1])
    ax = fig.add_subplot(gs[0, 0])
    ax2 = fig.add_subplot(gs[1, 0], sharex=ax)
    ax3 = fig.add_subplot(gs[:, 1])

    ax.errorbar(centers[keep], binned['z_frac_mean'].to_numpy()[keep],
                yerr=binned['z_frac_sem'].to_numpy()[keep], fmt='o-', capsize=3,
                label='mean ± SEM')
    ax.plot(centers[keep], binned['z_frac_median'].to_numpy()[keep], 's--',
            color='tab:orange', ms=4, label='median')
    ax.axvline(0, color='k', ls=':', lw=0.8)
    ax.axhline(0.5, color='gray', ls='--', lw=0.8)
    ax.set_ylabel(f'{category} position (fraction of thickness)')
    ax.set_title(f'{category} z vs. curvature\n {title}')
    ax.legend()
    sec = ax.secondary_yaxis('right', functions=(lambda f: f * thickness_nm,
                                                 lambda n: n / thickness_nm))
    sec.set_ylabel('z (nm from bottom)')

    ax2.bar(centers, binned['count'], width=np.diff(edges), color='gray', edgecolor='k')
    ax2.axvline(0, color='k', ls=':', lw=0.8)
    ax2.set_xlabel('local curvature κ')
    ax2.set_ylabel('# points')

    n_z = max(int(round(thickness_nm / voxel_nm)), 1)
    zedges = np.linspace(0, 1, n_z + 1)
    H, _, _ = np.histogram2d(d[curv_col], d['z_frac'], bins=[edges, zedges])
    colsum = H.sum(axis=1, keepdims=True)
    Hn = np.divide(H, colsum, out=np.full_like(H, np.nan), where=colsum >= min_count)
    pm = ax3.pcolormesh(edges, zedges, Hn.T, cmap='viridis')
    plt.colorbar(pm, ax=ax3, label='fraction of points within κ bin')
    ax3.axvline(0, color='w', ls=':', lw=0.8)
    ax3.set_xlabel('local curvature κ')
    ax3.set_ylabel('z (fraction of thickness)')
    ax3.set_title('z distribution within each κ bin')

    plt.tight_layout()
    plt.show()
    return binned.reset_index(drop=True), d

In [ ]:

def vertex_areas(verts_xyz, faces):
    """Area of mesh attributed to each vertex (1/3 of each adjacent triangle)."""
    v = np.asarray(verts_xyz, float)
    tri = v[faces]                                    # (M, 3, 3)
    tri_area = 0.5 * np.linalg.norm(
        np.cross(tri[:, 1] - tri[:, 0], tri[:, 2] - tri[:, 0]), axis=1)
    area = np.zeros(len(v))
    for k in range(3):
        np.add.at(area, faces[:, k], tri_area / 3.0)
    return area

wall_df['v_area'] = vertex_areas(verts_xyz, faces)

In [ ]:
binned_bloch, bloch_pts = plot_bloch_z_vs_curvature(
    wall_df, classified_normal, thickness_nm=thickness, title='sample 452')

binned_bloch, bloch_pts = plot_bloch_z_vs_curvature(
    wall_df, classified_normal, thickness_nm=thickness,exclude_boundary=False, title='sample 452')


# same thing, one value per lateral pixel column
binned_bloch_col, _ = plot_bloch_z_vs_curvature(
    wall_df, classified_normal, thickness_nm=thickness, per_column=True)

In [ ]:
# import numpy as np
# import pandas as pd
# import matplotlib.pyplot as plt
# from scipy import stats





def _slope(centers, means, keep):
    ok = keep & np.isfinite(means)
    if ok.sum() < 3:
        return np.nan
    return np.polyfit(centers[ok], means[ok], 1)[0]


def compare_weighting_z_vs_curvature(wall_df, classified_df, thickness_nm,
                                     category='bloch', curv_col='kappa',
                                     area_col='v_area', voxel_nm=2.0,
                                     exclude_boundary=True, edge_tol=1.0,
                                     n_bins=10, kappa_range=(-0.01, 0.01),
                                     min_count=10, n_boot=500, seed=0, title=None):
    """
    Three-way comparison of mean z (fraction of thickness) vs. curvature:
      'per vertex' : every vertex weighted equally
      'per column' : vertices averaged within each lateral (x,y) pixel column
                     (z and kappa both), then each column weighted equally
      'area'       : every vertex weighted by its attributed mesh area

    Error bars: cluster bootstrap, resampling lateral columns (std of bootstrap means).
    min_count   : bins with fewer points (vertices, or columns for 'per column') are hidden.
    Returns (summary, pts, cols):
      summary : long-form DataFrame, one row per (method, kappa bin), incl. slope info
      pts     : per-vertex data used
      cols    : per-column data used
    """
    if area_col not in wall_df.columns:
        raise ValueError(f"wall_df['{area_col}'] missing. Compute it with vertex_areas(verts_xyz, faces).")

    rng = np.random.default_rng(seed)
    z_min = wall_df['z'].min()

    d = classified_df[classified_df['category'] == category][['x', 'y', 'z']].copy()
    d[curv_col] = wall_df.loc[d.index, curv_col]
    d[area_col] = wall_df.loc[d.index, area_col]

    n0 = len(d)
    if exclude_boundary:
        on_edge = np.zeros(len(d), dtype=bool)
        for c in ['x', 'y', 'z']:
            lo, hi = wall_df[c].min(), wall_df[c].max()
            on_edge |= (d[c] <= lo + edge_tol) | (d[c] >= hi - edge_tol)
        d = d[~on_edge]
        print(f"excluded {n0 - len(d)} / {n0} {category} vertices on the volume boundary")

    d = d.dropna(subset=[curv_col, 'z', area_col])
    d = d[d[area_col] > 0]
    d['z_frac'] = (d['z'] - z_min) / thickness_nm
    d['cx'] = np.rint(d['x'] / voxel_nm).astype(int)
    d['cy'] = np.rint(d['y'] / voxel_nm).astype(int)
    d['col_id'] = d.groupby(['cx', 'cy']).ngroup()

    # per-column table
    cols = (d.groupby('col_id')
              .agg(cx=('cx', 'first'), cy=('cy', 'first'),
                   z_frac=('z_frac', 'mean'), kappa=(curv_col, 'mean'),
                   n_vert=('z_frac', 'size'), area=(area_col, 'sum'))
              .reset_index())
    print(f"{len(d)} vertices in {len(cols)} lateral columns")

    edges = np.linspace(*kappa_range, n_bins + 1)
    centers = 0.5 * (edges[:-1] + edges[1:])

    def to_bin(k):
        b = pd.cut(pd.Series(np.asarray(k)), edges, labels=False)
        return b.to_numpy()

    # ---- the three methods ----
    vb = to_bin(d[curv_col])
    vm = ~np.isnan(vb)
    cb = to_bin(cols['kappa'])
    cm = ~np.isnan(cb)

    methods = {}
    methods['per vertex'] = _binned_cluster_stats(
        vb[vm].astype(int), d['z_frac'].to_numpy()[vm], np.ones(vm.sum()),
        d['col_id'].to_numpy()[vm], n_bins, n_boot, rng)
    methods['per column'] = _binned_cluster_stats(
        cb[cm].astype(int), cols['z_frac'].to_numpy()[cm], np.ones(cm.sum()),
        np.arange(len(cols))[cm], n_bins, n_boot, rng)
    methods['area'] = _binned_cluster_stats(
        vb[vm].astype(int), d['z_frac'].to_numpy()[vm], d[area_col].to_numpy()[vm],
        d['col_id'].to_numpy()[vm], n_bins, n_boot, rng)

    colors = {'per vertex': 'tab:blue', 'per column': 'tab:orange', 'area': 'tab:green'}
    offsets = {'per vertex': -0.08, 'per column': 0.0, 'area': 0.08}
    bw = edges[1] - edges[0]

    rows, slopes = [], {}
    for name, (mean, err, n_pts, n_cl, boots) in methods.items():
        keep = n_pts >= min_count
        s = _slope(centers, mean, keep)
        s_boot = np.array([_slope(centers, b, keep) for b in boots])
        slopes[name] = (s, np.nanstd(s_boot))
        for i in range(n_bins):
            rows.append({'method': name, 'kappa_center': centers[i], 'z_frac_mean': mean[i],
                         'z_frac_err': err[i], 'z_nm_mean': mean[i] * thickness_nm,
                         'n_points': n_pts[i], 'n_columns': n_cl[i], 'shown': keep[i]})
    summary = pd.DataFrame(rows)

    # ---- plots ----
    fig = plt.figure(figsize=(13, 7))
    gs = fig.add_gridspec(2, 1, height_ratios=[3, 1], width_ratios=[1.4, 1])
    ax = fig.add_subplot(gs[0, 0])
    axc = fig.add_subplot(gs[1, 0], sharex=ax)
    axd = fig.add_subplot(gs[:, 1])

    for name, (mean, err, n_pts, n_cl, _) in methods.items():
        keep = n_pts >= min_count
        s, se = slopes[name]
        ax.errorbar(centers[keep] + offsets[name] * bw, mean[keep], yerr=err[keep],
                    fmt='o-', capsize=3, color=colors[name],
                    label=f'{name} (slope {s:.2g} ± {se:.2g})')
    ax.axvline(0, color='k', ls=':', lw=0.8)
    ax.axhline(0.5, color='gray', ls='--', lw=0.8)
    ax.set_ylabel(f'{category} position (fraction of thickness)')
    ax.set_title(f'{category} z vs. curvature: weighting comparison\n {title}')
    ax.legend(fontsize=8)
    sec = ax.secondary_yaxis('right', functions=(lambda f: f * thickness_nm,
                                                 lambda n: n / thickness_nm))
    sec.set_ylabel('z (nm from bottom)')

    for name, (mean, err, n_pts, n_cl, _) in methods.items():
        axc.plot(centers, n_pts, 'o-', color=colors[name], label=f'{name}: points')
    axc.plot(centers, methods['per vertex'][3], 'k^--', ms=4, label='columns (vertex-binned)')
    axc.set_yscale('log')
    axc.axvline(0, color='k', ls=':', lw=0.8)
    axc.set_xlabel('local curvature κ')
    axc.set_ylabel('count')
    axc.legend(fontsize=7)

    # # sampling-density diagnostic
    # hb = axd.hexbin(np.log10(cols['n_vert']), cols['z_frac'], gridsize=25,
    #                 cmap='viridis', mincnt=1, bins='log')
    # plt.colorbar(hb, ax=axd, label='# columns (log)')
    # rho, p = stats.spearmanr(cols['n_vert'], cols['z_frac'])
    # axd.set_xlabel('log10(# vertices in column)')
    # axd.set_ylabel('column-mean z (fraction of thickness)')
    # axd.set_title(f'Density diagnostic: Spearman ρ = {rho:.2f} (p = {p:.2g})')

    plt.tight_layout()
    plt.show()

    print("\nSlope of binned mean z_frac vs. kappa (per unit kappa), bootstrap ± std:")
    for name, (s, se) in slopes.items():
        print(f"  {name:11s}: {s:.4g} ± {se:.2g}")
    return summary, d, cols

In [ ]:
wall_df['v_area'] = vertex_areas(verts_xyz, faces)   # once

summary, bloch_pts, bloch_cols = compare_weighting_z_vs_curvature(
    wall_df, classified_normal, thickness_nm=thickness, title='sample 452')

# pivot for a side-by-side table
print(summary[summary['shown']].pivot(index='kappa_center', columns='method',
                                      values='z_frac_mean'))

In [ ]:

def plot_crossings_by_curvature_class(zero_df, bounds, kappa_tol=0.002, normalize=False):
    """
    Per-z-layer crossing counts, split into kappa < -tol, |kappa| <= tol, kappa > tol.
    normalize=True shows the fraction within each class instead of raw counts.
    """

    df = zero_df.dropna(subset=['kappa'])
    layers = np.arange(df.index.min(), df.index.max() + 1)
    
    classes = {
        f'κ < -{kappa_tol:.1f}': df[df['kappa'] < -kappa_tol],
        f'|κ| ≤ {kappa_tol:.1f}': df[df['kappa'].abs() <= kappa_tol],
        f'κ > {kappa_tol:.1f}': df[df['kappa'] > kappa_tol],
    }
    colors = ['tab:blue', 'tab:gray', 'tab:red']

    fig, ax = plt.subplots(figsize=(7, 4))
    w = 0.27
    for j, ((name, sub), c) in enumerate(zip(classes.items(), colors)):
        counts = sub['z0_layer'].value_counts().reindex(layers, fill_value=0)
        vals = counts / counts.sum() if (normalize and counts.sum()) else counts
        ax.bar(layers + (j - 1) * w, vals.to_numpy(), width=w, color=c,
               label=f'{name} (n={len(sub)})')
    ax.set_xlabel('z voxel layer')
    ax.set_ylabel('fraction of crossings' if normalize else '# crossings')
    ax.set_xticks(layers)
    ax.legend()
    plt.tight_layout()
    plt.show()

def plot_bloch_area_two_panel(wall_df, zero_df, classified_df, thickness_nm,
                              category='bloch', curv_col='kappa', area_col='v_area',
                              voxel_nm=2.0, exclude_boundary=True, edge_tol=1.0,
                              n_bins=10, kappa_range=(-0.01, 0.01), kappa_tol=0.002,
                              min_count=10, n_zbins=None, n_boot=500, seed=0,
                              title=None):
    """
    Left : area-weighted mean z (fraction of thickness) vs. kappa, cluster-bootstrap error bars.
    Right: area-weighted z distribution (density) for kappa < -tol, |kappa| <= tol, kappa > tol,
           with the weighted mean of each class marked.

    kappa_tol : |kappa| <= kappa_tol counts as "equal to 0".
    n_zbins   : number of z bins in the histogram (default: one per voxel, thickness/voxel_nm).
    Returns (summary, class_summary, d):
      summary       : per-kappa-bin means/errors (line plot)
      class_summary : per-class weighted mean, bootstrap error, n points, n columns
      d             : per-vertex data used
    """
    rng = np.random.default_rng(seed)
    z_min = wall_df['z'].min()

    d = classified_df[classified_df['category'] == category][['x', 'y', 'z']].copy()
    d[curv_col] = wall_df.loc[d.index, curv_col]
    d[area_col] = wall_df.loc[d.index, area_col]

    n0 = len(d)
    if exclude_boundary:
        on_edge = np.zeros(len(d), dtype=bool)
        for c in ['x', 'y', 'z']:
            lo, hi = wall_df[c].min(), wall_df[c].max()
            on_edge |= (d[c] <= lo + edge_tol) | (d[c] >= hi - edge_tol)
        d = d[~on_edge]
        print(f"excluded {n0 - len(d)} / {n0} {category} vertices on the volume boundary")

    d = d.dropna(subset=[curv_col, 'z', area_col])
    d = d[d[area_col] > 0].copy()
    d['z_frac'] = (d['z'] - z_min) / thickness_nm
    d['cx'] = np.rint(d['x'] / voxel_nm).astype(int)
    d['cy'] = np.rint(d['y'] / voxel_nm).astype(int)
    d['col_id'] = d.groupby(['cx', 'cy']).ngroup()

    z = d['z_frac'].to_numpy()
    w = d[area_col].to_numpy()
    k = d[curv_col].to_numpy()
    cl = d['col_id'].to_numpy()

    # ---------- left panel: area-weighted mean vs kappa bins ----------
    edges = np.linspace(*kappa_range, n_bins + 1)
    centers = 0.5 * (edges[:-1] + edges[1:])
    vb = pd.cut(pd.Series(k), edges, labels=False).to_numpy()
    vm = ~np.isnan(vb)
    mean, err, n_pts, n_cl, _ = _binned_cluster_stats(
        vb[vm].astype(int), z[vm], w[vm], cl[vm], n_bins, n_boot, rng)
    keep = n_pts >= min_count
    summary = pd.DataFrame({'kappa_center': centers, 'z_frac_mean': mean, 'z_frac_err': err,
                            'z_nm_mean': mean * thickness_nm, 'n_points': n_pts,
                            'n_columns': n_cl, 'shown': keep})

    # ---------- right panel: three kappa classes ----------
    cls = np.where(k < -kappa_tol, 0, np.where(k > kappa_tol, 2, 1))
    names = [f'κ < -{kappa_tol}', f'|κ| ≤ {kappa_tol}', f'κ > {kappa_tol}']
    colors = ['tab:blue', 'tab:gray', 'tab:red']
    c_mean, c_err, c_n, c_ncl, c_boots = _binned_cluster_stats(
        cls, z, w, cl, 3, n_boot, rng)
    class_summary = pd.DataFrame({'class': names, 'z_frac_mean': c_mean, 'z_frac_err': c_err,
                                  'z_nm_mean': c_mean * thickness_nm,
                                  'n_points': c_n, 'n_columns': c_ncl})

    if n_zbins is None:
        n_zbins = max(int(round(thickness_nm / voxel_nm)), 1)
    zedges = np.linspace(0, max(1.0, z.max()), n_zbins + 1)

    # ---------- plot ----------
    fig, (ax) = plt.subplots(1, 1, figsize=(8, 5))

    ax.errorbar(centers[keep], mean[keep], yerr=err[keep], fmt='o-', capsize=3,
                color='gold')
    ax.axvline(0, color='k', ls=':', lw=0.8, label = 'kappa = 0')
    ax.axhline(0.5, color='gray', ls='--', lw=0.8, label = 'film center')
    # ax.axvspan(-kappa_tol, kappa_tol, color='gray', alpha=0.12, label='"κ ≈ 0" range')
    ax.set_xlabel('local curvature κ')
    ax.set_ylabel(f'{category} position (fraction of thickness)')
    ax.set_title(f'Mean {category} z vs. curvature \n {title}')
    ax.legend(fontsize=8)
    ax.set_ylim(0.40, 0.60)
    # sec = ax.secondary_yaxis('right', functions=(lambda f: f * thickness_nm,
    #                                              lambda n: n / thickness_nm))
    # sec.set_ylabel('z (nm from bottom)')

    # for i, (name, c) in enumerate(zip(names, colors)):
    #     sel = cls == i
    #     if not sel.any():
    #         continue
    #     axh.hist(z[sel], bins=zedges, weights=w[sel], density=True,
    #              histtype='stepfilled', alpha=0.25, color=c)
    #     axh.hist(z[sel], bins=zedges, weights=w[sel], density=True,
    #              histtype='step', lw=2, color=c,
    #              label=f'{name} (n={sel.sum()}, mean {c_mean[i]:.3f} ± {c_err[i]:.3f})')
    #     axh.axvline(c_mean[i], color=c, ls='--', lw=1.5)
    # axh.axvline(0.5, color='k', ls=':', lw=0.8)
    # axh.set_xlabel(f'{category} position (fraction of thickness)')
    # axh.set_ylabel('area-weighted probability density')
    # axh.set_title('z distribution by curvature class\n(dashed = weighted mean)')
    # axh.legend(fontsize=8)

    plt.tight_layout()
    plt.show()

    #plot_crossings_by_curvature_class(zero_df, bounds, kappa_tol=0.0001, normalize=True)

    # ---------- printed summary ----------
    print("\nClass summary (area-weighted, cluster-bootstrap error):")
    print(class_summary.to_string(index=False))
    diff = c_boots[:, 2] - c_boots[:, 0]
    print(f"\nmean z_frac(κ>tol) - mean z_frac(κ<-tol) = {np.nanmean(diff):.3f} ± {np.nanstd(diff):.3f}"
          f"  ({np.nanmean(diff) * thickness_nm:.2f} nm)")
    return summary, class_summary, d

In [ ]:
summary, class_summary, pts = plot_bloch_area_two_panel(
    wall_df, zero_df, classified_normal, thickness_nm=thickness, kappa_tol=0.002, title=f'sample {sample_name}')

In [ ]:
def plot_Bloch_angle_vs_z(wall_df, bounds, angle_col='M_normal',
                          bloch_axis=None, neel_axis=None, title=None,
                          max_gap=2.0, drop_degenerate=False, dz=2.0):
    """
    Plots (top) mean angle_col vs z voxel layer and (bottom) the number of
    M_normal = 0 crossings found in each z voxel layer.

    Zero crossings are found per (x, y) voxel column by sorting vertices in z
    and linearly interpolating wherever angle_col changes sign (or is exactly 0).

    Parameters
    ----------
    max_gap : float
        Max z separation (in voxels) between consecutive vertices across which a
        sign change is trusted. Prevents interpolating across holes in the mesh.
    drop_degenerate : bool
        Exclude vertices flagged degenerate when finding crossings.
    dz : float
        z spacing in physical units, used only for the 'z0_phys' column.

    Returns
    -------
    grouped : DataFrame, mean/std/count of angle_col per z layer (as before)
    zero_df : DataFrame, one row per column with exactly ONE crossing
              columns: col_x, col_y, z0_vox (fractional voxel), z0_layer (rounded), z0_phys
    misc_df : DataFrame, columns with no crossing or multiple crossings
              columns: col_x, col_y, status ('none' or 'multiple'), z0_list (voxel units)
    """
    df = wall_df.copy()
    z_lo, z_hi = bounds
    z_min = z_lo - z_lo
    z_max = z_hi - z_lo

    df['z_layer'] = df['iz'].round().astype(int)

    # Restrict to the physical film thickness
    n_before = len(df)
    df = df[(df['z_layer'] >= z_min) & (df['z_layer'] <= z_max)]
    print(f"{len(df)} / {n_before} vertices fall within layers {z_min}-{z_max}")

    # ---- mean profile (same as before) ----
    grouped = df.groupby('z_layer')[angle_col].agg(['mean', 'std', 'count'])

    # ---- zero crossings per (x, y) column ----
    d = df
    if drop_degenerate and 'degenerate' in d.columns:
        d = d[~d['degenerate'].astype(bool)]
    d = d.dropna(subset=[angle_col]).copy()
    d['col_x'] = np.rint(d['ix']).astype(int)
    d['col_y'] = np.rint(d['iy']).astype(int)

    good_rows, misc_rows = [], []
    for (cx, cy), g in d.groupby(['col_x', 'col_y']):
        g = g.sort_values('iz')
        z = g['iz'].to_numpy()
        m = g[angle_col].to_numpy()

        crossings = z[m == 0].tolist()                      # exact zeros
        if len(z) >= 2:
            s = np.sign(m)
            for i in np.where((s[:-1] * s[1:]) < 0)[0]:     # sign changes
                if (z[i + 1] - z[i]) <= max_gap:
                    crossings.append(z[i] - m[i] * (z[i + 1] - z[i]) / (m[i + 1] - m[i]))

        if len(crossings) == 1:
            z0 = crossings[0]
            good_rows.append({'col_x': cx, 'col_y': cy, 'z0_vox': z0,
                              'z0_layer': int(round(z0)), 'z0_phys': z0 * dz})
        elif len(crossings) == 0:
            misc_rows.append({'col_x': cx, 'col_y': cy, 'status': 'none', 'z0_list': []})
        else:
            misc_rows.append({'col_x': cx, 'col_y': cy, 'status': 'multiple',
                              'z0_list': sorted(crossings)})

    zero_df = pd.DataFrame(good_rows, columns=['col_x', 'col_y', 'z0_vox', 'z0_layer', 'z0_phys'])
    misc_df = pd.DataFrame(misc_rows, columns=['col_x', 'col_y', 'status', 'z0_list'])

    n_cols = len(zero_df) + len(misc_df)
    print(f"{len(zero_df)} / {n_cols} columns have exactly one {angle_col}=0 crossing")
    if len(misc_df):
        print(misc_df['status'].value_counts().to_string())

    # ---- plots ----
    fig, (ax, ax2) = plt.subplots(1, 2, figsize=(18, 6), sharex=True)

    ax.errorbar(grouped.index, grouped['mean'], yerr=grouped['std'],
                fmt='o-', capsize=3)
    ax.axhline(0, color='k', lw=0.8, ls=':')
    if bloch_axis is not None:
        for deg in bloch_axis:
            ax.axhline(deg, color='magenta', label='Bloch Wall')
    if neel_axis is not None:
        for deg in neel_axis:
            ax.axhline(deg, color='cyan', label='Neel Wall')
    ax.set_ylabel(angle_col, fontsize = '15')
    ax.set_xlabel('z voxel layer', fontsize = '15')

    layers = np.arange(z_min, z_max + 1)
    counts = zero_df['z0_layer'].value_counts().reindex(layers, fill_value=0)
    ax2.bar(layers, counts.to_numpy(), width=0.9, color='green', alpha = 0.5)
    ax2.set_xlabel('z voxel layer', fontsize = '15')
    ax2.set_ylabel(f'# of {angle_col} = 0 crossings', fontsize = '15')
    ax2.set_xticks(layers)

    fig.suptitle(f'{angle_col} vs. thickness position \n {title}', fontsize = '20')


    plt.tight_layout()
    plt.show()

    return grouped, zero_df, misc_df

bounds = [0,int(volume['shape'][0])]
grouped, zero_df, misc_df = plot_Bloch_angle_vs_z(wall_df, bounds, bloch_axis = [0] , neel_axis = [np.min(Mz), np.max(Mz)], title=f'{sample_name}')

In [ ]:
bounds = [0,int(volume['shape'][0])]
grouped, zero_df, misc_df = plot_Bloch_angle_vs_z(wall_df, bounds, bloch_axis = [0] , neel_axis = [np.min(Mz), np.max(Mz)], title=f'{sample_name}')

In [ ]:
def find_M0_crossings(wall_df, bounds, angle_col='M_normal', curv_col='kappa',
                      max_gap=2.0, drop_degenerate=True, dz=8.0):
    """
    Find M_normal = 0 crossings per (x, y) voxel column and attach local curvature.

    Returns
    -------
    zero_df : one row per column with exactly ONE crossing
              columns: col_x, col_y, z0_vox, z0_layer, z0_phys, kappa
    misc_df : columns with no crossing or multiple crossings
              columns: col_x, col_y, status, z0_list
    """
    z_lo, z_hi = bounds
    z_min, z_max = 0, z_hi - z_lo

    d = wall_df.copy()
    d['z_layer'] = d['iz'].round().astype(int)
    d = d[(d['z_layer'] >= z_min) & (d['z_layer'] <= z_max)]
    if drop_degenerate and 'degenerate' in d.columns:
        d = d[~d['degenerate'].astype(bool)]
    d = d.dropna(subset=[angle_col]).copy()
    d['col_x'] = np.rint(d['ix']).astype(int)
    d['col_y'] = np.rint(d['iy']).astype(int)

    good_rows, misc_rows = [], []
    for (cx, cy), g in d.groupby(['col_x', 'col_y']):
        g = g.sort_values('iz')
        z = g['iz'].to_numpy()
        m = g[angle_col].to_numpy()
        k = g[curv_col].to_numpy()

        crossings = []                                   # list of (z0, kappa0)
        for i in np.where(m == 0)[0]:                    # exact zeros
            crossings.append((z[i], k[i]))
        if len(z) >= 2:
            s = np.sign(m)
            for i in np.where((s[:-1] * s[1:]) < 0)[0]:  # sign changes
                if (z[i + 1] - z[i]) <= max_gap:
                    t = m[i] / (m[i] - m[i + 1])         # fraction from i to i+1
                    crossings.append((z[i] + t * (z[i + 1] - z[i]),
                                      k[i] + t * (k[i + 1] - k[i])))

        if len(crossings) == 1:
            z0, k0 = crossings[0]
            good_rows.append({'col_x': cx, 'col_y': cy, 'z0_vox': z0,
                              'z0_layer': int(round(z0)), 'z0_phys': z0 * dz,
                              'kappa': k0})
        elif len(crossings) == 0:
            misc_rows.append({'col_x': cx, 'col_y': cy, 'status': 'none', 'z0_list': []})
        else:
            misc_rows.append({'col_x': cx, 'col_y': cy, 'status': 'multiple',
                              'z0_list': sorted(c[0] for c in crossings)})

    zero_df = pd.DataFrame(good_rows, columns=['col_x', 'col_y', 'z0_vox', 'z0_layer',
                                               'z0_phys', 'kappa'])
    misc_df = pd.DataFrame(misc_rows, columns=['col_x', 'col_y', 'status', 'z0_list'])

    print(f"{len(zero_df)} / {len(zero_df) + len(misc_df)} columns have exactly one crossing")
    print(f"{zero_df['kappa'].notna().sum()} of those have a valid kappa")
    if len(misc_df):
        print(misc_df['status'].value_counts().to_string())
    return zero_df, misc_df

def add_thickness_fraction(zero_df, bounds, voxel_nm=2.0, thickness_vox=None):
    """
    Adds z0_nm and z0_frac (0 = bottom layer, 1 = top layer) to zero_df.

    thickness_vox defaults to z_hi - z_lo, the span between the first and last
    layer centers. If you want the full film including the half-voxel edges,
    pass thickness_vox = z_hi - z_lo + 1.
    """
    z_lo, z_hi = bounds
    if thickness_vox is None:
        thickness_vox = z_hi - z_lo
    out = zero_df.copy()
    out['z0_nm'] = out['z0_vox'] * voxel_nm
    out['z0_frac'] = out['z0_vox'] / thickness_vox
    print(f"thickness = {thickness_vox} voxels = {thickness_vox * voxel_nm:.1f} nm")
    return out


def plot_z0_vs_curvature(zero_df, bounds, voxel_nm=2.0, thickness_vox=None,
                         n_bins=10, kappa_range=(-0.01, 0.01), min_count=10, title=None):
    """
    Mean z0 as a fraction of film thickness vs. local curvature (right axis in nm).
    Error bars are the SEM. Bins with fewer than min_count crossings are not plotted.
    Returns the binned DataFrame (z0_frac_mean, z0_frac_sem, z0_nm_mean, count, ...).
    """
    z_lo, z_hi = bounds
    if thickness_vox is None:
        thickness_vox = z_hi - z_lo
    thickness_nm = thickness_vox * voxel_nm

    df = add_thickness_fraction(zero_df, bounds, voxel_nm, thickness_vox)
    df = df.dropna(subset=['kappa'])
    edges = np.linspace(*kappa_range, n_bins + 1)
    df['k_bin'] = pd.cut(df['kappa'], bins=edges)

    g = df.groupby('k_bin', observed=False)
    binned = pd.DataFrame({
        'z0_frac_mean': g['z0_frac'].mean(),
        'z0_frac_std': g['z0_frac'].std(),
        'z0_nm_mean': g['z0_nm'].mean(),
        'count': g['z0_frac'].count(),
    })
    binned['z0_frac_sem'] = binned['z0_frac_std'] / np.sqrt(binned['count'])
    binned['kappa_center'] = 0.5 * (edges[:-1] + edges[1:])
    keep = binned['count'] >= min_count

    fig, (ax, ax2) = plt.subplots(2, 1, figsize=(6, 7), sharex=True,
                                  gridspec_kw={'height_ratios': [3, 1]})
    ax.errorbar(binned.loc[keep, 'kappa_center'], binned.loc[keep, 'z0_frac_mean'],
                yerr=binned.loc[keep, 'z0_frac_sem'], fmt='o-', capsize=3)
    ax.axvline(0, color='k', ls=':', lw=0.8)
    ax.axhline(0.5, color='gray', ls='--', lw=0.8)       # film midplane
    ax.set_ylabel('M_normal = 0 position (fraction of thickness)')
    ax.set_title(f'M_normal = 0 position vs. curvature\n {title}')
    sec = ax.secondary_yaxis('right', functions=(lambda f: f * thickness_nm,
                                                 lambda n: n / thickness_nm))
    sec.set_ylabel('z (nm from bottom layer)')

    ax2.bar(binned['kappa_center'], binned['count'], width=np.diff(edges),
            color='gray', edgecolor='k')
    ax2.axvline(0, color='k', ls=':', lw=0.8)
    ax2.set_xlabel('local curvature κ')
    ax2.set_ylabel('# crossings')
    plt.tight_layout()
    plt.show()
    return binned.reset_index(drop=True)


def plot_crossings_by_curvature_class(zero_df, bounds, kappa_tol=0.002, normalize=False):
    """
    Per-z-layer crossing counts, split into kappa < -tol, |kappa| <= tol, kappa > tol.
    normalize=True shows the fraction within each class instead of raw counts.
    """
    z_lo, z_hi = bounds
    layers = np.arange(0, z_hi - z_lo + 1)
    df = zero_df.dropna(subset=['kappa'])

    classes = {
        f'κ < -{kappa_tol:.1f}': df[df['kappa'] < -kappa_tol],
        f'|κ| ≤ {kappa_tol:.1f}': df[df['kappa'].abs() <= kappa_tol],
        f'κ > {kappa_tol:.1f}': df[df['kappa'] > kappa_tol],
    }
    colors = ['tab:blue', 'tab:gray', 'tab:red']

    fig, ax = plt.subplots(figsize=(7, 4))
    w = 0.27
    for j, ((name, sub), c) in enumerate(zip(classes.items(), colors)):
        counts = sub['z0_layer'].value_counts().reindex(layers, fill_value=0)
        vals = counts / counts.sum() if (normalize and counts.sum()) else counts
        ax.bar(layers + (j - 1) * w, vals.to_numpy(), width=w, color=c,
               label=f'{name}')
    ax.set_xlabel('z voxel layer')
    ax.set_ylabel('M_normal = 0 Crossings' if normalize else '# crossings')
    ax.set_xticks(layers)
    ax.legend()
    plt.tight_layout()
    plt.show()


In [ ]:
zero_df, misc_df = find_M0_crossings(wall_df, bounds)
# binned = plot_z0_vs_curvature(zero_df, bounds, n_bins=10, kappa_range=(-0.01, 0.01), title=f'sample {sample_name}')
plot_crossings_by_curvature_class(zero_df, bounds, kappa_tol=0.0001, normalize=True)

In [ ]:


def build_column_table(wall_df, zero_df, misc_df, bounds, angle_col='M_normal',
                       curv_col='kappa', drop_degenerate=True):
    """
    One row per (x, y) column: mean kappa, vertex count, z extent, crossing status.
    status in {'single', 'none', 'multiple'}; n_cross = number of crossings found.
    """
    z_lo, z_hi = bounds
    d = wall_df.copy()
    d['z_layer'] = d['iz'].round().astype(int)
    d = d[(d['z_layer'] >= 0) & (d['z_layer'] <= z_hi - z_lo)]
    if drop_degenerate and 'degenerate' in d.columns:
        d = d[~d['degenerate'].astype(bool)]
    d = d.dropna(subset=[angle_col]).copy()
    d['col_x'] = np.rint(d['ix']).astype(int)
    d['col_y'] = np.rint(d['iy']).astype(int)

    cols = (d.groupby(['col_x', 'col_y'])
              .agg(kappa=(curv_col, 'mean'), n_vert=(angle_col, 'size'),
                   iz_min=('iz', 'min'), iz_max=('iz', 'max'))
              .reset_index())
    cols['z_span'] = cols['iz_max'] - cols['iz_min']

    status = pd.concat([
        zero_df[['col_x', 'col_y']].assign(status='single', n_cross=1),
        misc_df[['col_x', 'col_y', 'status']].assign(n_cross=misc_df['z0_list'].apply(len)),
    ], ignore_index=True)
    cols = cols.merge(status, on=['col_x', 'col_y'], how='left')
    cols['status'] = cols['status'].fillna('unclassified')
    cols['abs_kappa'] = cols['kappa'].abs()

    print(cols['status'].value_counts().to_string())
    return cols


def compare_status_vs_curvature(cols, n_bins=10, kappa_range=(-0.01, 0.01), min_count=10):
    """
    (a) fraction of columns in each status vs. kappa
    (b) number of columns per kappa bin, stacked by status
    (c) kappa distribution (density) for each status
    (d) mean vertex count per column vs. kappa, by status (sampling confound check)
    Also prints median kappa / |kappa| per status and Mann-Whitney tests vs. 'single'.
    Returns (cols, frac) where frac is the status fraction per kappa bin.
    """
    order = [s for s in ['single', 'none', 'multiple'] if s in cols['status'].unique()]
    colors = {'single': 'tab:green', 'none': 'tab:red', 'multiple': 'tab:purple'}

    df = cols.dropna(subset=['kappa']).copy()
    edges = np.linspace(*kappa_range, n_bins + 1)
    centers = 0.5 * (edges[:-1] + edges[1:])
    width = np.diff(edges)
    df['bin_idx'] = pd.cut(df['kappa'], edges, labels=False)
    inr = df.dropna(subset=['bin_idx']).copy()
    inr['bin_idx'] = inr['bin_idx'].astype(int)

    ct = (pd.crosstab(inr['bin_idx'], inr['status'])
            .reindex(index=range(n_bins), columns=order, fill_value=0))
    total = ct.sum(axis=1)
    frac = ct.div(total.replace(0, np.nan), axis=0)
    frac.index = centers
    keep = (total >= min_count).to_numpy()

    fig, axs = plt.subplots(2, 2, figsize=(11, 8))

    ax = axs[0, 0]
    for s in order:
        ax.plot(centers[keep], frac[s].to_numpy()[keep], 'o-', color=colors[s], label=s)
    ax.axvline(0, color='k', ls=':', lw=0.8)
    ax.set_xlabel('column-mean curvature κ')
    ax.set_ylabel('fraction of columns')
    ax.set_title('Crossing status vs. curvature')
    ax.legend()

    ax = axs[0, 1]
    bottom = np.zeros(n_bins)
    for s in order:
        ax.bar(centers, ct[s].to_numpy(), width=width, bottom=bottom,
               color=colors[s], edgecolor='k', label=s)
        bottom += ct[s].to_numpy()
    ax.axvline(0, color='k', ls=':', lw=0.8)
    ax.set_xlabel('column-mean curvature κ')
    ax.set_ylabel('# columns')
    ax.set_title('Columns per κ bin')
    ax.legend()

    ax = axs[1, 0]
    for s in order:
        ax.hist(df.loc[df['status'] == s, 'kappa'], bins=edges, density=True,
                histtype='step', lw=2, color=colors[s], label=f"{s} (n={(df['status'] == s).sum()})")
    ax.axvline(0, color='k', ls=':', lw=0.8)
    ax.set_xlabel('column-mean curvature κ')
    ax.set_ylabel('probability density')
    ax.set_title('κ distribution by status')
    ax.legend()

    ax = axs[1, 1]
    for s in order:
        mv = inr[inr['status'] == s].groupby('bin_idx')['n_vert'].mean().reindex(range(n_bins))
        ax.plot(centers[keep], mv.to_numpy()[keep], 'o-', color=colors[s], label=s)
    ax.axvline(0, color='k', ls=':', lw=0.8)
    ax.set_xlabel('column-mean curvature κ')
    ax.set_ylabel('mean # vertices per column')
    ax.set_title('Sampling density check')
    ax.legend()

    plt.tight_layout()
    plt.show()

    # ---- summary stats ----
    print("\nMedians by status:")
    print(df.groupby('status')[['kappa', 'abs_kappa', 'n_vert', 'z_span']].median().to_string())
    if 'single' in order:
        ref = df.loc[df['status'] == 'single']
        for s in order:
            if s == 'single':
                continue
            oth = df.loc[df['status'] == s]
            if len(oth) > 2:
                for var in ['kappa', 'abs_kappa']:
                    u = stats.mannwhitneyu(ref[var], oth[var], alternative='two-sided')
                    print(f"single vs {s:8s} {var:9s}: Mann-Whitney p = {u.pvalue:.3g}")
    return cols, frac

In [ ]:
zero_df = add_thickness_fraction(zero_df, bounds, voxel_nm=8.0)
binned = plot_z0_vs_curvature(zero_df, bounds, voxel_nm=8.0, n_bins=10,
                              kappa_range=(-0.01, 0.01), title='sample 452')

cols = build_column_table(wall_df, zero_df, misc_df, bounds)
cols, frac = compare_status_vs_curvature(cols, n_bins=10, kappa_range=(-0.01, 0.01))

In [ ]:
stop

## compare to the center line calculated by ...

In [ ]:
import numpy as np
import pandas as pd
from skimage.measure import find_contours
from scipy.interpolate import splprep, splev

def extract_wall_contour_2d(Mz_layer, spacing=(8.0, 8.0)):
    """
    Extract the (largest) domain wall contour from a single 2D Mz slice,
    in physical (x, y) units, already correctly ordered along the curve.

    Parameters
    ----------
    Mz_layer : 2D array, shape (ny, nx)
    spacing : tuple
        (dy, dx) physical voxel size in nm.

    Returns
    -------
    x, y : 1D arrays, physical coordinates along the contour, in order.
    is_closed : bool
        True if the contour forms a closed loop (first/last points
        coincide), determined by find_contours itself; relevant for
        deciding whether to fit a periodic spline.
    """
    dy, dx = spacing
    contours = find_contours(Mz_layer, level=0.0)
    if len(contours) == 0:
        return None, None, None
    contour = max(contours, key=len)  # (row, col) = (y_idx, x_idx), already ordered

    y = contour[:, 0] * dy
    x = contour[:, 1] * dx

    is_closed = np.allclose(contour[0], contour[-1], atol=1e-6)
    return x, y, is_closed


def compute_curvature_for_layer(x, y, is_closed, smoothing=0.0, n_eval=300):
    """
    Fit a spline through ordered (x, y) contour points and compute
    curvature kappa analytically from the spline's derivatives.

    Parameters
    ----------
    x, y : 1D arrays
        Ordered physical coordinates along the wall contour.
    is_closed : bool
        Whether to fit a periodic spline (for closed loops, e.g. a
        skyrmion tube cross-section) or an open one (for a wall that
        exits the volume boundary).
    smoothing : float
        Spline smoothing factor passed to splprep's `s` parameter.
        0 = interpolate exactly through every point (sensitive to
        pixel-level jaggedness in the raw contour); larger values fit a
        smoother curve through the points rather than exactly on them,
        recommended if your contour looks noisy/jagged at the pixel
        scale. Start at 0 and increase gradually while visually
        checking the fit against the raw points.
    n_eval : int
        Number of points to evaluate along the fitted spline for
        reporting curvature and arc length.

    Returns
    -------
    df : pd.DataFrame
        Columns: 'x', 'y' (fitted spline positions), 's' (cumulative
        arc length along the spline, in nm), 'kappa' (signed curvature,
        per nm), 'radius' (1/|kappa|, nm; inf where kappa ~ 0).
    """
    if len(x) < 4:
        return None  # too few points to fit a meaningful spline

    per = 1 if is_closed else 0
    try:
        tck, u = splprep([x, y], s=smoothing, per=per)
    except Exception as e:
        print(f"Spline fit failed ({e}); try increasing `smoothing`.")
        return None

    u_eval = np.linspace(0, 1, n_eval, endpoint=not is_closed)
    x_fit, y_fit = splev(u_eval, tck, der=0)
    dx, dy_ = splev(u_eval, tck, der=1)
    ddx, ddy = splev(u_eval, tck, der=2)

    kappa = (dx * ddy - dy_ * ddx) / (dx**2 + dy_**2)**1.5

    seg_lengths = np.sqrt(np.diff(x_fit)**2 + np.diff(y_fit)**2)
    s = np.concatenate([[0], np.cumsum(seg_lengths)])

    radius = np.where(np.abs(kappa) > 1e-12, 1.0 / np.abs(kappa), np.inf)

    return pd.DataFrame({'x': x_fit, 'y': y_fit, 's': s,
                          'kappa': kappa, 'radius': radius})


def compute_wall_curvature_all_z(Mz_vol, z_min, z_max, spacing=(8.0, 8.0, 8.0),
                                  smoothing=0.0, n_eval=300):
    """
    Compute 2D lateral curvature of the domain wall centerline,
    independently for each z-layer, clearly tagging each result with
    its z-layer and physical z-depth.

    Parameters
    ----------
    Mz_vol : 3D array, shape (nz, ny, nx)
    z_min, z_max : int
        Range of z-layers to process (inclusive), in Mz_vol's own local
        index frame (apply any z_offset separately if needed).
    spacing : tuple
        (dz, dy, dx) physical voxel size in nm.
    smoothing, n_eval : passed through to compute_curvature_for_layer.

    Returns
    -------
    curvature_df : pd.DataFrame
        One row per evaluated spline point, with columns 'x', 'y', 's',
        'kappa', 'radius', 'z_layer' (int, local index), 'z_nm'
        (physical depth) -- so every curvature value is explicitly
        tied to the z-layer it came from.
    """
    dz, dy, dx = spacing
    all_rows = []

    for iz in range(z_min, z_max + 1):
        x, y, is_closed = extract_wall_contour_2d(Mz_vol[iz], spacing=(dy, dx))
        if x is None:
            print(f"z_layer {iz}: no contour found, skipping.")
            continue

        df_layer = compute_curvature_for_layer(x, y, is_closed,
                                                smoothing=smoothing, n_eval=n_eval)
        if df_layer is None:
            print(f"z_layer {iz}: curvature computation failed, skipping.")
            continue

        df_layer['z_layer'] = iz
        df_layer['z_nm'] = iz * dz
        df_layer['is_closed'] = is_closed
        all_rows.append(df_layer)

    if not all_rows:
        raise RuntimeError("No valid contours/curvature found in the given z range.")

    curvature_df = pd.concat(all_rows, ignore_index=True)
    print(f"Computed curvature for {curvature_df['z_layer'].nunique()} z-layers, "
          f"{len(curvature_df)} total points.")
    return curvature_df

In [ ]:
curvature_df = compute_wall_curvature_all_z(Mz_smooth, z_min=0, z_max=8,
                                             spacing=(2.0, 2.0, 2.0),
                                             smoothing=0.1)  # adjust after visual check

In [ ]:
import matplotlib.pyplot as plt

def plot_curvature_map(curvature_df, z_layers=None, cmap='coolwarm', clim=None):
    """
    Plot the wall centerline colored by signed curvature kappa, one
    panel per z-layer.

    Parameters
    ----------
    curvature_df : pd.DataFrame
        Output of compute_wall_curvature_all_z.
    z_layers : list of int or None
        Which z-layers to plot; defaults to all present in curvature_df.
    clim : tuple or None
        (vmin, vmax) for kappa color scale; if None, symmetric limits
        are computed from the max |kappa| across the selected layers.
    """
    if z_layers is None:
        z_layers = sorted(curvature_df['z_layer'].unique())

    if clim is None:
        sub = curvature_df[curvature_df['z_layer'].isin(z_layers)]
        abs_max = np.nanmax(np.abs(sub['kappa']))
        clim = (-abs_max, abs_max)

    n = len(z_layers)
    fig, axes = plt.subplots(1, n, figsize=(5 * n, 5), squeeze=False)
    axes = axes[0]

    for ax, iz in zip(axes, z_layers):
        layer = curvature_df[curvature_df['z_layer'] == iz]
        sc = ax.scatter(layer['x'], layer['y'], c=layer['kappa'],
                         cmap=cmap, vmin=clim[0]/10, vmax=clim[1]/10, s=8)
        ax.plot(layer['x'], layer['y'], color='gray', lw=0.5, alpha=0.3, zorder=0)
        ax.set_aspect('equal')
        ax.set_xlabel('x (nm)')
        ax.set_ylabel('y (nm)')
        ax.set_title(f'z_layer = {iz} (z = {layer["z_nm"].iloc[0]:.0f} nm)')
        fig.colorbar(sc, ax=ax, label='kappa (1/nm)')

    plt.tight_layout()
    plt.show()

In [ ]:
plot_curvature_map(curvature_df)

# More complicated curvature

In [ ]:
# results = curve.segment_surface_by_curvature(verts_xyz, faces,zero_tol_frac=0.05 )

# H = results["mean_curvature"]
# K = results["gaussian_curvature"]
# labels_mean  = results["labels_mean"]   # 'concave' / 'flat' / 'convex'
# labels_gauss = results["labels_gauss"]  # 'hyperbolic (saddle)' / 'parabolic/flat' / 'elliptic (dome/cup)'

In [ ]:
stop

# Analyze Iso-Surfaces


In [ ]:
def plot_angle_vs_z(wall_df, bounds, angle_col='psi_chiral',  bloch_axis = None , neel_axis = None , title = None):
    df = wall_df.copy()
    z_lo, z_hi = bounds
    z_min = z_lo - z_lo
    z_max = z_hi - z_lo
   

    df['z_layer'] = df['iz'].round().astype(int)
    # print(df['iz'].min(), df['iz'].max())
    # print(df['z_layer'].value_counts().sort_index() if 'z_layer' in df else 'not yet computed')

    # Restrict to the physical film thickness, excluding any background/
    # padding layers from the tomography reconstruction.
    n_before = len(df)
    df = df[(df['z_layer'] >= z_min) & (df['z_layer'] <= z_max)]
    print(f"{len(df)} / {n_before} vertices fall within layers {z_min}-{z_max}")


    grouped = df.groupby('z_layer')[angle_col].agg(['mean', 'std', 'count'])

    fig, ax = plt.subplots(figsize=(6, 4))
    ax.errorbar(grouped.index, grouped['mean'], yerr=grouped['std'],
                fmt='o-', capsize=3)

    if bloch_axis== None:
        print('no axis')
    else:
        print(bloch_axis)
        for deg in bloch_axis: 
            ax.axhline(deg, color = 'magenta', label = 'Bloch Wall')
        for deg in neel_axis:
            ax.axhline(deg, color = 'cyan', label = 'Neel Wall')
    ax.set_xlabel('z voxel layer')
    ax.set_xticks(range(z_min, z_max))
    ax.set_ylabel(f'{angle_col} (deg)')
    ax.set_title(f'{angle_col} vs. thickness position \n {title}')
    # plt.legend()
    plt.tight_layout()
    plt.show()
    return grouped




In [ ]:
print(np.min(Mz))

In [ ]:
bounds = [0,int(volume['shape'][0])]
plot_angle_vs_z(wall_df, bounds,angle_col='psi_unsigned',  bloch_axis = [45] , neel_axis = [0, 180], title = sample_name )
plot_angle_vs_z(wall_df,bounds, angle_col='psi_chiral', bloch_axis = [-45, 45] , neel_axis = [-180, 0, 180], title = sample_name)
plot_angle_vs_z(wall_df,bounds, angle_col='M_normal', bloch_axis = [0] , neel_axis = [np.min(Mz), np.max(Mz)], title = sample_name )



In [ ]:
plot_angle_vs_z(wall_df, bounds,angle_col='curl_magnitude', title = sample_name )
plot_angle_vs_z(wall_df,bounds, angle_col='PMA',  title = sample_name)
plot_angle_vs_z(wall_df,bounds, angle_col='iDMI', title = sample_name )

# bloch wall vs thickness and curvature

In [ ]:
def compute_average_curvature_by_z(curvature_df):
    """
    Summarize curvature per z-layer: mean, std, and count of kappa
    (and radius) across all contour points in that layer.

    Parameters
    ----------
    curvature_df : pd.DataFrame
        Output of compute_wall_curvature_all_z (columns: 'kappa',
        'radius', 'z_layer', 'z_nm').

    Returns
    -------
    per_z : pd.DataFrame
        One row per z-layer: 'z_layer', 'z_nm', 'kappa_mean',
        'kappa_std', 'radius_mean', 'radius_std', 'count'.
    """
    grouped = curvature_df.groupby('z_layer').agg(
        z_nm=('z_nm', 'first'),
        kappa_mean=('kappa', 'mean'),
        kappa_std=('kappa', 'std'),
        radius_mean=('radius', lambda r: np.nanmean(r[np.isfinite(r)])),
        radius_std=('radius', lambda r: np.nanstd(r[np.isfinite(r)])),
        count=('kappa', 'size'),
    ).reset_index()
    return grouped

per_z_curvature = compute_average_curvature_by_z(curvature_df)

In [ ]:
from scipy.spatial import cKDTree

def assign_curvature_to_wall_df(wall_df, curvature_df, max_match_dist=None):
    """
    For each wall_df vertex, find the nearest curvature_df point within
    the same z-layer (matched via rounded 'iz' vs 'z_layer'), and
    assign that point's kappa/radius to the vertex.

    Parameters
    ----------
    wall_df : pd.DataFrame
        Must contain 'x', 'y', 'iz'.
    curvature_df : pd.DataFrame
        Output of compute_wall_curvature_all_z; must contain 'x', 'y',
        'z_layer', 'kappa', 'radius'.
    max_match_dist : float or None
        If set, vertices whose nearest curvature point is farther than
        this (in nm) get NaN instead of a potentially bad match; useful
        if the two point sets don't cover exactly the same wall extent.

    Returns
    -------
    df : pd.DataFrame
        wall_df with new columns 'kappa', 'radius', 'match_dist' added.
    """
    df = wall_df.copy()
    df['z_layer_match'] = df['iz'].round().astype(int)
    df['kappa'] = np.nan
    df['radius'] = np.nan
    df['match_dist'] = np.nan

    for iz, group in df.groupby('z_layer_match'):
        curve_layer = curvature_df[curvature_df['z_layer'] == iz]
        if len(curve_layer) == 0:
            print(f"z_layer {iz}: no matching curvature points, skipping "
                  f"({len(group)} wall_df vertices unmatched).")
            continue

        tree = cKDTree(curve_layer[['x', 'y']].values)
        dist, idx = tree.query(group[['x', 'y']].values)

        kappa_vals = curve_layer['kappa'].values[idx]
        radius_vals = curve_layer['radius'].values[idx]

        if max_match_dist is not None:
            bad = dist > max_match_dist
            kappa_vals = np.where(bad, np.nan, kappa_vals)
            radius_vals = np.where(bad, np.nan, radius_vals)
            if bad.any():
                print(f"z_layer {iz}: {bad.sum()} vertices exceeded "
                      f"max_match_dist, set to NaN.")

        df.loc[group.index, 'kappa'] = kappa_vals
        df.loc[group.index, 'radius'] = radius_vals
        df.loc[group.index, 'match_dist'] = dist

    n_matched = df['kappa'].notna().sum()
    print(f"{n_matched} / {len(df)} vertices matched to a curvature value.")
    return df

wall_df = assign_curvature_to_wall_df(wall_df, curvature_df, max_match_dist=16.0)

In [ ]:
def plot_angle_vs_curvature(wall_df, angle_col='M_normal', kappa_col='kappa',
                             n_bins=15, bounds=None, bloch_axis=None, neel_axis=None,
                             title=''):
    """
    Bin wall vertices by local curvature and plot the mean +/- std of
    angle_col in each curvature bin, analogous to plot_angle_vs_z but
    with curvature as the independent variable instead of z-layer.

    Parameters
    ----------
    wall_df : pd.DataFrame
        Must contain angle_col and kappa_col (e.g. from
        assign_curvature_to_wall_df).
    angle_col : str
        Column to average, e.g. 'M_normal', 'psi_unsigned', 'psi_chiral'.
    kappa_col : str
        Column to bin by, default 'kappa'.
    n_bins : int
        Number of curvature bins to use.
    bounds : tuple (kappa_min, kappa_max) or None
        If given, restrict to vertices with kappa in this range before
        binning (e.g. to exclude extreme/noisy curvature outliers).
    bloch_axis, neel_axis : list of float or None
        Reference horizontal lines, same convention as plot_angle_vs_z.
    title : str
        Plot title.

    Returns
    -------
    grouped : pd.DataFrame
        Per-bin summary: 'kappa_bin_center', 'mean', 'std', 'count'.
    """
    df = wall_df.dropna(subset=[angle_col, kappa_col]).copy()

    if bounds is not None:
        df = df[(df[kappa_col] >= bounds[0]) & (df[kappa_col] <= bounds[1])]

    df['kappa_bin'] = pd.cut(df[kappa_col], bins=n_bins)
    grouped = df.groupby('kappa_bin')[angle_col].agg(['mean', 'std', 'count'])
    grouped['kappa_bin_center'] = grouped.index.map(lambda b: b.mid).astype(float)
    grouped = grouped.reset_index(drop=True)

    fig, ax = plt.subplots(figsize=(6, 4))
    ax.errorbar(grouped['kappa_bin_center'], grouped['mean'], yerr=grouped['std'],
                fmt='o-', capsize=3, label=angle_col)

    if bloch_axis is not None:
        for i, deg in enumerate(bloch_axis):
            ax.axhline(deg, color='magenta', linestyle='--',
                        label='Bloch ref' if i == 0 else None)
    if neel_axis is not None:
        for i, deg in enumerate(neel_axis):
            ax.axhline(deg, color='cyan', linestyle='--',
                        label='Neel ref' if i == 0 else None)

    ax.set_xlabel('local curvature, kappa (1/nm)')
    ax.set_ylabel(f'{angle_col}')
    ax.set_title(f'{angle_col} vs. wall curvature\n{title}')
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()

    return grouped

In [ ]:
grouped = plot_angle_vs_curvature(
    wall_df, angle_col='M_normal', n_bins = 15, bounds=None,
    bloch_axis=[0], neel_axis=[np.min(Mz), np.max(Mz)],
    title=sample_name
)

In [ ]:
def compute_bloch_z_vs_curvature(wall_df, angle_col='M_normal', kappa_col='kappa',
                                  ref_layer=None, max_match_dist=16.0, dz=8.0):
    """
    Locate where M_normal crosses zero through the film thickness at each
    lateral position along the wall, and pair it with local signed curvature.

    Lateral positions ("anchors") are the wall vertices in a reference layer.
    For each anchor, the nearest vertex in every other layer (in x,y, within
    max_match_dist) is linked to build an M_normal(z) profile. Zero crossings
    are found by linear interpolation between adjacent layers.

    Parameters
    ----------
    wall_df : DataFrame with 'x','y','iz', angle_col, kappa_col
        (kappa from assign_curvature_to_wall_df).
    ref_layer : int or None  (None -> middle layer)
    max_match_dist : float, nm
    dz : float, nm per z-layer, used only for the z_cross_nm column.

    Returns
    -------
    dict with:
      'anchors'     : one row per anchor with a complete profile:
                      anchor_id, x, y, n_crossings, kappa_mean
      'crossings'   : every zero crossing (anchor_id, z_cross, z_cross_nm,
                      kappa at the crossing, n_crossings)
      'selected'    : one crossing per anchor, the one closest to mid-thickness
      'no_crossing' : anchors with no zero crossing (x, y, kappa_mean)
      'z_mid'       : mid-thickness position, in iz units
    z_cross is in the same frame as 'iz' (so includes any z_offset you applied).
    """
    df = wall_df.dropna(subset=[angle_col, kappa_col]).copy()
    df['z_layer'] = df['iz'].round().astype(int)
    layers = sorted(df['z_layer'].unique())
    z = np.array(layers, dtype=float)
    z_mid = 0.5 * (z.min() + z.max())
    if ref_layer is None:
        ref_layer = layers[len(layers) // 2]

    anchors = df[df['z_layer'] == ref_layer]
    anchor_xy = anchors[['x', 'y']].values
    n = len(anchors)

    # Build M_normal(z) and kappa(z) profiles per anchor
    M = np.full((n, len(layers)), np.nan)
    K = np.full((n, len(layers)), np.nan)
    for j, L in enumerate(layers):
        g = df[df['z_layer'] == L]
        dist, idx = cKDTree(g[['x', 'y']].values).query(anchor_xy)
        ok = dist <= max_match_dist
        M[ok, j] = g[angle_col].values[idx[ok]]
        K[ok, j] = g[kappa_col].values[idx[ok]]

    complete = ~np.isnan(M).any(axis=1) & ~np.isnan(K).any(axis=1)
    print(f"{complete.sum()} / {n} anchors have a matched vertex in all "
          f"{len(layers)} layers (others dropped).")

    anchor_rows, cross_rows = [], []
    for a in np.where(complete)[0]:
        m, k = M[a], K[a]
        zc, kc = [], []
        for j in range(len(z) - 1):
            if m[j] * m[j + 1] < 0:                  # sign change between layers
                t = m[j] / (m[j] - m[j + 1])         # fractional position in [0,1]
                zc.append(z[j] + t * (z[j + 1] - z[j]))
                kc.append(k[j] + t * (k[j + 1] - k[j]))  # kappa at the crossing
        anchor_rows.append(dict(anchor_id=a, x=anchor_xy[a, 0], y=anchor_xy[a, 1],
                                n_crossings=len(zc), kappa_mean=k.mean()))
        for z_c, k_c in zip(zc, kc):
            cross_rows.append(dict(anchor_id=a, z_cross=z_c, z_cross_nm=z_c * dz,
                                   kappa=k_c, n_crossings=len(zc)))

    anchors_df = pd.DataFrame(anchor_rows)
    crossings_df = pd.DataFrame(cross_rows)

    # One crossing per anchor: closest to mid-thickness
    if len(crossings_df):
        crossings_df['dist_to_mid'] = (crossings_df['z_cross'] - z_mid).abs()
        selected_df = (crossings_df.sort_values('dist_to_mid')
                                   .groupby('anchor_id', as_index=False).first())
    else:
        selected_df = crossings_df.copy()

    no_crossing_df = anchors_df[anchors_df['n_crossings'] == 0][
        ['anchor_id', 'x', 'y', 'kappa_mean']].reset_index(drop=True)

    # --- Report ---
    counts = anchors_df['n_crossings'].value_counts().sort_index()
    total = len(anchors_df)
    print(f"\nCrossing counts across {total} anchors:")
    for nc, c in counts.items():
        label = "NO Bloch crossing (same-sign M_normal through thickness)" if nc == 0 \
                else f"{nc} crossing(s)"
        print(f"  {label}: {c} ({100 * c / total:.1f}%)")
    if len(no_crossing_df):
        print(f"\nNo-crossing anchors: mean kappa = {no_crossing_df['kappa_mean'].mean():.4g} "
              f"+/- {no_crossing_df['kappa_mean'].std():.4g} 1/nm "
              f"(all anchors: {anchors_df['kappa_mean'].mean():.4g} "
              f"+/- {anchors_df['kappa_mean'].std():.4g})")
    print(f"Mid-thickness reference z = {z_mid:.2f} (iz units)")

    return dict(anchors=anchors_df, crossings=crossings_df, selected=selected_df,
                no_crossing=no_crossing_df, z_mid=z_mid)


def plot_bloch_z_vs_curvature(result, n_bins=10, color='C0', title=''):
    """
    Left: z of the (mid-thickness-closest) M_normal = 0 crossing vs signed
    curvature, with binned mean +/- std. Red ticks along the bottom mark the
    curvature of anchors with NO crossing.
    Right: fraction of anchors per curvature bin with no crossing and with
    multiple crossings.
    """
    sel, anc, nocross = result['selected'], result['anchors'], result['no_crossing']

    k_all = np.concatenate([sel['kappa'].values, anc['kappa_mean'].values])
    edges = np.linspace(k_all.min(), k_all.max(), n_bins + 1)
    centers = 0.5 * (edges[:-1] + edges[1:])

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    # --- Left panel ---
    ax = axes[0]
    ax.scatter(sel['kappa'], sel['z_cross'], s=8, alpha=0.25, color=color)
    b = pd.cut(sel['kappa'], bins=edges, labels=False)
    g = sel.assign(b=b).groupby('b')['z_cross'].agg(['mean', 'std', 'count'])
    ax.errorbar(centers[g.index.astype(int)], g['mean'], yerr=g['std'],
                fmt='o-', color=color, capsize=3, lw=2, label='binned mean +/- std')
    ax.axhline(result['z_mid'], color='gray', ls=':', label='mid-thickness')
    if len(nocross):
        y_tick = ax.get_ylim()[0]
        ax.plot(nocross['kappa_mean'], np.full(len(nocross), y_tick), '|',
                color='red', ms=12, alpha=0.5, label=f'no crossing (n={len(nocross)})',
                clip_on=False)
    ax.axvline(0, color='k', lw=0.5)
    ax.set_xlabel('signed curvature, kappa (1/nm)')
    ax.set_ylabel('z of M_normal = 0 (iz units)')
    ax.set_title(f'Bloch crossing position vs. curvature {title}')
    ax.legend(fontsize=8)

    # --- Right panel ---
    ax = axes[1]
    ab = pd.cut(anc['kappa_mean'], bins=edges, labels=False)
    grp = anc.assign(b=ab).groupby('b')
    frac_none = grp['n_crossings'].apply(lambda s: (s == 0).mean())
    frac_multi = grp['n_crossings'].apply(lambda s: (s >= 2).mean())
    cnt = grp.size()
    w = 0.4 * (edges[1] - edges[0])
    xs = centers[frac_none.index.astype(int)]
    ax.bar(xs - w / 2, frac_none.values, width=w, color='red', alpha=0.7,
           label='no crossing')
    ax.bar(xs + w / 2, frac_multi.values, width=w, color='orange', alpha=0.7,
           label='2+ crossings')
    for xv, c in zip(xs, cnt.loc[frac_none.index]):
        ax.text(xv, 1.02, f'n={c}', ha='center', fontsize=7)
    ax.set_ylim(0, 1.12)
    ax.set_xlabel('signed curvature, kappa (1/nm)  [mean over layers]')
    ax.set_ylabel('fraction of anchors in bin')
    ax.set_title('Anomalous profiles vs. curvature')
    ax.legend(fontsize=8)

    plt.tight_layout()
    plt.show()

In [ ]:
# 1) attach curvature to the wall vertices (needs curvature_df from compute_wall_curvature_all_z)
wall_df = assign_curvature_to_wall_df(wall_df, curvature_df, max_match_dist=16.0)

# 2) compute and plot
result = compute_bloch_z_vs_curvature(wall_df, max_match_dist=16.0, dz=8.0)
plot_bloch_z_vs_curvature(result, n_bins=10, color='limegreen', title=sample_name)

# Everything is stored in tidy tables:
result['crossings']     # all crossings, including multiples
result['selected']      # the mid-thickness-closest crossing per anchor
result['no_crossing']   # locations and curvature of anchors with no Bloch crossing

In [ ]:
def plot_bloch_z_vs_curvature(result, n_bins=10, color='C0', title='', show_no_crossing=True):
    """
    z of the M_normal = 0 crossing (the one closest to mid-thickness) vs.
    signed, layer-averaged curvature, with binned mean +/- std.
    Red ticks along the bottom mark anchors with no crossing.
    """
    # attach layer-averaged curvature to each selected crossing
    sel = result['selected'].merge(
        result['anchors'][['anchor_id', 'kappa_mean']], on='anchor_id')
    nocross = result['no_crossing']

    # bin edges span all anchors so the no-crossing ticks fall in the same range
    k_all = result['anchors']['kappa_mean']
    edges = np.linspace(k_all.min(), k_all.max(), n_bins + 1)
    centers = 0.5 * (edges[:-1] + edges[1:])

    sel['bin'] = pd.cut(sel['kappa_mean'], bins=edges, labels=False, include_lowest=True)
    g = sel.groupby('bin')['z_cross'].agg(['mean', 'std', 'count']).dropna(subset=['mean'])

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.scatter(sel['kappa_mean'], sel['z_cross'], s=8, alpha=0.25, color=color)
    ax.errorbar(centers[g.index.astype(int)], g['mean'], yerr=g['std'],
                fmt='o-', color=color, capsize=3, lw=2, label='binned mean ± std')
    ax.axhline(result['z_mid'], color='gray', ls=':', label='mid-thickness')
    ax.axvline(0, color='k', lw=0.5)

    if show_no_crossing and len(nocross):
        ax.plot(nocross['kappa_mean'], np.full(len(nocross), ax.get_ylim()[0]), '|',
                color='red', ms=12, alpha=0.5, clip_on=False,
                label=f'no crossing (n={len(nocross)})')

    ax.set_xlabel('signed curvature, κ (1/nm), layer-averaged')
    ax.set_ylabel('z of M_normal = 0 (iz units)')
    ax.set_title(f'Bloch crossing position vs. curvature {title}')
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()

    return g  # per-bin summary: mean, std, count

In [ ]:
result = compute_bloch_z_vs_curvature(wall_df, max_match_dist=16.0, dz=8.0)
bin_summary = plot_bloch_z_vs_curvature(result, n_bins=10, color='limegreen', title=sample_name)

In [ ]:
stop

# highlight Bloch and Neel

In [ ]:
def classify_pure_bloch_neel(wall_df, method='normal', tol=None, z_offset_included=True):
    """
    Classify wall vertices as 'bloch', 'neel', or 'unclassified' using one
    of two strict criteria.

    Parameters
    ----------
    wall_df : pd.DataFrame
        Must contain Mx, My, Mz, nx, ny, nz (and ix, iy, iz / x, y, z for
        location bookkeeping).
    method : 'normal' or 'psi'
        'normal' -- classify using M_n = M.n_hat directly.
            Bloch: |M_n| < tol           (tangential, no normal component)
            Neel:  |M_n| within tol of |M| (fully radial/along normal)
        'psi' -- classify using the unsigned wall-type angle, computed in
            RADIANS directly from M_n and the tangential magnitude (not
            the pre-computed degree column, to avoid round-trip error).
            Bloch: |psi_rad - pi/2| < tol
            Neel:  psi_rad within tol of 0 or pi
    tol : float
        Tolerance for the chosen method. Recommended: 0.0005 for 'normal'
        (dimensionless, dot-product units), 0.005 for 'psi' (radians).
    z_offset_included : bool
        Set True if wall_df['iz'] already reflects true experimental
        voxel numbering (as it should if you applied z_offset earlier).
        This is just a bookkeeping flag/reminder, not used in the math.

    Returns
    -------
    df : pd.DataFrame
        Copy of wall_df with an added 'category' column
        ('bloch' / 'neel' / 'unclassified') and the raw comparison
        quantity used (M_n and mag, or psi_rad) as new columns.
    """
    df = wall_df.copy()
    Mx, My, Mz = df['Mx'].values, df['My'].values, df['Mz'].values
    nx_, ny_, nz_ = df['nx'].values, df['ny'].values, df['nz'].values
    M_v = np.stack([Mx, My, Mz], axis=1)
    n_v = np.stack([nx_, ny_, nz_], axis=1)

    M_n = np.einsum('ij,ij->i', M_v, n_v)
    mag_v = np.linalg.norm(M_v, axis=1)

    category = np.full(len(df), 'unclassified', dtype=object)

    if method == 'normal':
        tol = 0.0005 if tol is None else tol
        is_bloch = np.abs(M_n) < tol
        is_neel = np.abs(np.abs(M_n) - mag_v) < tol
        df['M_n'] = M_n
        df['mag'] = mag_v

    elif method == 'psi':
        tol = 0.005 if tol is None else tol
        M_t_vec = M_v - M_n[:, None] * n_v
        M_t_mag = np.linalg.norm(M_t_vec, axis=1)
        psi_rad = np.arctan2(M_t_mag, M_n)  # range [0, pi]
        is_bloch = np.abs(psi_rad - np.pi / 2) < tol
        is_neel = (np.abs(psi_rad - 0.0) < tol) | (np.abs(psi_rad - np.pi) < tol)
        df['psi_rad'] = psi_rad

    else:
        raise ValueError("method must be 'normal' or 'psi'")

    category[is_bloch] = 'bloch'
    category[is_neel] = 'neel'
    overlap = is_bloch & is_neel
    if overlap.any():
        print(f"Warning: {overlap.sum()} vertices satisfied both criteria "
              f"simultaneously (tol may be too loose, or |M| is near zero "
              f"there); labeled as 'bloch' by default overwrite order.")

    df['category'] = category
    n_bloch, n_neel = (category == 'bloch').sum(), (category == 'neel').sum()
    print(f"[{method}] tol={tol}: {n_bloch} bloch, {n_neel} neel, "
          f"{len(df) - n_bloch - n_neel} unclassified (of {len(df)} total)")

    return df

In [ ]:
def plot_pure_bloch_neel_3d(mesh_verts_xyz, classified_df,
                             bloch_color='magenta', neel_color='cyan',
                             show_unclassified=True, point_size=6):
    """
    3D PyVista scatter of classified wall vertices.
    mesh_verts_xyz : (N, 3) array matching classified_df row order
        (i.e. the same verts_xyz used to build wall_df originally).
    """
    plotter = pv.Plotter()

    is_bloch = (classified_df['category'] == 'bloch').values
    is_neel = (classified_df['category'] == 'neel').values
    is_other = ~(is_bloch | is_neel)

    if show_unclassified and is_other.any():
        plotter.add_points(mesh_verts_xyz[is_other], color='lightgray',
                            point_size=point_size * 0.5, opacity=0.15,
                            render_points_as_spheres=True)

    if is_bloch.any():
        plotter.add_points(mesh_verts_xyz[is_bloch], color=bloch_color,
                            point_size=point_size, render_points_as_spheres=True,
                            label='Bloch (pure)')
    if is_neel.any():
        plotter.add_points(mesh_verts_xyz[is_neel], color=neel_color,
                            point_size=point_size, render_points_as_spheres=True,
                            label='Neel (pure)')

    plotter.add_legend()
    plotter.add_axes()
    plotter.show()

In [ ]:
def extract_pure_locations(classified_df):
    """
    Pull out (x, y, z) physical coords and iz (true experimental voxel
    layer) for bloch and neel vertices separately.
    """
    cols = ['x', 'y', 'z', 'ix', 'iy', 'iz']
    bloch_locs = classified_df[classified_df['category'] == 'bloch'][cols].copy()
    neel_locs = classified_df[classified_df['category'] == 'neel'][cols].copy()
    return bloch_locs, neel_locs

In [ ]:
classified_normal = classify_pure_bloch_neel(wall_df, method='normal', tol=0.01)
bloch_normal_locs, neel_normal_locs = extract_pure_locations(classified_normal)

classified_psi = classify_pure_bloch_neel(wall_df, method='psi', tol=0.1)
bloch_psi_locs, neel_psi_locs = extract_pure_locations(classified_psi)

plot_pure_bloch_neel_3d(verts_xyz, classified_normal)
plot_pure_bloch_neel_3d(verts_xyz, classified_psi)

In [ ]:


def compute_arclength_per_layer(wall_df, z_bin_size=1.0):
    """
    For each z-layer, order the wall vertices along their in-plane (x, y)
    trace and assign an arc-length coordinate s. Returns wall_df with an
    added 's' column (NaN where a layer has too few points to order).
    """
    df = wall_df.copy()
    df['z_bin'] = (df['z'] / z_bin_size).round() * z_bin_size
    df['s'] = np.nan

    for z_val, group in df.groupby('z_bin'):
        if len(group) < 3:
            continue
        pts = group[['x', 'y']].values
        centroid = pts.mean(axis=0)
        angles = np.arctan2(pts[:, 1] - centroid[1], pts[:, 0] - centroid[0])
        order = np.argsort(angles)  # crude ordering; fine for a single closed/simple wall trace
        ordered_idx = group.index.values[order]
        ordered_pts = pts[order]
        seg_lengths = np.linalg.norm(np.diff(ordered_pts, axis=0), axis=1)
        s_vals = np.concatenate([[0], np.cumsum(seg_lengths)])
        df.loc[ordered_idx, 's'] = s_vals

    return df

def plot_unrolled_wall(wall_df, angle_col='psi_chiral', z_bin_size=1.0, cmap='twilight'):
    df = compute_arclength_per_layer(wall_df, z_bin_size)
    fig, ax = plt.subplots(figsize=(8, 5))
    sc = ax.scatter(df['s'], df['z'], c=df[angle_col], cmap=cmap,
                     vmin=-180 if 'chiral' in angle_col else 0,
                     vmax=180, s=15)
    fig.colorbar(sc, ax=ax, label=f'{angle_col} (deg)')
    ax.set_xlabel('arc length along wall, s (nm)')
    ax.set_ylabel('z (nm, thickness)')
    ax.set_title(f'Unrolled wall: {angle_col}')
    plt.tight_layout()
    plt.show()
    return df


def plot_slice_with_wall_overlay(sample, wall_df, field='Mz', axis='z', index=None,
                                  angle_col='psi_chiral', spacing=(8.0, 8.0, 8.0),
                                  tol=4.0, cmap_field='viridis', cmap_angle='seismic'):
    """
    Plot a 2D voxel slice (matching plot_three_slice conventions) with an
    overlay of wall_df points lying within `tol` (physical units) of that
    slice plane, colored by angle_col.
    """
    field_aliases = {'mag': 'magnitude', 'magnitude': 'magnitude',
                      'mx': 'Mx', 'my': 'My', 'mz': 'Mz'}
    key = field_aliases.get(field.lower())
    data = sample[key]
    nz, ny, nx = sample['shape']
    dz, dy, dx = spacing

    default_index = {'x': nx // 2, 'y': ny // 2, 'z': nz // 2}
    if index is None:
        index = default_index[axis]

    axis_col = {'x': 'x', 'y': 'y', 'z': 'z'}[axis]
    slice_spacing = {'x': dx, 'y': dy, 'z': dz}[axis]
    slice_phys = index * slice_spacing

    if axis == 'z':
        img = data[index, :, :]; extent = [0, nx * dx, 0, ny * dy]
        px_col, py_col = 'x', 'y'
    elif axis == 'y':
        img = data[:, index, :]; extent = [0, nx * dx, 0, nz * dz]
        px_col, py_col = 'x', 'z'
    else:  # 'x'
        img = data[:, :, index]; extent = [0, ny * dy, 0, nz * dz]
        px_col, py_col = 'y', 'z'

    nearby = wall_df[np.abs(wall_df[axis_col] - slice_phys) < tol]

    fig, ax = plt.subplots(figsize=(10, 7))
    im = ax.imshow(img, origin='lower', extent=extent, cmap=cmap_field,
                    vmin=np.nanmin(data), vmax=np.nanmax(data), aspect='auto')
    fig.colorbar(im, ax=ax, label=key)

    if len(nearby) > 0:
        sc = ax.scatter(nearby[px_col], nearby[py_col], c=nearby[angle_col],
                         cmap=cmap_angle,
                         vmin=np.min(wall_df[angle_col]), vmax=np.max(wall_df[angle_col]),
                         s=25, edgecolors='k', linewidths=0.3)
        fig.colorbar(sc, ax=ax, label=f'{angle_col} (deg)', pad=0.12)

    ax.set_title(f'{key} slice ({axis}={index}) with wall {angle_col} overlay\n'
                 f'({len(nearby)} wall points within {tol} nm of slice)')
    ax.set_xlabel(px_col); ax.set_ylabel(py_col)
    plt.tight_layout()
    plt.show()
    return nearby

In [ ]:
plot_slice_with_wall_overlay(samples[sample_name]['cropped'], wall_df, field='Mx', axis='z', index=None,
                                  angle_col='M_normal', spacing=(8.0, 8.0, 8.0),
                                  tol=4.0, cmap_field='viridis', cmap_angle='seismic')

In [ ]:
plot_slice_with_wall_overlay(samples[sample_name]['cropped'], wall_df, field='Mx', axis='x', index=None,
                                  angle_col='psi_unsigned', spacing=(8.0, 8.0, 8.0),
                                  tol=4.0, cmap_field='viridis', cmap_angle='seismic')

plot_slice_with_wall_overlay(samples[sample_name]['cropped'], wall_df, field='My', axis='x', index=None,
                                  angle_col='M_normal', spacing=(8.0, 8.0, 8.0),
                                  tol=4.0, cmap_field='viridis', cmap_angle='seismic')

# compare interpolated data

In [ ]:
def compare_interpolation_effect(classified_df, Mx_vol, My_vol, Mz_vol,
                                  method='normal', tol=None):
    """
    Compare Bloch/Neel classification using trilinear-interpolated M
    (the standard pipeline) against nearest-neighbor voxel M (no
    interpolation), to quantify how much interpolation is influencing
    the classification near strict tolerance boundaries.

    Parameters
    ----------
    classified_df : pd.DataFrame
        Output of classify_pure_bloch_neel, must contain ix, iy, iz
        (fractional voxel indices), nx, ny, nz (normal components),
        and 'category' (the interpolation-based classification).
    Mx_vol, My_vol, Mz_vol : ndarray, shape (nz, ny, nx)
        The raw (or smoothed, whichever was used upstream) voxel volumes.
    method, tol : same meaning as in classify_pure_bloch_neel; must match
        whatever was used to produce classified_df['category'], so the
        two classifications are directly comparable.

    Returns
    -------
    df : pd.DataFrame
        classified_df with added columns for the nearest-neighbor M,
        the nearest-neighbor category ('category_nn'), and a boolean
        'agrees' flag.
    """
    df = classified_df.copy()
    nz, ny, nx = Mz_vol.shape

    # Round fractional indices to nearest integer voxel, clip to valid range
    iz_nn = np.clip(df['iz'].round().astype(int), 0, nz - 1)
    iy_nn = np.clip(df['iy'].round().astype(int), 0, ny - 1)
    ix_nn = np.clip(df['ix'].round().astype(int), 0, nx - 1)

    Mx_nn = Mx_vol[iz_nn, iy_nn, ix_nn]
    My_nn = My_vol[iz_nn, iy_nn, ix_nn]
    Mz_nn = Mz_vol[iz_nn, iy_nn, ix_nn]
    M_nn = np.stack([Mx_nn, My_nn, Mz_nn], axis=1)

    n_v = df[['nx', 'ny', 'nz']].values  # reuse existing normals; only M sampling changes
    M_n_nn = np.einsum('ij,ij->i', M_nn, n_v)
    mag_nn = np.linalg.norm(M_nn, axis=1)

    category_nn = np.full(len(df), 'unclassified', dtype=object)

    if method == 'normal':
        tol = 0.0005 if tol is None else tol
        is_bloch = np.abs(M_n_nn) < tol
        is_neel = np.abs(np.abs(M_n_nn) - mag_nn) < tol
        df['M_n_nn'] = M_n_nn

    elif method == 'psi':
        tol = 0.005 if tol is None else tol
        M_t_vec_nn = M_nn - M_n_nn[:, None] * n_v
        M_t_mag_nn = np.linalg.norm(M_t_vec_nn, axis=1)
        psi_rad_nn = np.arctan2(M_t_mag_nn, M_n_nn)
        is_bloch = np.abs(psi_rad_nn - np.pi / 2) < tol
        is_neel = (np.abs(psi_rad_nn - 0.0) < tol) | (np.abs(psi_rad_nn - np.pi) < tol)
        df['psi_rad_nn'] = psi_rad_nn

    else:
        raise ValueError("method must be 'normal' or 'psi'")

    category_nn[is_bloch] = 'bloch'
    category_nn[is_neel] = 'neel'
    df['Mx_nn'], df['My_nn'], df['Mz_nn'] = Mx_nn, My_nn, Mz_nn
    df['category_nn'] = category_nn
    df['agrees'] = df['category'] == df['category_nn']

    # --- Summary ---
    n_total = len(df)
    n_agree = df['agrees'].sum()
    print(f"[{method}] Agreement: {n_agree} / {n_total} "
          f"({100 * n_agree / n_total:.1f}%)")

    confusion = pd.crosstab(df['category'], df['category_nn'],
                             rownames=['interpolated'], colnames=['nearest-neighbor'])
    print("\nConfusion matrix (interpolated vs. nearest-neighbor):")
    print(confusion)

    # Flag cases that specifically flipped between a "pure" label and something else,
    # since these are the ones most likely to be interpolation artifacts.
    flips = df[~df['agrees'] & ((df['category'].isin(['bloch', 'neel'])) |
                                  (df['category_nn'].isin(['bloch', 'neel'])))]
    print(f"\n{len(flips)} vertices flipped in/out of a 'pure' label "
          f"due to interpolation vs. nearest-neighbor sampling.")

    return df

In [ ]:
def plot_interpolation_mismatch(df, spacing=(8.0, 8.0, 8.0)):
    """
    Two-panel plot: (1) mismatch count per z-layer, (2) spatial map of
    mismatched vertices (x-y positions, colored by which category they
    flipped between).
    """
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    mismatch_by_layer = (~df['agrees']).groupby(df['iz'].round().astype(int)).sum()
    total_by_layer = df.groupby(df['iz'].round().astype(int)).size()
    frac_by_layer = mismatch_by_layer / total_by_layer

    axes[0].bar(frac_by_layer.index, frac_by_layer.values, color='tab:orange')
    axes[0].set_xlabel('z voxel layer')
    axes[0].set_ylabel('fraction mismatched')
    axes[0].set_title('Interpolation-driven classification disagreement by layer')

    mismatched = df[~df['agrees']]
    agree_pts = df[df['agrees']]
    axes[1].scatter(agree_pts['x'], agree_pts['y'], color='lightgray', s=5, alpha=0.3, label='agree')
    axes[1].scatter(mismatched['x'], mismatched['y'], color='red', s=15, label='mismatch')
    axes[1].set_xlabel('x (nm)')
    axes[1].set_ylabel('y (nm)')
    axes[1].set_title('Spatial location of mismatched vertices')
    axes[1].legend()

    plt.tight_layout()
    plt.show()

In [ ]:
def plot_interpolation_mismatch(df, spacing=(8.0, 8.0, 8.0)):
    """
    Two-panel plot: (1) mismatch count per z-layer, (2) spatial map of
    mismatched vertices (x-y positions, colored by which category they
    flipped between).
    """
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    mismatch_by_layer = (~df['agrees']).groupby(df['iz'].round().astype(int)).sum()
    total_by_layer = df.groupby(df['iz'].round().astype(int)).size()
    frac_by_layer = mismatch_by_layer / total_by_layer

    axes[0].bar(frac_by_layer.index, frac_by_layer.values, color='tab:orange')
    axes[0].set_xlabel('z voxel layer')
    axes[0].set_ylabel('fraction mismatched')
    axes[0].set_title('Interpolation-driven classification disagreement by layer')

    mismatched = df[~df['agrees']]
    agree_pts = df[df['agrees']]
    axes[1].scatter(agree_pts['x'], agree_pts['y'], color='lightgray', s=5, alpha=0.3, label='agree')
    axes[1].scatter(mismatched['x'], mismatched['y'], color='red', s=15, label='mismatch')
    axes[1].set_xlabel('x (nm)')
    axes[1].set_ylabel('y (nm)')
    axes[1].set_title('Spatial location of mismatched vertices')
    axes[1].legend()

    plt.tight_layout()
    plt.show()

In [ ]:
classified_normal = classify_pure_bloch_neel(wall_df, method='normal', tol=0.0005)
comparison_normal = compare_interpolation_effect(classified_normal, Mx, My, Mz, method='normal', tol=0.0005)
plot_interpolation_mismatch(comparison_normal)

classified_psi = classify_pure_bloch_neel(wall_df, method='psi', tol=0.005)
comparison_psi = compare_interpolation_effect(classified_psi, Mx, My, Mz, method='psi', tol=0.005)
plot_interpolation_mismatch(comparison_psi)

# save data_frame

In [ ]:
stop

In [ ]:
def make_json_safe(obj):
    """
    Recursively convert numpy types within a nested dict/list/tuple
    structure into native Python types, so it can be safely passed to
    json.dump. Handles numpy ints, floats, bools, and arrays.
    """
    if isinstance(obj, dict):
        return {k: make_json_safe(v) for k, v in obj.items()}
    elif isinstance(obj, (list, tuple)):
        return [make_json_safe(v) for v in obj]
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    elif isinstance(obj, (np.integer,)):
        return int(obj)
    elif isinstance(obj, (np.floating,)):
        return float(obj)
    elif isinstance(obj, (np.bool_,)):
        return bool(obj)
    else:
        return obj

In [ ]:
def save_sample_data(save_dir, sample_name, wall_df,
                      spacing=(2.0, 2.0, 2.0), z_offset=0, date = None,
                      extra_metadata=None):
    """
    Save the cropped volume and wall-analysis DataFrame for one sample,
    in a format suitable for later cross-sample/material comparison.

    Parameters
    ----------
    save_dir : str
        Directory to save into (created if it doesn't exist).
    sample_name : str
        Identifier for this sample/material, used in filenames and
        stamped into the saved data for later grouping.
    wall_df : pd.DataFrame
        The per-vertex wall analysis table (normals, psi, chirality, etc.)
    spacing : tuple
        (dz, dy, dx) physical voxel size, in nm.
    z_offset : int
        Offset applied to bring local z-indices back into the original
        volume's index frame (as discussed earlier).
    extra_metadata : dict or None
        Any additional key-value info you want stored alongside (e.g.
        material composition, deposition conditions, notes).
    """
    os.makedirs(save_dir, exist_ok=True)


    # --- Save wall_df, tagged with sample identity ---
    df_out = wall_df.copy()
    df_out['sample_name'] = sample_name
    df_path = os.path.join(save_dir, f'{sample_name}_wall_df_{date}.csv')
    df_out.to_csv(df_path, index=False)

    # --- Save human-readable metadata alongside, for provenance ---
    metadata = {
        'sample_name': sample_name,
        'spacing_nm': list(spacing),
        'z_offset': z_offset,
        'n_wall_vertices': len(wall_df),
        'volume_shape': list(volume['shape']),
    }

    if extra_metadata:
        metadata.update(extra_metadata)
    meta_path = os.path.join(save_dir, f'{sample_name}_wall_analysis_metadata_{date}.json')
    with open(meta_path, 'w') as f:
        json.dump(make_json_safe(metadata), f, indent=2)

    print(f"Saved: {df_path}\n       {meta_path}")
    return df_path, meta_path

In [ ]:
save_dir = data_dir
date_save = 'Interpolated_12OCT26'

save_sample_data(save_dir, sample_name, wall_df, spacing=(2.0, 2.0, 2.0), z_offset=0, date = date_save, extra_metadata=None)

# Save for paraview

In [ ]:
def export_wall_mesh_to_paraview(verts_xyz, faces, wall_df, save_path,
                                  scalar_cols=None, vector_col_groups=None):
    """
    Export the domain wall isosurface mesh, with all computed scalar and
    vector fields attached as point data, to a .vtp file readable
    directly in ParaView.

    Parameters
    ----------
    verts_xyz : (N, 3) array
        Mesh vertex coordinates, as produced by marching_cubes.
    faces : (M, 3) array
        Triangle connectivity, as produced by marching_cubes.
    wall_df : pd.DataFrame
        Per-vertex table (must have the same row order/length as
        verts_xyz) containing scalar and vector components to export.
    save_path : str
        Output file path, should end in '.vtp'.
    scalar_cols : list of str or None
        Column names in wall_df to attach as scalar point-data fields
        (e.g. ['psi_unsigned', 'psi_chiral', 'curl_magnitude',
        'dmi_interfacial']). If None, a sensible default set is used
        if present in wall_df.
    vector_col_groups : dict of {name: (x_col, y_col, z_col)} or None
        Named vector fields to attach, built from three wall_df columns
        each, e.g. {'Normal': ('nx','ny','nz'), 'Tangent': ('tx','ty','tz'),
        'M': ('Mx','My','Mz')}. If None, a sensible default set is used
        if present in wall_df.

    Returns
    -------
    mesh : pv.PolyData
        The mesh object that was saved, in case you want to inspect or
        further modify it in the same session.
    """
    faces_pv = np.hstack([np.full((faces.shape[0], 1), 3), faces]).astype(np.int64)
    mesh = pv.PolyData(verts_xyz, faces_pv)

    if scalar_cols is None:
        candidates = ['psi_unsigned', 'psi_chiral', 'curl_magnitude',
                      'dmi_interfacial', 'divergence', 'category']
        scalar_cols = [c for c in candidates if c in wall_df.columns]

    if vector_col_groups is None:
        candidates = {
            'Normal': ('nx', 'ny', 'nz'),
            'Tangent': ('tx', 'ty', 'tz'),
            'M': ('Mx', 'My', 'Mz'),
        }
        vector_col_groups = {
            name: cols for name, cols in candidates.items()
            if all(c in wall_df.columns for c in cols)
        }

    for col in scalar_cols:
        values = wall_df[col].values
        if values.dtype == object:  # e.g. 'category' being string labels
            # ParaView can't handle string arrays as colorable scalars directly;
            # encode as integer codes instead, and keep a lookup printed for reference.
            codes, uniques = pd.factorize(values)
            mesh.point_data[f'{col}_code'] = codes.astype(np.float32)
            print(f"'{col}' is categorical; saved as '{col}_code' with mapping: "
                  f"{dict(enumerate(uniques))}")
        else:
            mesh.point_data[col] = values.astype(np.float32)

    for vec_name, (cx, cy, cz) in vector_col_groups.items():
        vec_array = wall_df[[cx, cy, cz]].values.astype(np.float32)
        mesh.point_data[vec_name] = vec_array

    mesh.save(save_path)
    print(f"Saved mesh with {mesh.n_points} points, {mesh.n_cells} cells to {save_path}")
    print(f"  Scalar fields: {[c for c in scalar_cols]}")
    print(f"  Vector fields: {list(vector_col_groups.keys())}")

    return mesh

In [ ]:

save_path = os.path.join(data_dir, f'{sample_name}_wall_mesh_{date_save}.vtp')

mesh = export_wall_mesh_to_paraview(
    verts_xyz, faces, wall_df,
    save_path=save_path,
    scalar_cols=['M_normal', 'psi_unsigned', 'psi_chiral', 'curl_magnitude', 'iDMI', 'PMA', 'Nsk', 'Mx', 'My'],
    vector_col_groups={'Normal': ('nx', 'ny', 'nz'),
                        'Tangent': ('tx', 'ty', 'tz'),
                        'M': ('Mx', 'My', 'Mz')}
)

In [ ]:
save_path = os.path.join(data_dir, f'{sample_name}_div_wall_mesh_{date_save}.vtp')

div_mesh = export_wall_mesh_to_paraview(
    verts_xyz, faces, wall_df,
    save_path=save_path,
    scalar_cols=['div_M','source', 'sink'],
    vector_col_groups={'Normal': ('nx', 'ny', 'nz'),
                        'Tangent': ('tx', 'ty', 'tz'),
                        'M': ('Mx', 'My', 'Mz')}
)

save cropped volume

In [ ]:
volume.keys()

In [ ]:

# Swap array axes: (nz, ny, nx) -> (nx, ny, nz)
Mx = volume["Mx"].transpose(2, 1, 0)
My = volume["My"].transpose(2, 1, 0)
Mz = volume["Mz"].transpose(2, 1, 0)
magnitude = volume["magnitude"].transpose(2, 1, 0)

nx, ny, nz = Mx.shape

dx = dy = dz = 1.0   # voxel size, if known
x = np.arange(nx) * dx
y = np.arange(ny) * dy
z = np.arange(nz) * dz
X, Y, Z = np.meshgrid(x, y, z, indexing="ij")

grid = pv.StructuredGrid(X, Y, Z)

grid.point_data["M"] = np.stack([Mx.flatten(order="F"),
                                 My.flatten(order="F"),
                                 Mz.flatten(order="F")], axis=1)
grid.point_data["Mx"] = Mx.flatten(order="F")
grid.point_data["My"] = My.flatten(order="F")
grid.point_data["Mz"] = Mz.flatten(order="F")
grid.point_data["magnitude"] = magnitude.flatten(order="F")

grid.save(f"{sample_name}_{date_save}_cropped.vtk")




In [ ]:
stop